# Does Tokenization Fragmentation Predict Miscalibration in Bangla LLMs?
## A Unified Cross-Lingual Evaluation & Econometric Analysis Framework

### Research Objective & Hypotheses
This self-contained research notebook investigates the empirical relationship between **tokenization fragmentation** and **model miscalibration** in Large Language Models (LLMs) evaluated on Bengali (Bangla) tasks, using **BnMMLU** as the primary multilingual benchmark (~134K multiple-choice questions across 41 subjects) with a matched **English MMLU** subject control.

#### Core Research Hypotheses:
1. **$H_1$ (Vocabulary Coverage & Calibration Error):** Tokenizers with lower vocabulary coverage for Bangla (e.g., standard Latin-centric Llama BPE tokenizers relying on byte fallbacks) produce severe subword fragmentation, which statistically predicts increased model miscalibration (higher Expected Calibration Error, Brier Score, and Maximum Calibration Error).
2. **$H_2$ (Item-Level Residual Prediction):** At the question level, tokenization fragmentation metrics (Tokens-per-Word, Single-Token Retention Rate, Tokens-per-Character) positively predict the magnitude of individual model calibration residuals ($e_i = |\hat{p}_i - \mathbb{I}(\hat{y}_i = y_i)|$) after controlling for correctness, subject fixed effects, and prompt length.
3. **$H_3$ (Cross-Lingual Divergence):** The miscalibration-fragmentation relationship exhibits significant divergence across languages, reflecting the distinct morphological and script characteristics of Bengali (an inflectional abugida with complex conjuncts) compared to English (Latin alphabetic).

> [!IMPORTANT]
> **Epistemic & Non-Causal Framing:** All statistical models in this study estimate **observational associations**, never causal effects. Observational evaluations cannot eliminate confounding from pretraining corpus composition, alignment regimes, or parameter scale. All findings are strictly framed as "predicts" or "is associated with."

---

### Mathematical Formulations

#### 1. Tokenization Fragmentation Metrics
* **Tokens-per-Word (TPW / Fertility):**
  $$\text{TPW} = \frac{N_{\text{tokens}}}{N_{\text{words}}}$$
  Whitespace-delimited word baseline directly comparable to prior literature (e.g., TituLLMs, AdaptMix).
* **Single-Token Retention Rate (STRR):**
  $$\text{STRR} = \frac{1}{|W|} \sum_{w \in W} \mathbb{I}(\text{len}(\text{tokenize}(w)) = 1)$$
  Reported in both `isolated` mode (tokenizing words in isolation) and `in_context` mode (tokenizing `" " + word` to account for BPE space-prefix tokens).
* **Tokens-per-Character (TPC / Character Fertility):**
  $$\text{TPC} = \frac{N_{\text{tokens}}}{N_{\text{non-whitespace characters}}}$$
  Normalizes token count against non-whitespace character count, providing a script-agnostic cross-lingual baseline.

#### 2. Option Extraction, Uncertainty Quantification & Calibration Metrics
* **Candidate Option Probability Extraction (Logsumexp & Softmax):**
  For candidate option letters $k \in \{A, B, C, D\}$, all single-token variants (e.g., `"A"` and `" A"`) are resolved to token IDs $V_k$. Log-probabilities over the full vocabulary are pooled via logsumexp and renormalized over $A-D$:
  $$s_k = \log \sum_{v \in V_k} \exp(\log P(v \mid \text{prompt})), \quad P(k) = \frac{\exp(s_k)}{\sum_{j \in \{A, B, C, D\}} \exp(s_j)}$$
  **Confidence** is defined as the Max Softmax Probability (MSP): $\hat{p} = \max_{k \in \{A, B, C, D\}} P(k)$.
* **Option Mass ($\text{Mass}_{A-D}$):**
  Total unnormalized full-vocabulary probability mass allocated to the candidate option tokens:
  $$\text{Option Mass} = \sum_{k \in \{A, B, C, D\}} \sum_{v \in V_k} P(v \mid \text{prompt})$$
* **Expected Calibration Error (ECE):**
  $$\text{ECE} = \sum_{m=1}^M \frac{|B_m|}{N} |\text{acc}(B_m) - \text{conf}(B_m)|$$
* **Brier Score (BS) & Multiclass Brier Score:**
  $$\text{BS}_{\text{top1}} = \frac{1}{N} \sum_{i=1}^N (\hat{p}_i - \mathbb{I}(\hat{y}_i = y_i))^2, \quad \text{BS}_{\text{multi}} = \frac{1}{N} \sum_{i=1}^N \sum_{k=1}^4 (P_{i,k} - y_{i,k})^2$$
* **Negative Log-Likelihood (NLL):**
  $$\text{NLL} = -\frac{1}{N} \sum_{i=1}^N \log(\max(P_{i, y_i}, \epsilon))$$
* **Item-Level Calibration Residual:**
  $$e_i = |\hat{p}_i - \mathbb{I}(\hat{y}_i = y_i)|$$


## Section 1: Kaggle Runtime, GPU Setup, Caching & Logging
Configures environment detection, initializes a dedicated run logger writing to `results/run.log`, sets up `/kaggle/working/hf_cache` to avoid redundant downloads, and inspects hardware capabilities (NVIDIA T4 16GB).


In [ ]:
import os
import sys
import platform
import logging
import warnings

# Detect execution platform
IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = "google.colab" in sys.modules

if IS_KAGGLE:
    BASE_WORKING_DIR = "/kaggle/working"
    BASE_INPUT_DIR = "/kaggle/input"
    print("🌐 Running on Kaggle Notebook Environment")
elif IS_COLAB:
    BASE_WORKING_DIR = "/content"
    BASE_INPUT_DIR = "/content"
    print("🌐 Running on Google Colab Environment")
else:
    BASE_WORKING_DIR = os.path.abspath(".")
    BASE_INPUT_DIR = os.path.abspath(".")
    print("💻 Running on Local / Standalone Environment")

# Unified output and cache directories
OUTPUT_DIR = os.path.join(BASE_WORKING_DIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

HF_CACHE_DIR = os.path.join(BASE_WORKING_DIR, "hf_cache")
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

# Initialize structured file + console logger
def setup_logger(output_dir: str) -> logging.Logger:
    logger = logging.getLogger("BanglaLLMCalibration")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    c_handler = logging.StreamHandler(sys.stdout)
    c_handler.setLevel(logging.INFO)
    c_format = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S")
    c_handler.setFormatter(c_format)
    logger.addHandler(c_handler)

    log_file = os.path.join(output_dir, "run.log")
    f_handler = logging.FileHandler(log_file, encoding="utf-8")
    f_handler.setLevel(logging.INFO)
    f_format = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    f_handler.setFormatter(f_format)
    logger.addHandler(f_handler)
    return logger

logger = setup_logger(OUTPUT_DIR)
logger.info(f"Output directory: {OUTPUT_DIR}")
logger.info(f"HuggingFace cache directory: {HF_CACHE_DIR}")

# Hardware and accelerator check
try:
    import torch
    HAS_TORCH = True
    GPU_AVAILABLE = torch.cuda.is_available()
    DEVICE_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "CPU"
    DEVICE_COUNT = torch.cuda.device_count() if GPU_AVAILABLE else 0
    logger.info(f"PyTorch: {torch.__version__} | GPU Available: {GPU_AVAILABLE} ({DEVICE_NAME}, count={DEVICE_COUNT})")
    if GPU_AVAILABLE:
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        logger.info(f"VRAM: {vram_gb:.2f} GB | Compute Capability: {torch.cuda.get_device_capability(0)}")
        if "T4" in DEVICE_NAME:
            logger.info("NVIDIA T4 detected: Always use 'float16' or 'float32'. Never use 'bfloat16' on T4.")
except ImportError:
    HAS_TORCH = False
    GPU_AVAILABLE = False
    DEVICE_NAME = "CPU (PyTorch not installed)"
    raise RuntimeError("PyTorch is not installed. In STRICT_REAL_MODE, PyTorch is required.")


## Section 2: Imports, Reproducibility & Central Constants
Defines single sources of truth for experimental constants, random seeds, and imports core scientific computing libraries.


In [ ]:
import ast
from dataclasses import dataclass, asdict
from datetime import datetime
import gc
import json
import math
import random
import re
import threading
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests
from statsmodels.formula.api import ols
from tqdm.auto import tqdm

# Visualization libraries with headless display support
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_VISUALIZATION = True
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
    plt.rcParams["figure.dpi"] = 120
    plt.rcParams["font.family"] = "sans-serif"
except ImportError:
    HAS_VISUALIZATION = False
    logger.warning("Matplotlib / Seaborn not available. Plot generation functions will skip rendering.")

# ==========================================
# CENTRAL CONSTANTS (SINGLE SOURCE OF TRUTH)
# ==========================================
OPTION_LETTERS: List[str] = ["A", "B", "C", "D"]
DEFAULT_SEED: int = 42

# Research thresholds
MAX_COLLAPSE_THRESHOLD: float = 0.80       # Abort/Invalid if single letter exceeds 80% predictions
MIN_OPTION_MASS_THRESHOLD: float = 0.50     # Abort/Invalid if mean probability mass on A-D is < 0.50
MIN_QUARTILE_SAMPLE_THRESHOLD: int = 100    # Warn if any TPW quartile contains < 100 samples
CHECKPOINT_FREQ: int = 25                   # Flush CSV every 25 items to prevent progress loss
MEM_LOG_FREQ: int = 200                     # Log GPU memory usage every 200 items

# Prompt variant answer cues (for the "answer_cue" prompt variant)
ANSWER_CUES: Dict[str, str] = {
    "bn": "উত্তর:",
    "en": "Answer:",
}
VALID_PROMPT_VARIANTS = {"plain", "answer_cue", "few_shot"}

def seed_everything(seed: int = DEFAULT_SEED) -> None:
    """Fix random seeds across random, numpy, and torch for strict reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if HAS_TORCH:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    logger.info(f"Global random seed locked to: {seed}")

seed_everything(DEFAULT_SEED)


## Section 3: Model Registry & Central Experiment Configuration
Defines the `MODEL_REGISTRY` catalog and central `CONFIG`. To run any model independently, change only `CONFIG.model_key` or call `run_one_model(key)` directly.

| Model Key | Hugging Face Model ID | Params | Architecture / tokenizer | Target Role |
|---|---|---|---|---|
| `llama32_3b` | `meta-llama/Llama-3.2-3B-Instruct` (gated) | 3.21B | Llama-3.2, 128K byte-level BPE | High-fragmentation anchor |
| `titullm_1b` | `hishab/titulm-llama-3.2-1b-v1.1` | 1.23B | Llama-3.2 continued pre-training; tokenizer extension **to be verified (Section 14b)** | Bangla-adapted (base model) |
| `tigerllm_1b` | `md-nishat-008/TigerLLM-1B-it` | 1.0B | **Gemma-3** (Hub config `gemma3_text`), different tokenizer family | Out-of-family replication |

### Prompt Variants & Position Debiasing Strategy
Small and base models frequently suffer from single-letter logit collapse or low option mass. We provide four orthogonal degrees of adaptation:
1. **`plain` (default):** Prompt ends with the task instruction followed by `:\n`, expecting an immediate bare letter (`A`, `B`, `C`, or `D`).
2. **`answer_cue`:** Appends a language-appropriate completion cue (`উত্তর:` for Bangla, `Answer:` for English) directly after the instruction. Many instruction-tuned models follow completion cues more reliably.
3. **`few_shot` (4-shot or 8-shot):** Prepends balanced, fixed in-context exemplars drawn from the held-out exemplar pool (`fewshot_exemplars.json`).
4. **`cyclic4` position debiasing:** Computes predictions across 4 cyclic permutations of option order (positions shifted by $s \in \{0, 1, 2, 3\}$) and averages mapped probabilities back to canonical options. Items with self-referential / cross-referencing options (e.g. "both A and B", "all of the above", "উপরের সবগুলো") are automatically exempted from permutation.

> **Methodology & Validity Selection Rules:**
> - Settings (`prompt_variant`, `n_shots`, `use_chat_template`, `debias`) are selected **per model** using **validity gates only** (`mean_mass >= MIN_OPTION_MASS_THRESHOLD` and `top_letter_share <= MAX_COLLAPSE_THRESHOLD`).
> - **Never select settings by accuracy**: the diagnostic subset overlaps the evaluation pool; selecting variants by accuracy would create severe data leakage and invalidate calibration findings.
> - **Calibration interpretation under cyclic4:** `cyclic4` averages across permutations, changing the empirical definition of model "confidence". Both raw (`pred_raw`, `confidence_raw`) and debiased calibration metrics must be reported.
> - **Reporting:** Every chosen per-model setting must be explicitly reported in the paper's methodology and limitations sections.

**Notes**
- Default backend is now `transformers` (real models, real data). Use `backend="mock"` only for an offline dry run; it uses synthetic items and writes to `results/mock_run/`.
- **BnMMLU is gated** and has two configs (`full`, `hard`). Accept the conditions on the dataset page and provide `HF_TOKEN`. Llama-3.2 is gated too.
- `exclude_markup=True` drops items containing MathML, because tag tokens would be counted as language fragmentation.
- `subject_match_mode="intersection"` keeps only BnMMLU subjects that have an MMLU analogue and draws equal per-subject counts from both languages.
- Only `batch_size = 1` is supported.

In [ ]:
# ==========================================
# GLOBAL STRICT REAL-ONLY CONFIGURATION
# ==========================================
STRICT_REAL_MODE: bool = True
os.environ["STRICT_REAL_MODE"] = "True"

# ==========================================
# MODEL REGISTRY  (all IDs verified on the Hugging Face Hub)
# ==========================================
MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "llama32_3b": {
        "hf_id": "meta-llama/Llama-3.2-3B-Instruct",
        "use_chat_template": True,   # PATCHED: Instruct model needs chat template for option mass >= 0.50
        "dtype": "float16",
        "needs_hf_token": True,                 # GATED: accept the Llama 3.2 license on the model page first
        "tokenizer_family": "llama3_bpe_128k",
        "expected_model_type": "llama",
        "notes": "Anchor model: original Llama-3.2 128K BPE tokenizer (byte-level, no Bangla-specific vocabulary). use_chat_template=True required for instruction-tuned variant: plain next-token prediction yields option_mass < 0.50.",
    },
    "titullm_1b": {
        "hf_id": "hishab/titulm-llama-3.2-1b-v1.1",
        "use_chat_template": False,
        "prompt_variant": "answer_cue",    # PATCHED: base model needs answer cue to avoid logit collapse
        "collapse_threshold_override": None, # None = use global MAX_COLLAPSE_THRESHOLD; set float to relax
        "dtype": "float16",
        "needs_hf_token": False,
        "tokenizer_family": "llama3_continued_pretrain",   # Audited in Section 14b
        "expected_model_type": "llama",
        "notes": ("Bangla continued-pretraining of Llama-3.2-1B (base model, no chat tuning). "
                  "prompt_variant=answer_cue applied to prevent logit collapse on base models. "
                  "Section 14b tokenizer audit tests whether vocabulary/tokenization differs from Llama-3.2."),
    },
    "tigerllm_1b": {
        "hf_id": "md-nishat-008/TigerLLM-1B-it",
        "use_chat_template": False,
        "dtype": "float32",                      # Gemma-3 is unstable in fp16 and NVIDIA T4 has no native bf16
        "needs_hf_token": False,
        "tokenizer_family": "gemma3",           # Hub repo tagged gemma3_text; Gemma-3 262K vocabulary
        "expected_model_type": "gemma3_text",
        "notes": ("Out-of-family replication model. Hub config is Gemma-3 (262K vocab), NOT Llama, "
                  "providing a distinct tokenizer family comparison."),
    },
}


@dataclass
class ExperimentConfig:
    # Model selector: "llama32_3b", "titullm_1b", or "tigerllm_1b"
    model_key: str = "llama32_3b"

    # Backend engine: STRICT REAL evaluation uses "transformers"
    backend: str = "transformers"

    # Evaluation mode: "extended" (primary) or "baseline"
    mode: str = "extended"

    # Dataset file path (optional local override; real runs download from Hub)
    dataset_path: Optional[str] = None

    # Number of items PER LANGUAGE: Exactly 1500 samples per language
    n_samples: int = 1500

    # Batch size (only 1 is supported; batched inference is NOT implemented)
    batch_size: int = 1

    # Optional path to an attached Kaggle dataset directory containing previous session outputs
    previous_results_dir: Optional[str] = None

    # Reproducibility seed
    seed: int = DEFAULT_SEED

    # Calibration binning
    n_bins: int = 10
    bin_strategy: str = "uniform"

    # Run legacy baseline text generation (opt-in; default False)
    run_baseline: bool = False

    # Output directory for results: Default /kaggle/working/results/ on Kaggle
    output_dir: str = OUTPUT_DIR

    # Hugging Face access token (retrieved securely from Kaggle Secrets or env)
    hf_token: Optional[str] = None

    # ---------------- Real-data settings ----------------
    # BnMMLU is a GATED dataset with two configs ("full" and "hard"); config 'full' is required.
    bn_dataset_id: str = "samanjoy2/BnMMLU"
    bn_hf_config: str = "full"
    # Optional local parquet/csv copy of BnMMLU (overrides the Hub download when set)
    bn_local_path: Optional[str] = None
    # Only needed if a local copy stores answers as bare digits; the Hub version uses letters a-d
    bn_index_base: Optional[int] = None
    # English control: cais/mmlu, configuration all, split test
    en_dataset_id: str = "cais/mmlu"
    en_hf_config: str = "all"
    en_split: str = "test"

    # BnMMLU keeps MathML tags inside question text. Those tags inflate token counts and would be
    # measured as "fragmentation" of the language. True = drop items containing MathML.
    exclude_markup: bool = True

    # "intersection": only BnMMLU subjects that map to an MMLU subject; equal per-subject counts in both languages.
    # "all_bn": stratified sample over ALL BnMMLU subjects; English matched where possible, topped up otherwise.
    subject_match_mode: str = "intersection"

    # Diagnostic / dev set items per language
    dev_n: int = 200

    # Smoke test items
    smoke_n: int = 30
    # Strict smoke test abort on collapse
    strict_smoke: bool = False

    # Prompt variant: "plain" (original) or "answer_cue" (appends language-specific answer cue)
    prompt_variant: str = "answer_cue"  # PATCHED: answer_cue + chat_template is the winning variant for llama32_3b

    # Few-shot count: 4 = 4-shot (default), 8 = 8-shot. Allowed values: {4, 8}
    n_shots: int = 4

    # Debiasing strategy: "none" (default) or "cyclic4" (v3b+)
    debias: str = "none"

    # Allow few-shot English candidates to fall back to unused test pool if dev split fails
    allow_fewshot_test_pool_fallback: bool = False

    # Per-run collapse threshold override (None = use global MAX_COLLAPSE_THRESHOLD = 0.80).
    # Set to 1.0 to fully suppress the collapse validity gate for base-model analysis runs.
    collapse_threshold_override: Optional[float] = None

    # Analysis validity filter: if True, filter out runs with validity_flag != 'VALID'
    analysis_valid_only: bool = True


def validate_config(config: "ExperimentConfig") -> None:
    if getattr(config, "dev_n", 200) < 0:
        raise ValueError(f"CONFIG.dev_n must be non-negative, got {config.dev_n}")
    """Validate global experiment configuration parameters."""
    if STRICT_REAL_MODE and config.backend != "transformers":
        raise ValueError(f"STRICT_REAL_MODE=True requires CONFIG.backend='transformers', but got '{config.backend}'. Mock modes are prohibited.")

    if config.batch_size != 1:
        raise NotImplementedError("Batched inference is not implemented in this notebook. Keep CONFIG.batch_size = 1.")

    if config.prompt_variant not in VALID_PROMPT_VARIANTS:
        raise ValueError(f"Invalid CONFIG.prompt_variant='{config.prompt_variant}'. Allowed: {VALID_PROMPT_VARIANTS}")

    if config.prompt_variant == "few_shot":
        if config.n_shots not in {4, 8}:
            raise ValueError(f"Invalid CONFIG.n_shots={config.n_shots} with prompt_variant='few_shot'. Allowed: {{4, 8}}")
    else:
        if config.n_shots not in {0, 4, 8}:
            raise ValueError(f"Invalid CONFIG.n_shots={config.n_shots} with prompt_variant='{config.prompt_variant}'. Allowed: {{0, 4, 8}}")

    if config.debias not in ("none", "cyclic4"):
        raise ValueError(f"Invalid CONFIG.debias='{config.debias}'. Allowed: ('none', 'cyclic4')")


CONFIG = ExperimentConfig(
    model_key="llama32_3b",
    backend="transformers",
    mode="extended",
    n_samples=1500,
    dev_n=200,
    batch_size=1,
    run_baseline=False,
    output_dir=OUTPUT_DIR,
)

validate_config(CONFIG)
os.makedirs(CONFIG.output_dir, exist_ok=True)

# Securely retrieve Hugging Face token without exposing it
if not CONFIG.hf_token:
    CONFIG.hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if not CONFIG.hf_token and IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            CONFIG.hf_token = user_secrets.get_secret("HF_TOKEN")
            logger.info("Successfully retrieved HF_TOKEN from Kaggle Secrets.")
        except Exception as e:
            logger.warning(f"Could not retrieve HF_TOKEN from Kaggle Secrets: {e}")
    if not CONFIG.hf_token and IS_COLAB:
        try:
            from google.colab import userdata
            CONFIG.hf_token = userdata.get("HF_TOKEN")
            logger.info("Successfully retrieved HF_TOKEN from Colab secrets.")
        except Exception:
            pass

# Verify HF_TOKEN is present for real gated resources
if not CONFIG.hf_token and not CONFIG.bn_local_path:
    raise RuntimeError(
        "No Hugging Face token found (HF_TOKEN). BnMMLU (samanjoy2/BnMMLU) is a gated benchmark dataset "
        "and LLaMA-3.2 (meta-llama/Llama-3.2-3B-Instruct) is a gated model.\n"
        "  1. Accept access conditions on https://huggingface.co/datasets/samanjoy2/BnMMLU\n"
        "  2. Accept license on https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct\n"
        "  3. Add HF_TOKEN under Kaggle Secrets or environment variables.\n"
        "STRICT_REAL_MODE is active: mock execution and synthetic data fallbacks are disabled."
    )

logger.info(f"STRICT_REAL_MODE: {STRICT_REAL_MODE}")
logger.info(f"Active Model Key: {CONFIG.model_key}")
logger.info(f"Backend Engine:   {CONFIG.backend}")
logger.info(f"Sample Count:     {CONFIG.n_samples} per language")
logger.info(f"Dev Set Count:    {CONFIG.dev_n} per language")
logger.info(f"BnMMLU source:    {CONFIG.bn_local_path or (CONFIG.bn_dataset_id + ' [' + CONFIG.bn_hf_config + ']')}")
logger.info(f"MMLU source:      {CONFIG.en_dataset_id} ['{CONFIG.en_hf_config}', split='{CONFIG.en_split}']")
logger.info(f"Markup excluded:  {CONFIG.exclude_markup} | Subject matching: {CONFIG.subject_match_mode}")
logger.info(f"Output dir:       {CONFIG.output_dir}")
logger.info(f"Prompt variant:   {CONFIG.prompt_variant}")


## Section 4: Dataset Discovery, Parsing, Label Validation & Stratified Sampling
Core data handling functions:
1. **`find_dataset_path`:** Ensures English searches never resolve to a Bangla file; raises a clear error in real mode if `dataset_path` is None (disallowing auto-guessing in production).
2. **`parse_answer_label`:** Disambiguates 0-indexed vs. 1-indexed numeric labels via explicit `index_base` parameter (raising `ValueError` if a bare digit is encountered without an explicit base).
3. **`validate_items`:** Drops items with `len(options) != 4` or invalid gold answer indices. Logs dropped counts and gold distributions. Never scores an invalid item as "wrong."
4. **`sample_stratified_subset`:** Proportional subject sampling with locked seeds.


In [ ]:
def find_dataset_path(preferred_path: Optional[str] = None, language: str = "bn", is_mock: bool = False) -> str:
    """Discover dataset file or raise error in real mode to avoid auto-guessing."""
    if preferred_path and os.path.exists(preferred_path):
        return preferred_path

    if STRICT_REAL_MODE or not is_mock:
        raise ValueError(
            f"Explicit 'dataset_path' must be provided for real evaluation (language='{language}'). "
            "Auto-discovery and synthetic fallback are strictly prohibited."
        )

    # Search candidates (Legacy offline development only)
    search_dirs = [
        "/kaggle/input",
        "/kaggle/input/bnmmlu",
        "/kaggle/input/bnmmlu-full",
        "/kaggle/input/bangla-mmlu",
        "/home/naeem/Downloads/cn",
        "/home/naeem/Downloads/thesis",
        ".",
        "data"
    ]

    extensions = [".parquet", ".csv", ".jsonl", ".json"]
    for d in search_dirs:
        if not os.path.exists(d):
            continue
        for root, _, files in os.walk(d):
            for f in files:
                f_lower = f.lower()
                if any(f_lower.endswith(ext) for ext in extensions):
                    if language == "en":
                        # CRITICAL FIX: English search must NEVER resolve to a Bangla BnMMLU file
                        if "bnmmlu" in f_lower or "bangla" in f_lower:
                            continue
                        if "mmlu" in f_lower or "english" in f_lower:
                            candidate = os.path.join(root, f)
                            logger.info(f"Discovered candidate English dataset: {candidate}")
                            return candidate
                    else:
                        if any(kw in f_lower for kw in ["bnmmlu", "bangla"]):
                            candidate = os.path.join(root, f)
                            logger.info(f"Discovered candidate Bangla dataset: {candidate}")
                            return candidate

    return "FALLBACK_SYNTHETIC"


def parse_options(options_raw: Any) -> List[str]:
    """Robustly parse options into a list of 4 strings."""
    if options_raw is None:
        return []

    if isinstance(options_raw, (list, tuple, np.ndarray)):
        return [str(x).strip() for x in options_raw]

    if isinstance(options_raw, float) and pd.isna(options_raw):
        return []

    s = str(options_raw).strip()

    # 1. Try JSON
    try:
        obj = json.loads(s)
        if isinstance(obj, list):
            return [str(x).strip() for x in obj]
    except Exception:
        pass

    # 2. Try Python literal eval
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list):
            return [str(x).strip() for x in obj]
    except Exception:
        pass

    # 3. Extract quoted strings using regex
    try:
        parts = []
        for m in re.finditer(r"'([^']*)'", s):
            parts.append(m.group(1).strip())
        if not parts:
            for m in re.finditer(r'"([^"]*)"', s):
                parts.append(m.group(1).strip())
        if parts:
            return parts
    except Exception:
        pass

    # 4. Fallback comma split
    try:
        return [p.strip().strip("'").strip('"') for p in s.strip("[]").split(",") if p.strip()]
    except Exception:
        return []


def parse_answer_label(ans_raw: Any, index_base: Optional[int] = None) -> Tuple[str, int]:
    """Parse raw answer into (letter, index).
    
    Supports:
    - 'A', 'B', 'C', 'D' / 'a', 'b', 'c', 'd'
    - Bengali numerals ১, ২, ৩, ৪
    - Digits: Requires explicit index_base (0 for 0-3, 1 for 1-4).
    """
    if ans_raw is None:
        return "", -1

    s = str(ans_raw).strip().upper()

    # Bengali numeral mapping
    bn_map = {"১": "A", "২": "B", "৩": "C", "৪": "D"}
    if s in bn_map:
        letter = bn_map[s]
        return letter, OPTION_LETTERS.index(letter)

    if s in OPTION_LETTERS:
        return s, OPTION_LETTERS.index(s)

    # Integer digits: Must strictly adhere to index_base
    if s in ["0", "1", "2", "3", "4"]:
        if index_base is None:
            raise ValueError(
                f"Ambiguous numeric answer label '{s}'. Must explicitly specify index_base=0 (0-3) or index_base=1 (1-4)."
            )
        val = int(s)
        if index_base == 0:
            if 0 <= val <= 3:
                return OPTION_LETTERS[val], val
            else:
                raise ValueError(f"Value '{s}' out of range for index_base=0 (expected 0-3).")
        elif index_base == 1:
            if 1 <= val <= 4:
                idx = val - 1
                return OPTION_LETTERS[idx], idx
            else:
                raise ValueError(f"Value '{s}' out of range for index_base=1 (expected 1-4).")
        else:
            raise ValueError(f"Invalid index_base={index_base}. Must be 0 or 1.")

    return s, -1


def validate_items(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Validate MCQ items: drop items with len(options) != 4 or answer_idx not in {0,1,2,3}.
    Prints dropped item counts with reasons and the gold label distribution.
    Never scores an invalid item as 'wrong'.
    """
    valid_items = []
    dropped_reasons: Dict[str, int] = {}
    gold_distribution: Dict[str, int] = {letter: 0 for letter in OPTION_LETTERS}

    for item in items:
        options = item.get("options", [])
        ans_idx = item.get("answer_idx", -1)
        gold_letter = item.get("correct_answer", "")
        question = str(item.get("question", "")).strip()

        if not question:
            dropped_reasons["empty_question"] = dropped_reasons.get("empty_question", 0) + 1
            continue

        if not isinstance(options, list) or len(options) != 4:
            count_key = f"invalid_options_count_{len(options) if isinstance(options, list) else 'non_list'}"
            dropped_reasons[count_key] = dropped_reasons.get(count_key, 0) + 1
            continue

        if any(not str(opt).strip() for opt in options):
            dropped_reasons["empty_option_string"] = dropped_reasons.get("empty_option_string", 0) + 1
            continue

        if ans_idx not in [0, 1, 2, 3] or gold_letter not in OPTION_LETTERS:
            dropped_reasons["invalid_answer_label"] = dropped_reasons.get("invalid_answer_label", 0) + 1
            continue

        valid_items.append(item)
        gold_distribution[gold_letter] += 1

    total_input = len(items)
    total_valid = len(valid_items)
    total_dropped = total_input - total_valid

    logger.info(f"Data Validation: {total_valid:,} valid items retained ({total_dropped:,} dropped).")
    if dropped_reasons:
        logger.info(f"Dropped items breakdown: {dropped_reasons}")
    logger.info(f"Gold label distribution: {gold_distribution}")

    return valid_items


def create_synthetic_benchmark_dataset() -> List[Dict[str, Any]]:
    """Create synthetic MCQ dataset in Bangla and English for testing without external files."""
    synthetic_items = [
        # Bangla items across varied subjects
        {"question_id": "SYN_BN_001", "subject": "Financial Accounting", "question": "প্রকৃত জাবেদায় নিচের কোনটি লিপিবদ্ধ করা হয়?", "options": ["ক্রয় জাবেদা", "বিক্রয় জাবেদা", "সমন্বয় জাবেদা", "বিশেষ জাবেদা"], "correct_answer": "C", "answer_idx": 2, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_002", "subject": "Physics", "question": "কোন মাধ্যমের মধ্য দিয়ে শব্দ দ্রুততম গতিতে সঞ্চালিত হয়?", "options": ["বায়বীয় মাধ্যম", "কঠিন মাধ্যম", "তরল মাধ্যম", "শূন্যস্থান"], "correct_answer": "B", "answer_idx": 1, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_003", "subject": "History", "question": "ঐতিহাসিক ভাষা আন্দোলনের চূড়ান্ত পর্যায় কোন সালে সংঘটিত হয়েছিল?", "options": ["১৯৪৭", "১৯৫২", "১৯৬৯", "১৯৭১"], "correct_answer": "B", "answer_idx": 1, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_004", "subject": "Computer Science", "question": "বাইনারি সার্চ ট্রিতে উপাদান অনুসন্ধানের গড় সময় জটিলতা কোনটি?", "options": ["O(1)", "O(n)", "O(log n)", "O(n log n)"], "correct_answer": "C", "answer_idx": 2, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_005", "subject": "General Science", "question": "উদ্ভিদ কোষে সালোকসংশ্লেষণ প্রক্রিয়ার জন্য প্রধানত কোন অঙ্গাণু দায়ী?", "options": ["মাইটোকন্ড্রিয়া", "রাইবোসোম", "প্লাস্টিড বা ক্লোরোপ্লাস্ট", "গলগি বডি"], "correct_answer": "C", "answer_idx": 2, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_006", "subject": "Bengali Grammar", "question": "'সন্ধি' ব্যাকরণের কোন অংশে আলোচিত হয়?", "options": ["রূপতত্ত্বে", "ধ্বনিযোগে ধ্বনিতত্ত্বে", "বাক্যতত্ত্বে", "অর্থতত্ত্বে"], "correct_answer": "B", "answer_idx": 1, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_007", "subject": "Geography", "question": "পৃথিবীর সর্ববৃহৎ ব-দ্বীপ অঞ্চল কোনটি?", "options": ["নীলনদের ব-দ্বীপ", "আমাজন ব-দ্বীপ", "গঙ্গা-ব্রহ্মপুত্র ব-দ্বীপ", "মিসিসিপি ব-দ্বীপ"], "correct_answer": "C", "answer_idx": 2, "language": "bn", "benchmark": "BnMMLU"},
        {"question_id": "SYN_BN_008", "subject": "Chemistry", "question": "সাধারণ তাপমাত্রায় নিচের কোন ধাতুটি তরল অবস্থায় থাকে?", "options": ["সোডিয়াম", "পারদ", "লোহা", "অ্যালুমিনিয়াম"], "correct_answer": "B", "answer_idx": 1, "language": "bn", "benchmark": "BnMMLU"},
        # English matched control items
        {"question_id": "SYN_EN_001", "subject": "Physics", "question": "Through which medium does sound travel at the highest speed?", "options": ["Gas", "Solid", "Liquid", "Vacuum"], "correct_answer": "B", "answer_idx": 1, "language": "en", "benchmark": "MMLU"},
        {"question_id": "SYN_EN_002", "subject": "Computer Science", "question": "What is the average time complexity of searching in a balanced BST?", "options": ["O(1)", "O(n)", "O(log n)", "O(n log n)"], "correct_answer": "C", "answer_idx": 2, "language": "en", "benchmark": "MMLU"},
        {"question_id": "SYN_EN_003", "subject": "General Science", "question": "Which organelle is responsible for photosynthesis in plants?", "options": ["Mitochondria", "Ribosome", "Chloroplast", "Golgi apparatus"], "correct_answer": "C", "answer_idx": 2, "language": "en", "benchmark": "MMLU"},
        {"question_id": "SYN_EN_004", "subject": "Chemistry", "question": "Which metal remains liquid at room temperature?", "options": ["Sodium", "Mercury", "Iron", "Aluminum"], "correct_answer": "B", "answer_idx": 1, "language": "en", "benchmark": "MMLU"},
    ]
    for idx, item in enumerate(synthetic_items):
        item["raw_index"] = idx
    return synthetic_items


# ---------------------------------------------------------------------------
# MathML / markup detection
# BnMMLU stores equations as MathML (outer <math> tags removed, inner tags kept). Those tags are not Bangla,
# but a tokenizer still splits them into many pieces, which would be mis-measured as "fragmentation".
# ---------------------------------------------------------------------------
MATHML_TAG_RE = re.compile(
    r"</?(?:math|mi|mn|mo|mrow|msup|msub|msubsup|mfrac|msqrt|mroot|mtext|mstyle|mover|munder|munderover|"
    r"mtable|mtr|mtd|mspace|mfenced|menclose|mpadded|mphantom|semantics|annotation)(?:\s[^<>]*)?/?>"
    r"|&lt;/?m(?:i|n|o|row|sup|sub|frac|sqrt|text)\b",
    re.IGNORECASE,
)


def contains_markup(question: str, options: List[str]) -> bool:
    """True if the question or any option contains MathML tags."""
    blob = str(question or "") + " " + " ".join(str(o or "") for o in (options or []))
    return bool(MATHML_TAG_RE.search(blob))


# ---------------------------------------------------------------------------
# Subject normalisation and the BnMMLU -> MMLU subject map
# ---------------------------------------------------------------------------
def normalize_subject(s: Any) -> str:
    """Lower-case, underscores to spaces, strip punctuation. Non-Latin names fall back to the stripped original."""
    raw = str(s or "").strip()
    norm = re.sub(r"[^a-z0-9 ]+", "", raw.lower().replace("_", " ").replace("-", " "))
    norm = re.sub(r"\s+", " ", norm).strip()
    return norm if norm else raw.lower()


# RESEARCH-DESIGN CHOICE (review and justify this in the paper): BnMMLU is built from Bangladeshi school/college and
# competitive-exam material, MMLU from US-oriented exams, so "subject matching" is approximate. Keys are normalised
# BnMMLU subject names (Table 5 of the BnMMLU paper); values are exact MMLU subject ids. Subjects with no MMLU
# analogue (Bengali, Agricultural Studies, General Knowledge, Finance, Science, ...) are deliberately unmapped.
SUBJECT_MAP_BN_TO_MMLU: Dict[str, List[str]] = {
    "accounting": ["professional_accounting"],
    "advanced mathematics": ["college_mathematics", "high_school_mathematics"],
    "general mathematics": ["elementary_mathematics", "high_school_mathematics"],
    "mathematics": ["elementary_mathematics", "high_school_mathematics"],
    "advanced physics": ["college_physics", "high_school_physics"],
    "general physics": ["conceptual_physics", "high_school_physics"],
    "physics": ["conceptual_physics", "high_school_physics"],
    "biology": ["high_school_biology", "college_biology"],
    "chemistry": ["high_school_chemistry", "college_chemistry"],
    "statistics": ["high_school_statistics"],
    "ict": ["high_school_computer_science", "college_computer_science"],
    "information and communication technology": ["high_school_computer_science", "college_computer_science"],
    "economics": ["high_school_macroeconomics", "high_school_microeconomics", "econometrics"],
    "geography": ["high_school_geography"],
    "psychology": ["high_school_psychology", "professional_psychology"],
    "logic": ["formal_logic", "logical_fallacies"],
    "civics and citizenship": ["high_school_government_and_politics"],
    "religion and moral education": ["world_religions", "philosophy"],
    "religious studies": ["world_religions", "philosophy"],
    "business entrepreneurship": ["management", "marketing", "business_ethics"],
    "production management": ["management"],
    "social work": ["sociology"],
}

# RESEARCH-DESIGN CHOICE: approximate subject matching between Bangladeshi
# curriculum (BnMMLU) and US-oriented MMLU; justify in the paper. Subjects with no MMLU analogue
# (Bengali language/literature/poetry, agriculture, history/culture, banking, entrepreneurship,
# production, general knowledge, general science, miscellaneous GK) are deliberately unmapped.
SUBJECT_MAP_EXTRA: Dict[str, List[str]] = {
    "cell biology genetics": ["high_school_biology", "college_biology", "medical_genetics"],
    "human biology anatomy": ["anatomy", "human_aging", "clinical_knowledge"],
    "ecology environmental biology": ["high_school_biology"],
    "organic chemistry": ["high_school_chemistry", "college_chemistry"],
    "inorganic chemistry": ["high_school_chemistry", "college_chemistry"],
    "physical analytical chemistry": ["high_school_chemistry", "college_chemistry"],
    "conceptual physics basic laws": ["conceptual_physics", "high_school_physics"],
    "mechanics": ["high_school_physics", "college_physics"],
    "thermodynamics electromagnetism": ["college_physics", "high_school_physics"],
    "relativity modern physics": ["astronomy", "college_physics"],
    "algebra number theory": ["abstract_algebra", "high_school_mathematics"],
    "calculus analysis": ["college_mathematics", "high_school_mathematics"],
    "statistics probability inference": ["high_school_statistics"],
    "programming algorithms": ["high_school_computer_science", "college_computer_science"],
    "networking security": ["computer_security"],
    "ai data science basics": ["machine_learning"],
    "cognitive psychology": ["high_school_psychology", "professional_psychology"],
    "behavioral psychology": ["high_school_psychology", "professional_psychology"],
    "civics governance": ["high_school_government_and_politics"],
    "financial accounting": ["professional_accounting"],
    "corporate finance": ["high_school_macroeconomics", "high_school_microeconomics"],
    "business strategy management": ["management", "marketing"],
    "comparative religion": ["world_religions"],
    "moral ethical studies": ["moral_disputes", "philosophy"],
    "critical thinking": ["logical_fallacies", "formal_logic"],
    "social work welfare": ["sociology"],
}


def build_subject_map() -> Dict[str, List[str]]:
    """Merged map with normalised BnMMLU keys and normalised MMLU values."""
    merged: Dict[str, List[str]] = {}
    for src in (SUBJECT_MAP_BN_TO_MMLU, SUBJECT_MAP_EXTRA):
        for k, v in src.items():
            merged[normalize_subject(k)] = [normalize_subject(x) for x in v]
    return merged


def items_from_dataframe(
    df: pd.DataFrame,
    language: str = "bn",
    benchmark: str = "BnMMLU",
    index_base: Optional[int] = None,
) -> List[Dict[str, Any]]:
    """Harmonise a DataFrame (from the Hub, parquet, csv or json) into MCQ item dicts."""
    Q_KEYS = ["question", "question_bn", "question_en"]
    O_KEYS = ["options", "options_bn", "options_en", "choices"]
    A_KEYS = ["correct_answer", "answer", "answer_idx", "correct"]
    for name, keys in (("question", Q_KEYS), ("options", O_KEYS), ("answer", A_KEYS)):
        if not any(k in df.columns for k in keys):
            raise ValueError(f"No {name} column found (looked for {keys}). Columns present: {list(df.columns)}")

    def _get_row_field(row, keys, default=None):
        for k in keys:
            if k in row:
                val = row[k]
                if val is not None:
                    if isinstance(val, float) and pd.isna(val):
                        continue
                    return val
        return default

    items = []
    for idx, row in df.iterrows():
        qid = _get_row_field(row, ["Unique_Serial", "question_id", "id", "item_id"], str(idx))
        subject = _get_row_field(row, ["subject_name", "subject", "category"], "general")
        question = _get_row_field(row, Q_KEYS, "")
        raw_options = _get_row_field(row, O_KEYS, [])
        raw_ans = _get_row_field(row, A_KEYS, "")
        subject_group = _get_row_field(row, ["subject_group"], None)

        options = parse_options(raw_options)
        correct_letter, answer_idx = parse_answer_label(raw_ans, index_base=index_base)

        items.append({
            "question_id": str(qid),
            "subject": str(subject),
            "subject_group": (str(subject_group) if subject_group is not None else normalize_subject(subject)),
            "question": str(question),
            "options": options,
            "correct_answer": correct_letter,
            "answer_idx": answer_idx,
            "language": language,
            "benchmark": benchmark,
            "has_markup": contains_markup(str(question), options),
            "raw_index": int(idx) if isinstance(idx, (int, np.integer)) else len(items),
        })

    # Duplicate question_ids would silently break resume logic and merges: make them unique
    ids = [it["question_id"] for it in items]
    if len(set(ids)) != len(ids):
        n_dup = len(ids) - len(set(ids))
        logger.warning(f"{n_dup:,} duplicate question_ids detected. Appending raw_index to make IDs unique.")
        for it in items:
            it["question_id"] = f"{it['question_id']}_{it['raw_index']}"

    logger.info(f"Harmonised {len(items):,} {benchmark} items ({sum(it['has_markup'] for it in items):,} contain MathML markup).")
    return items


def load_dataset_from_file(
    file_path: str,
    language: str = "bn",
    benchmark: str = "BnMMLU",
    index_base: Optional[int] = None,
    is_mock: bool = False,
) -> List[Dict[str, Any]]:
    """Load and harmonize dataset from Parquet, CSV, or JSONL. Synthetic data fallback is strictly prohibited."""
    if STRICT_REAL_MODE or not is_mock:
        if file_path == "FALLBACK_SYNTHETIC" or not os.path.exists(file_path):
            raise FileNotFoundError(
                f"Dataset file '{file_path}' not found. STRICT_REAL_MODE is active and authentic benchmark data is required. "
                "Silent fallback to synthetic or mock data is strictly prohibited."
            )
    else:
        if file_path == "FALLBACK_SYNTHETIC" or not os.path.exists(file_path):
            logger.info(f"Isolated unit test: Using synthetic fixtures (language='{language}').")
            items = [dict(it) for it in create_synthetic_benchmark_dataset() if it["language"] == language]
            for it in items:
                it.setdefault("subject_group", normalize_subject(it["subject"]))
                it.setdefault("has_markup", False)
            return items

    logger.info(f"Loading dataset from disk: {file_path}")
    if file_path.endswith(".parquet"):
        df = pd.read_parquet(file_path)
    elif file_path.endswith(".csv"):
        df = pd.read_csv(file_path, encoding="utf-8")
    elif file_path.endswith(".json") or file_path.endswith(".jsonl"):
        df = pd.read_json(file_path, lines=file_path.endswith(".jsonl"))
    else:
        raise ValueError(f"Unsupported file format for: {file_path}")
    return items_from_dataframe(df, language=language, benchmark=benchmark, index_base=index_base)


def sample_stratified_subset(
    items: List[Dict[str, Any]],
    n_samples: int,
    seed: int = DEFAULT_SEED,
) -> List[Dict[str, Any]]:
    """Reproducibly extract a stratified random sample across benchmark subjects."""
    if n_samples <= 0 or n_samples >= len(items):
        return items

    rng = np.random.RandomState(seed)
    by_subject: Dict[str, List[int]] = {}
    for i, it in enumerate(items):
        subj = it.get("subject", "general")
        by_subject.setdefault(subj, []).append(i)

    total_len = len(items)
    selected_indices: List[int] = []

    for subj, indices in by_subject.items():
        prop = len(indices) / total_len
        k = int(round(prop * n_samples))
        if k > 0 and indices:
            chosen = rng.choice(indices, size=min(k, len(indices)), replace=False).tolist()
            selected_indices.extend(chosen)

    # Adjust rounding differences
    if len(selected_indices) < n_samples:
        remaining = list(set(range(len(items))) - set(selected_indices))
        if remaining:
            needed = min(n_samples - len(selected_indices), len(remaining))
            extra = rng.choice(remaining, size=needed, replace=False).tolist()
            selected_indices.extend(extra)
    elif len(selected_indices) > n_samples:
        selected_indices = rng.choice(selected_indices, size=n_samples, replace=False).tolist()

    selected_indices.sort()
    return [items[i] for i in selected_indices]


## Section 5: Data Preparation (real BnMMLU & subject-matched real MMLU)
Prepares the standardized subsets (`bn_subset.csv`, `en_subset.csv`) of up to $N=1500$ items per language:
1. **BnMMLU:** `load_dataset("samanjoy2/BnMMLU", "full", token=...)` (gated). Items are validated (4 options, valid answer), optionally stripped of MathML items, and stratified-sampled with seed 42.
2. **MMLU:** `cais/mmlu`, config `all`, split `test`. English subjects are matched to BnMMLU subjects through the explicit `SUBJECT_MAP_BN_TO_MMLU` (Section 4); the matched BnMMLU label is stored as `subject_group` in **both** files so the paired Bangla-vs-English bootstrap works.
3. **Provenance:** `subset_meta.json` records the Hub commit SHAs of both datasets, `subject_mapping.json` records the per-subject mapping and which BnMMLU subjects have no MMLU analogue, and `sampled_question_ids.json` locks the exact items for every model.

> [!NOTE]
> **Cross-lingual control framing:** the English subset is a **subject-matched control**, not an item-parallel translation. Subject matching is approximate (Bangladeshi school/exam content vs. US-oriented MMLU) and is a design choice to justify in the paper.
>
> The Hub card marks this BnMMLU release as *preliminary / non-final*. Report the recorded commit SHA.


In [ ]:
import hashlib


def _hub_revision(repo_id: str, repo_type: str, token: Optional[str]) -> str:
    """Commit SHA of a Hub repo, recorded so the exact data version can be reported in the paper."""
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        info = api.dataset_info(repo_id) if repo_type == "dataset" else api.model_info(repo_id)
        return str(info.sha)
    except Exception as e:
        return f"unavailable ({type(e).__name__})"


def _explain_hub_failure(repo_id: str, err: Exception) -> RuntimeError:
    msg = str(err)
    hint = ""
    low = msg.lower()
    if "gated" in low or "401" in low or "403" in low or "access" in low or "restricted" in low or "authenticat" in low:
        hint = (f"\n=> '{repo_id}' is gated. While logged in to Hugging Face, open https://huggingface.co/datasets/{repo_id} "
                "and accept the access conditions, then make sure HF_TOKEN is a valid READ token for the SAME account.")
    elif "config" in low:
        hint = f"\n=> '{repo_id}' needs an explicit config name. Set CONFIG.bn_hf_config to 'full' or 'hard'."
    return RuntimeError(f"Could not load '{repo_id}' from the Hugging Face Hub: {type(err).__name__}: {msg}{hint}")


def load_bnmmlu_items(
    config: "ExperimentConfig",
    hf_token: Optional[str],
    is_mock: bool,
    bn_index_base: Optional[int],
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Load the REAL BnMMLU (Hub config 'full' by default) or a local copy. Strict real execution enforced."""
    if STRICT_REAL_MODE or not is_mock:
        if is_mock:
            raise RuntimeError("Mock mode is disabled when STRICT_REAL_MODE=True. Authentic BnMMLU dataset is required.")
    else:
        items = load_dataset_from_file("FALLBACK_SYNTHETIC", "bn", "BnMMLU", is_mock=True)
        return items, {"source": "synthetic_mock"}

    if config.bn_local_path:
        if not os.path.exists(config.bn_local_path):
            raise FileNotFoundError(f"CONFIG.bn_local_path does not exist: {config.bn_local_path}")
        items = load_dataset_from_file(config.bn_local_path, "bn", "BnMMLU", index_base=bn_index_base, is_mock=False)
        return items, {"source": f"local:{config.bn_local_path}"}

    try:
        from datasets import load_dataset
        tok = {"token": hf_token} if hf_token else {}
        logger.info(f"Downloading BnMMLU from the Hub: {config.bn_dataset_id} (config='{config.bn_hf_config}')...")
        dd = load_dataset(config.bn_dataset_id, config.bn_hf_config, **tok)
    except Exception as e:
        raise _explain_hub_failure(config.bn_dataset_id, e)

    splits = list(dd.keys())
    if "test" in splits:
        use = ["test"]
    elif len(splits) == 1:
        use = splits
    else:
        use = splits  # several splits and no 'test': use all rows (zero-shot evaluation needs no train/test separation)
    logger.info(f"BnMMLU splits available: {splits} -> using {use}")
    df = pd.concat([dd[s].to_pandas() for s in use], ignore_index=True)
    logger.info(f"BnMMLU columns: {list(df.columns)} | rows: {len(df):,}")
    logger.info(f"BnMMLU distinct subjects: {df['subject_name'].nunique() if 'subject_name' in df.columns else 'n/a'}")
    items = items_from_dataframe(df, language="bn", benchmark="BnMMLU", index_base=bn_index_base)
    return items, {"source": f"hub:{config.bn_dataset_id}:{config.bn_hf_config}", "splits_used": use, "n_rows": int(len(df))}


def load_mmlu_items(config: "ExperimentConfig", hf_token: Optional[str], is_mock: bool, n_items: int) -> List[Dict[str, Any]]:
    """Load the REAL English MMLU (cais/mmlu, config 'all', split 'test'). Strict real execution enforced."""
    if STRICT_REAL_MODE or not is_mock:
        if is_mock:
            raise RuntimeError("Mock mode is disabled when STRICT_REAL_MODE=True. Authentic MMLU dataset is required.")
    else:
        synth_en = load_dataset_from_file("FALLBACK_SYNTHETIC", "en", "MMLU", is_mock=True)
        return [dict(it, question_id=f"{it['question_id']}_{k}") for k in range(n_items // 4 + 1) for it in synth_en]

    try:
        from datasets import load_dataset
        tok = {"token": hf_token} if hf_token else {}
        logger.info(f"Downloading English MMLU from the Hub: {config.en_dataset_id} ('{config.en_hf_config}', split='{config.en_split}')...")
        hf_en = load_dataset(config.en_dataset_id, config.en_hf_config, split=config.en_split, **tok)
    except Exception as e:
        raise _explain_hub_failure(config.en_dataset_id, e)

    items = []
    for idx, row in enumerate(hf_en):
        choices = [str(c) for c in row["choices"]]
        ans = int(row["answer"])
        subj = str(row.get("subject", "general"))
        q = str(row["question"])
        items.append({
            "question_id": f"MMLU_EN_{idx:05d}",
            "subject": subj,
            "subject_group": normalize_subject(subj),
            "question": q,
            "options": choices,
            "correct_answer": OPTION_LETTERS[ans] if 0 <= ans < 4 else "",
            "answer_idx": ans,
            "language": "en",
            "benchmark": "MMLU",
            "has_markup": contains_markup(q, choices),
            "raw_index": idx,
        })
    logger.info(f"Loaded {len(items):,} English MMLU items across {len({it['subject'] for it in items})} subjects.")
    return items



EXEMPLAR_VERSION = "v3_1_fewshot_v2"

def select_fewshot_exemplars(
    candidates: List[Dict[str, Any]],
    eval_ids: Union[set, List[str]],
    language: str,
    seed: int = DEFAULT_SEED,
) -> List[Dict[str, Any]]:
    """Select exactly 8 balanced few-shot exemplars disjoint from evaluation items.

    Each letter A-D appears exactly twice across the 8 exemplars, and any prefix of 4
    contains each letter A-D exactly once.
    Constraints on each exemplar:
      - 4 options
      - Valid gold label in {'A', 'B', 'C', 'D'}
      - has_markup is False (no MathML)
      - total character length (question + joined options) <= 350
      - question_id not in eval_ids
      - distinct subjects where possible
      - deterministic selection using fixed seed
    """
    eval_set = set(str(qid) for qid in eval_ids)

    # Filter eligible items
    eligible: List[Dict[str, Any]] = []
    for it in candidates:
        qid = str(it.get("question_id", ""))
        if qid in eval_set:
            continue
        gold = str(it.get("correct_answer", "")).upper()
        if gold not in OPTION_LETTERS:
            continue
        opts = it.get("options", [])
        if not isinstance(opts, list) or len(opts) != 4:
            continue
        if it.get("has_markup", False):
            continue
        q_text = str(it.get("question", ""))
        opts_text = "".join(str(o) for o in opts)
        if len(q_text) + len(opts_text) > 350:
            continue
        eligible.append(it)

    rng = np.random.RandomState(seed)
    shuffled_idx = rng.permutation(len(eligible))
    shuffled = [eligible[i] for i in shuffled_idx]

    # Pool by gold letter
    by_letter: Dict[str, List[Dict[str, Any]]] = {letter: [] for letter in OPTION_LETTERS}
    for it in shuffled:
        by_letter[str(it.get("correct_answer", "")).upper()].append(it)

    # Group 1: 1 of each letter A-D
    # Group 2: 1 of each letter A-D
    # Select with subject diversity where possible
    chosen_1: List[Dict[str, Any]] = []
    chosen_2: List[Dict[str, Any]] = []
    used_ids: set = set()
    used_subjects: set = set()

    for letter in OPTION_LETTERS:
        pool = by_letter[letter]
        pick = None
        # Try distinct subject
        for it in pool:
            subj = it.get("subject", "")
            if subj not in used_subjects:
                pick = it
                break
        if pick is None and pool:
            pick = pool[0]
        if pick is None:
            raise RuntimeError(f"Cannot build balanced few-shot set for {language}: insufficient items for letter {letter}")
        chosen_1.append(pick)
        used_ids.add(str(pick.get("question_id", "")))
        if pick.get("subject"):
            used_subjects.add(pick.get("subject"))

    for letter in OPTION_LETTERS:
        pool = [it for it in by_letter[letter] if str(it.get("question_id", "")) not in used_ids]
        pick = None
        for it in pool:
            subj = it.get("subject", "")
            if subj not in used_subjects:
                pick = it
                break
        if pick is None and pool:
            pick = pool[0]
        if pick is None:
            raise RuntimeError(f"Cannot build balanced few-shot set for {language}: insufficient items for second pass of letter {letter}")
        chosen_2.append(pick)
        used_ids.add(str(pick.get("question_id", "")))
        if pick.get("subject"):
            used_subjects.add(pick.get("subject"))

    # Verify no eval_ids overlap
    overlap = used_ids.intersection(eval_set)
    if overlap:
        raise RuntimeError(f"Few-shot exemplars overlap with evaluation items: {overlap}")

    # Build standardized exemplar dicts
    def _standardize(it):
        return {
            "question_id": str(it.get("question_id", "")),
            "subject": it.get("subject", ""),
            "question": str(it.get("question", "")),
            "options": [str(o) for o in it.get("options", [])],
            "correct_answer": str(it.get("correct_answer", "")).upper(),
            "source": it.get("benchmark", language),
        }

    return [_standardize(it) for it in chosen_1] + [_standardize(it) for it in chosen_2]

def prepare_research_subsets(
    output_dir: str = OUTPUT_DIR,
    n_items: int = 1500,
    seed: int = DEFAULT_SEED,
    hf_token: Optional[str] = None,
    is_mock: bool = False,
    bn_index_base: Optional[int] = None,
    config: Optional["ExperimentConfig"] = None,
) -> Tuple[str, str]:
    """Prepare standardised, subject-matched Bangla (BnMMLU) and English (MMLU) subsets."""
    config = config or CONFIG
    bn_subset_path = os.path.join(output_dir, "bn_subset.csv")
    en_subset_path = os.path.join(output_dir, "en_subset.csv")
    dev_bn_path = os.path.join(output_dir, "dev_bn.csv")
    dev_en_path = os.path.join(output_dir, "dev_en.csv")
    sampled_ids_path = os.path.join(output_dir, "sampled_question_ids.json")
    mapping_path = os.path.join(output_dir, "subject_mapping.json")
    meta_path = os.path.join(output_dir, "subset_meta.json")

    subj_map = build_subject_map()
    map_hash = hashlib.md5(json.dumps(subj_map, sort_keys=True).encode("utf-8")).hexdigest()[:12]
    dev_n = getattr(config, "dev_n", 200)
    expected_meta = {
        "n_items": int(n_items), "seed": int(seed), "is_mock": bool(is_mock),
        "bn_source": config.bn_local_path or f"{config.bn_dataset_id}:{config.bn_hf_config}",
        "en_source": f"{config.en_dataset_id}:{config.en_hf_config}:{config.en_split}",
        "exclude_markup": bool(config.exclude_markup),
        "subject_match_mode": config.subject_match_mode,
        "subject_map_hash": map_hash,
        "n_shots_max": 8,
        "exemplar_version": EXEMPLAR_VERSION,
        "sampling_version": "v3_3_disjoint_english",
        "dev_n": int(dev_n),
        "dev_version": "v4a_dev",
    }

    # Reuse prepared subsets ONLY if they were built with identical settings (never reuse stale/mock subsets)
    if all(os.path.exists(p) for p in (bn_subset_path, en_subset_path, dev_bn_path, dev_en_path, meta_path)):
        try:
            with open(meta_path, "r", encoding="utf-8") as f:
                saved_meta = json.load(f)
        except Exception:
            saved_meta = {}
        if {k: saved_meta.get(k) for k in expected_meta} == expected_meta:
            logger.info(f"Found existing prepared subsets (settings match):\n   - {bn_subset_path}\n   - {en_subset_path}")
            return bn_subset_path, en_subset_path
        logger.warning("Existing subsets were built with different settings. Rebuilding.")

    if config.subject_match_mode not in ("intersection", "all_bn"):
        raise ValueError("CONFIG.subject_match_mode must be 'intersection' or 'all_bn'.")

    logger.info(f"Preparing {n_items}-item research subsets for BnMMLU and MMLU...")

    # ---------------- 1. LOAD REAL DATA ----------------
    bn_raw, bn_info = load_bnmmlu_items(config, hf_token, is_mock, bn_index_base)
    en_raw = load_mmlu_items(config, hf_token, is_mock, n_items)

    bn_valid = validate_items(bn_raw)
    en_valid = validate_items(en_raw)

    if config.exclude_markup:
        n_bn_before, n_en_before = len(bn_valid), len(en_valid)
        bn_valid = [it for it in bn_valid if not it.get("has_markup")]
        en_valid = [it for it in en_valid if not it.get("has_markup")]
        logger.info(f"Markup filter: BnMMLU {n_bn_before:,} -> {len(bn_valid):,} items "
                    f"({n_bn_before - len(bn_valid):,} MathML items removed); MMLU {n_en_before:,} -> {len(en_valid):,}.")

    # Make the real BnMMLU subject list visible so the subject map can be checked/extended
    bn_subj_counts = pd.Series([it["subject_group"] for it in bn_valid]).value_counts()
    logger.info(f"BnMMLU subjects after filtering ({len(bn_subj_counts)}): {bn_subj_counts.to_dict()}")

    # MMLU pools keyed by normalised subject
    en_pool: Dict[str, List[Dict[str, Any]]] = {}
    for it in en_valid:
        en_pool.setdefault(it["subject_group"], []).append(it)
    if not is_mock:
        bad = sorted({m for v in subj_map.values() for m in v} - set(en_pool))
        if bad:
            logger.warning(f"Subject map references MMLU subjects that were not found: {bad}")

    def mmlu_names_for(group: str) -> List[str]:
        names = list(subj_map.get(group, []))
        if group in en_pool and group not in names:   # identical names also match (used by mock data / custom names)
            names.append(group)
        return [n for n in names if n in en_pool]

    rng = np.random.RandomState(seed)

    # ---------------- 2. SAMPLE ----------------
    mapped_groups = {g for g in bn_subj_counts.index if mmlu_names_for(g)}
    unmapped = {g: int(c) for g, c in bn_subj_counts.items() if g not in mapped_groups}
    if unmapped:
        logger.info(f"BnMMLU subjects with NO MMLU analogue ({len(unmapped)}): {unmapped}")

    mapped_pool_counts = {
        g: {"n_bn": int(bn_subj_counts.get(g, 0)), "n_en": sum(len(en_pool[n]) for n in mmlu_names_for(g))}
        for g in sorted(mapped_groups)
    }
    logger.info(f"Mapped subject groups available for sampling ({len(mapped_groups)}): {mapped_pool_counts}")

    if config.subject_match_mode == "intersection":
        candidates = [it for it in bn_valid if it["subject_group"] in mapped_groups]
        if not candidates:
            raise RuntimeError(
                "No BnMMLU subject could be mapped to an MMLU subject. Compare the BnMMLU subject list logged above with "
                "SUBJECT_MAP_BN_TO_MMLU (Section 4) and add entries to SUBJECT_MAP_EXTRA."
            )

        # Pre-group available valid Bangla candidates
        bn_cand_by_group: Dict[str, List[Dict[str, Any]]] = {}
        for it in candidates:
            bn_cand_by_group.setdefault(it["subject_group"], []).append(it)

        # Available pairs per mapped group
        available_pairs = {
            g: min(len(bn_cand_by_group.get(g, [])), len([it for n in mmlu_names_for(g) for it in en_pool[n]]))
            for g in sorted(mapped_groups)
        }
        total_available_pairs = sum(available_pairs.values())

        if total_available_pairs < n_items:
            pairs_per_subj = {g: f"{available_pairs[g]}/{available_pairs[g]}" for g in sorted(mapped_groups)}
            raise RuntimeError(
                f"Fewer than {n_items} valid, subject-matched Bangla-English samples available (found {total_available_pairs}). "
                f"Subject-matched pairs per mapped subject (n_bn/n_en): {pairs_per_subj}. "
                f"STRICT_REAL_MODE prohibits silently reducing the sample count below exactly {n_items} items."
            )

        n_mapped_groups = len(mapped_groups)
        per_subject_cap = math.ceil(n_items / n_mapped_groups * 1.5)

        # Balanced capped allocation with leftover redistribution
        alloc: Dict[str, int] = {g: 0 for g in sorted(mapped_groups)}
        target_remaining = n_items

        # Pass 1: allocate up to min(available_pairs, per_subject_cap) near-equally
        while target_remaining > 0:
            eligible = [g for g in sorted(mapped_groups) if alloc[g] < min(available_pairs[g], per_subject_cap)]
            if not eligible:
                break
            step = max(1, target_remaining // len(eligible))
            for g in sorted(eligible):
                room = min(available_pairs[g], per_subject_cap) - alloc[g]
                add = min(room, step, target_remaining)
                alloc[g] += add
                target_remaining -= add
                if target_remaining == 0:
                    break

        # Pass 2: redistribute leftover quota to groups that still have spare pairs
        while target_remaining > 0:
            eligible = [g for g in sorted(mapped_groups) if alloc[g] < available_pairs[g]]
            if not eligible:
                break
            perm = rng.permutation(len(eligible))
            for idx in perm:
                g = eligible[idx]
                alloc[g] += 1
                target_remaining -= 1
                if target_remaining == 0:
                    break

        # ---- DISJOINT English sampling ----------------------------------------------------------------------
        # Several BnMMLU subjects share MMLU subjects (e.g. high_school_biology serves 'cell biology genetics' AND
        # 'ecology environmental biology'). Sampling each group independently would pick the SAME English question for
        # several groups, duplicating items. Every English item is therefore used AT MOST ONCE across all groups:
        # scarcest pools are filled first and any shortfall is redistributed to groups that still have unused
        # Bangla AND English items.
        used_en_ids: set = set()
        full_pool: Dict[str, List[Dict[str, Any]]] = {
            g: [it for n in mmlu_names_for(g) for it in en_pool[n]] for g in sorted(mapped_groups)
        }
        en_pick_by_group: Dict[str, List[Dict[str, Any]]] = {g: [] for g in sorted(mapped_groups)}

        def _unused_en(g: str) -> List[Dict[str, Any]]:
            return [it for it in full_pool[g] if it["question_id"] not in used_en_ids]

        for g in sorted(mapped_groups, key=lambda x: (len(full_pool[x]), x)):
            avail = _unused_en(g)
            take = min(alloc[g], len(avail))
            if take > 0:
                for j in sorted(rng.choice(len(avail), size=take, replace=False).tolist()):
                    en_pick_by_group[g].append(avail[j])
                    used_en_ids.add(avail[j]["question_id"])

        shortfall = n_items - sum(len(v) for v in en_pick_by_group.values())
        while shortfall > 0:
            spare = [
                g for g in sorted(mapped_groups)
                if len(en_pick_by_group[g]) < len(bn_cand_by_group.get(g, [])) and _unused_en(g)
            ]
            if not spare:
                break
            g = min(spare, key=lambda x: (len(en_pick_by_group[x]), x))
            avail = _unused_en(g)
            j = int(rng.randint(len(avail)))
            en_pick_by_group[g].append(avail[j])
            used_en_ids.add(avail[j]["question_id"])
            shortfall -= 1

        if shortfall > 0:
            pairs_per_subj = {
                g: f"{len(bn_cand_by_group.get(g, []))}/{len(full_pool[g])}" for g in sorted(mapped_groups)
            }
            raise RuntimeError(
                f"Fewer than {n_items} valid, subject-matched Bangla-English samples available "
                f"(found {n_items - shortfall} with every English item used at most once). "
                f"Subject-matched pairs per mapped subject (n_bn/n_en, English pools are shared between groups): {pairs_per_subj}. "
                f"STRICT_REAL_MODE prohibits silently reducing the sample count below exactly {n_items} items."
            )

        bn_final: List[Dict[str, Any]] = []
        en_final: List[Dict[str, Any]] = []
        mapping_report: Dict[str, Any] = {}
        for group in sorted(mapped_groups):
            names = mmlu_names_for(group)
            en_pick = [dict(it, subject_group=group) for it in en_pick_by_group[group]]
            k = len(en_pick)
            if k == 0:
                mapping_report[group] = {"mmlu_subjects": names, "n_bn": 0, "n_en": 0, "n_bn_available": len(bn_cand_by_group.get(group, [])),
                                         "n_en_available": len(full_pool[group]), "quota": int(alloc[group])}
                continue
            bn_items_g = bn_cand_by_group[group]
            bn_idx = sorted(rng.choice(len(bn_items_g), size=k, replace=False).tolist())
            bn_final.extend(bn_items_g[j] for j in bn_idx)
            en_final.extend(en_pick)
            mapping_report[group] = {"mmlu_subjects": names, "n_bn": k, "n_en": k, "n_bn_available": len(bn_items_g),
                                     "n_en_available": len(full_pool[group]), "quota": int(alloc[group])}

        used_groups_count = sum(1 for g in mapped_groups if en_pick_by_group[g])
        logger.info(
            f"Balanced subject sampling complete: {len(bn_final)} paired items across {used_groups_count} subject groups "
            f"(per-subject cap={per_subject_cap}, every English item used at most once). "
            f"Allocation: { {g: len(v) for g, v in en_pick_by_group.items()} }"
        )
    else:
        bn_sampled = sample_stratified_subset(bn_valid, n_samples=n_items, seed=seed)

        # ---------------- 3. MATCH ENGLISH ITEMS PER SUBJECT ----------------
        bn_by_group: Dict[str, List[Dict[str, Any]]] = {}
        for it in bn_sampled:
            bn_by_group.setdefault(it["subject_group"], []).append(it)

        bn_final: List[Dict[str, Any]] = []
        en_final: List[Dict[str, Any]] = []
        mapping_report: Dict[str, Any] = {}
        used_en_ids: set = set()

        for group, bn_items_g in sorted(bn_by_group.items()):
            names = mmlu_names_for(group)
            pool = [it for n in names for it in en_pool[n] if it["question_id"] not in used_en_ids]
            if not pool:
                bn_final.extend(bn_items_g)
                mapping_report[group] = {"mmlu_subjects": [], "n_bn": len(bn_items_g), "n_en": 0}
                continue
            k = min(len(bn_items_g), len(pool))
            bn_keep = bn_items_g
            en_idx = rng.choice(len(pool), size=k, replace=False).tolist()
            en_pick = [dict(pool[j], subject_group=group) for j in en_idx]
            used_en_ids.update(it["question_id"] for it in en_pick)
            bn_final.extend(bn_items_g)
            en_final.extend(en_pick)
            mapping_report[group] = {"mmlu_subjects": names, "n_bn": len(bn_items_g), "n_en": k}

    if config.subject_match_mode == "all_bn":
        target_n = len(bn_final)
        if len(en_final) < target_n:
            remaining = [it for it in en_valid if it["question_id"] not in used_en_ids]
            needed = min(target_n - len(en_final), len(remaining))
            if needed > 0:
                extra = rng.choice(len(remaining), size=needed, replace=False).tolist()
                en_final.extend(dict(remaining[j], subject_group="unmatched") for j in extra)
                logger.info(f"Topped up English subset with {needed} subject-UNMATCHED items (subject_group='unmatched').")

    if len(bn_final) != len(en_final):
        raise RuntimeError(f"Subset size mismatch: Bangla has {len(bn_final)} items, English has {len(en_final)} items.")
    if len(bn_final) < n_items:
        pairs_per_subj = {g: f"{rep['n_bn']}/{rep['n_en']}" for g, rep in sorted(mapping_report.items())}
        raise RuntimeError(
            f"Fewer than {n_items} valid, subject-matched Bangla-English samples available (found {len(bn_final)}). "
            f"Subject-matched pairs per mapped subject (n_bn/n_en): {pairs_per_subj}. "
            f"STRICT_REAL_MODE prohibits silently reducing the sample count below exactly {n_items} items."
        )
    # Ensure exactly n_items in final paired sets
    if len(bn_final) > n_items:
        bn_final = bn_final[:n_items]
        en_final = en_final[:n_items]

    # ---------------- 4. SAVE ----------------
    def _save(items: List[Dict[str, Any]], path: str) -> pd.DataFrame:
        out = pd.DataFrame(items)
        out["options"] = out["options"].apply(lambda opts: json.dumps(opts, ensure_ascii=False))
        out.to_csv(path, index=False, encoding="utf-8")
        return out

    bn_df = _save(bn_final, bn_subset_path)
    en_df = _save(en_final, en_subset_path)
    logger.info(f"Saved Bangla subset: {bn_subset_path} ({len(bn_df):,} items)")
    logger.info(f"Saved English subset: {en_subset_path} ({len(en_df):,} items)")
    if not en_df["question_id"].is_unique or not bn_df["question_id"].is_unique:
        raise RuntimeError("Prepared subsets contain duplicate question_ids (an item was selected more than once).")

    with open(mapping_path, "w", encoding="utf-8") as f:
        json.dump({"mode": config.subject_match_mode, "per_subject": mapping_report, "unmapped_bn_subjects": unmapped,
                   "coverage_of_bn_pool": round(sum(int(bn_subj_counts.get(g, 0)) for g in mapped_groups) / max(1, len(bn_valid)), 4),
                   "n_unique_en_ids": int(en_df["question_id"].nunique()), "n_unique_bn_ids": int(bn_df["question_id"].nunique())},
                  f, indent=2, ensure_ascii=False)

    meta_out = dict(expected_meta)
    meta_out["data_provenance"] = {
        "bn": bn_info,
        "bn_hub_revision": "n/a (local/mock)" if (is_mock or config.bn_local_path) else _hub_revision(config.bn_dataset_id, "dataset", hf_token),
        "en_hub_revision": "n/a (mock)" if is_mock else _hub_revision(config.en_dataset_id, "dataset", hf_token),
        "n_bn": int(len(bn_df)), "n_en": int(len(en_df)),
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta_out, f, indent=2, ensure_ascii=False)

    with open(sampled_ids_path, "w", encoding="utf-8") as f:
        json.dump({"bn_question_ids": bn_df["question_id"].tolist(), "en_question_ids": en_df["question_id"].tolist()}, f, indent=2)
    logger.info(f"Sampled question IDs locked to: {sampled_ids_path} | subject mapping: {mapping_path}")

    # Build and persist fewshot_exemplars.json
    fewshot_file = os.path.join(output_dir, "fewshot_exemplars.json")
    bn_eval_set = set(bn_df["question_id"].astype(str).tolist())
    en_eval_set = set(en_df["question_id"].astype(str).tolist())

    # Bangla candidates: valid BnMMLU items outside evaluation subset
    bn_candidates = [it for it in bn_valid if str(it.get("question_id", "")) not in bn_eval_set]

    # English candidates: dev split of cais/mmlu (or unused valid en items)
    en_candidates = []
    english_exemplar_source = "test_pool_fallback"
    allow_fallback = getattr(config, "allow_fewshot_test_pool_fallback", False)

    if not is_mock and hf_token:
        try:
            logger.info("Loading English MMLU dev split for few-shot exemplars...")
            from datasets import load_dataset
            en_dev_ds = load_dataset(config.en_dataset_id, "all", split="dev", token=hf_token)
            en_dev_raw = []
            for row_idx, r in enumerate(en_dev_ds):
                sub = str(r.get("subject", "general"))
                choices = list(r.get("choices", []))
                raw_ans = r.get("answer", 0)
                ans_idx = int(raw_ans)
                q_text = str(r.get("question", ""))
                has_markup = contains_markup(q_text, choices)

                en_dev_raw.append({
                    "question_id": f"en_dev_{row_idx}",
                    "subject": sub,
                    "subject_group": normalize_subject(sub),
                    "question": q_text,
                    "options": choices,
                    "correct_answer": OPTION_LETTERS[ans_idx],
                    "answer_idx": ans_idx,
                    "language": "en",
                    "benchmark": "MMLU",
                    "has_markup": has_markup,
                    "raw_index": row_idx,
                })
            en_candidates = validate_items(en_dev_raw)
            english_exemplar_source = "mmlu_dev"
        except Exception as e:
            if STRICT_REAL_MODE and not allow_fallback:
                logger.error(f"Failed to load English MMLU dev split under STRICT_REAL_MODE: {e}")
                raise RuntimeError(f"Could not load English MMLU dev split under STRICT_REAL_MODE (fallback disabled): {e}") from e
            logger.warning(f"Could not load MMLU dev split ({e}); falling back to unused test pool.")
            en_candidates = [it for it in en_valid if str(it.get("question_id", "")) not in en_eval_set]
            english_exemplar_source = "test_pool_fallback"
    else:
        en_candidates = [it for it in en_valid if str(it.get("question_id", "")) not in en_eval_set]
        english_exemplar_source = "test_pool_fallback"

    bn_exemplars = select_fewshot_exemplars(bn_candidates, bn_eval_set, language="bn", seed=seed)
    en_exemplars = select_fewshot_exemplars(en_candidates, en_eval_set, language="en", seed=seed)

    exemplars_payload = {
        "bn": bn_exemplars,
        "en": en_exemplars,
        "version": EXEMPLAR_VERSION,
        "n_shots_max": 8,
        "english_exemplar_source": english_exemplar_source,
    }
    with open(fewshot_file, "w", encoding="utf-8") as f:
        json.dump(exemplars_payload, f, indent=2, ensure_ascii=False)
    logger.info(f"Few-shot exemplars persisted to: {fewshot_file} (bn: {len(bn_exemplars)}, en: {len(en_exemplars)})")

    # ---------------- 5. DEV POOL CREATION (dev_bn.csv, dev_en.csv) ----------------
    bn_ex_ids = {str(it.get("question_id", "")) for it in bn_exemplars}
    en_ex_ids = {str(it.get("question_id", "")) for it in en_exemplars}

    if dev_n > 0:
        bn_dev_candidates = [it for it in bn_valid if str(it.get("question_id", "")) not in bn_eval_set and str(it.get("question_id", "")) not in bn_ex_ids]
        en_dev_candidates = [it for it in en_valid if str(it.get("question_id", "")) not in en_eval_set and str(it.get("question_id", "")) not in en_ex_ids]

        if len(bn_dev_candidates) < dev_n:
            raise RuntimeError(f"Fewer than {dev_n} valid items remain for Bangla dev pool (found {len(bn_dev_candidates)}).")
        if len(en_dev_candidates) < dev_n:
            raise RuntimeError(f"Fewer than {dev_n} valid items remain for English dev pool (found {len(en_dev_candidates)}).")

        # Separate RandomState(seed + 7) for dev pool sampling
        dev_rng = np.random.RandomState(seed + 7)

        bn_dev_idx = sorted(dev_rng.choice(len(bn_dev_candidates), size=dev_n, replace=False).tolist())
        en_dev_idx = sorted(dev_rng.choice(len(en_dev_candidates), size=dev_n, replace=False).tolist())

        bn_dev_items = [bn_dev_candidates[i] for i in bn_dev_idx]
        en_dev_items = [en_dev_candidates[i] for i in en_dev_idx]

        bn_dev_ids = {str(it.get("question_id", "")) for it in bn_dev_items}
        en_dev_ids = {str(it.get("question_id", "")) for it in en_dev_items}

        # Assert eval, exemplars, and dev are pairwise disjoint
        assert len(bn_eval_set & bn_ex_ids) == 0, "Bangla eval and exemplars overlap!"
        assert len(bn_eval_set & bn_dev_ids) == 0, "Bangla eval and dev overlap!"
        assert len(bn_ex_ids & bn_dev_ids) == 0, "Bangla exemplars and dev overlap!"

        assert len(en_eval_set & en_ex_ids) == 0, "English eval and exemplars overlap!"
        assert len(en_eval_set & en_dev_ids) == 0, "English eval and dev overlap!"
        assert len(en_ex_ids & en_dev_ids) == 0, "English exemplars and dev overlap!"

        # Save dev_bn.csv and dev_en.csv
        dev_bn_df = _save(bn_dev_items, dev_bn_path)
        dev_en_df = _save(en_dev_items, dev_en_path)
        logger.info(f"Saved Bangla dev set:  {dev_bn_path} ({len(dev_bn_df):,} items)")
        logger.info(f"Saved English dev set: {dev_en_path} ({len(dev_en_df):,} items)")

        bn_dev_subj_counts = {str(k): int(v) for k, v in pd.Series([it.get("subject_group", "") for it in bn_dev_items]).value_counts().items()}
        en_dev_subj_counts = {str(k): int(v) for k, v in pd.Series([it.get("subject_group", "") for it in en_dev_items]).value_counts().items()}
        n_bn_dev = len(dev_bn_df)
        n_en_dev = len(dev_en_df)
    else:
        # dev_n == 0: create empty dev files, mapping records 0
        empty_cols = ["question_id", "subject", "subject_group", "question", "options", "correct_answer", "answer_idx", "is_mock"]
        dev_bn_df = pd.DataFrame(columns=empty_cols)
        dev_en_df = pd.DataFrame(columns=empty_cols)
        dev_bn_df.to_csv(dev_bn_path, index=False, encoding="utf-8")
        dev_en_df.to_csv(dev_en_path, index=False, encoding="utf-8")
        bn_dev_subj_counts = {}
        en_dev_subj_counts = {}
        n_bn_dev = 0
        n_en_dev = 0

    # Add dev counts to subject_mapping.json
    with open(mapping_path, "r", encoding="utf-8") as f:
        mapping_data = json.load(f)
    mapping_data["dev_counts"] = {
        "dev_n": dev_n,
        "n_bn": n_bn_dev,
        "n_en": n_en_dev,
        "bn_by_subject": bn_dev_subj_counts,
        "en_by_subject": en_dev_subj_counts,
    }
    with open(mapping_path, "w", encoding="utf-8") as f:
        json.dump(mapping_data, f, indent=2, ensure_ascii=False)

    return bn_subset_path, en_subset_path


# Execute subset preparation
bn_subset_file, en_subset_file = prepare_research_subsets(
    output_dir=CONFIG.output_dir,
    n_items=CONFIG.n_samples,
    seed=CONFIG.seed,
    hf_token=CONFIG.hf_token,
    is_mock=(CONFIG.backend == "mock"),
    bn_index_base=CONFIG.bn_index_base,
    config=CONFIG,
)


## Section 6: Tokenization Fragmentation Metrics (TPW, STRR Isolated/In-Context, TPC)
Quantifies subword fragmentation:
1. **Tokens-per-Word (TPW / Fertility):** Whitespace word baseline.
2. **STRR (`isolated` and `in_context`):** Evaluates atomic retention with and without space prefixes.
3. **Tokens-per-Character (TPC):** Script-normalized character fertility.
4. **`compute_dataset_fragmentation`:** Lightweight profiling loading *only* tokenizers without model weights.


In [ ]:
def _tokenize_text(tokenizer: Any, text: str) -> List[Union[int, str]]:
    """Safely tokenize text across diverse tokenizer interfaces."""
    if not text:
        return []
    if hasattr(tokenizer, "encode"):
        try:
            return tokenizer.encode(text, add_special_tokens=False)
        except TypeError:
            return tokenizer.encode(text)
    elif hasattr(tokenizer, "tokenize"):
        return tokenizer.tokenize(text)
    elif callable(tokenizer):
        return tokenizer(text)
    else:
        return text.strip().split()


def extract_words(text: str) -> List[str]:
    """Extract words using whitespace splitting."""
    if not text:
        return []
    return text.strip().split()


def compute_fertility(tokenizer: Any, text: str) -> float:
    """Compute Tokens-per-Word (TPW / Fertility)."""
    words = extract_words(text)
    if not words:
        return 0.0
    tokens = _tokenize_text(tokenizer, text)
    return float(len(tokens) / len(words))


def compute_strr(tokenizer: Any, words_or_text: Union[List[str], str], strr_mode: str = "isolated") -> float:
    """Compute Single-Token Retention Rate (STRR).
    
    Modes:
    - 'isolated': tokenizes words in isolation.
    - 'in_context': tokenizes ' ' + word (accounting for BPE leading space prefixes).
    """
    words = extract_words(words_or_text) if isinstance(words_or_text, str) else words_or_text
    if not words:
        return 0.0

    single_token_count = 0
    for w in words:
        cand = f" {w}" if strr_mode == "in_context" else w
        toks = _tokenize_text(tokenizer, cand)
        if len(toks) == 1:
            single_token_count += 1

    return float(single_token_count / len(words))


def compute_char_fertility(tokenizer: Any, text: str, exclude_spaces: bool = True) -> float:
    """Compute Tokens-per-Character (TPC / Character Fertility)."""
    if not text:
        return 0.0
    chars = text.replace(" ", "") if exclude_spaces else text
    if not chars:
        return 0.0
    tokens = _tokenize_text(tokenizer, text)
    return float(len(tokens) / len(chars))


def compute_fragmentation_metrics(
    tokenizer: Any,
    question: str,
    options: List[str],
) -> Dict[str, Any]:
    """Calculate tokenization fragmentation metrics for an MCQ item (both isolated & in_context)."""
    q_text = str(question or "").strip()
    clean_options = [str(opt or "").strip() for opt in options]
    opts_text = " ".join(clean_options)
    full_text = (q_text + " " + opts_text).strip()

    # Question
    q_words = extract_words(q_text)
    q_tokens = _tokenize_text(tokenizer, q_text)
    q_char_count = len(q_text.replace(" ", ""))
    q_tpw = float(len(q_tokens) / len(q_words)) if q_words else 0.0
    q_strr = compute_strr(tokenizer, q_words, strr_mode="isolated")
    q_strr_in_context = compute_strr(tokenizer, q_words, strr_mode="in_context")
    q_tpc = float(len(q_tokens) / q_char_count) if q_char_count else 0.0

    # Options
    opt_tokens_per_choice = [_tokenize_text(tokenizer, opt) for opt in clean_options]
    opts_tokens_total = sum(len(toks) for toks in opt_tokens_per_choice)
    opts_words = extract_words(opts_text)
    opts_char_count = len(opts_text.replace(" ", ""))
    opts_tpw = float(opts_tokens_total / len(opts_words)) if opts_words else 0.0
    opts_strr = compute_strr(tokenizer, opts_words, strr_mode="isolated")
    opts_strr_in_context = compute_strr(tokenizer, opts_words, strr_mode="in_context")
    opts_tpc = float(opts_tokens_total / opts_char_count) if opts_char_count else 0.0

    # Total Prompt
    full_tokens = _tokenize_text(tokenizer, full_text)
    full_words = extract_words(full_text)
    full_char_count = len(full_text.replace(" ", ""))
    full_tpw = float(len(full_tokens) / len(full_words)) if full_words else 0.0
    full_strr = compute_strr(tokenizer, full_words, strr_mode="isolated")
    full_strr_in_context = compute_strr(tokenizer, full_words, strr_mode="in_context")
    full_tpc = float(len(full_tokens) / full_char_count) if full_char_count else 0.0

    return {
        "question_token_count": len(q_tokens),
        "question_word_count": len(q_words),
        "question_char_count": q_char_count,
        "question_tpw": q_tpw,
        "question_strr": q_strr,
        "question_strr_in_context": q_strr_in_context,
        "question_tpc": q_tpc,
        "options_token_count": opts_tokens_total,
        "options_word_count": len(opts_words),
        "options_tpw": opts_tpw,
        "options_strr": opts_strr,
        "options_strr_in_context": opts_strr_in_context,
        "options_tpc": opts_tpc,
        "total_token_count": len(full_tokens),
        "total_word_count": len(full_words),
        "overall_tpw": full_tpw,
        "overall_strr": full_strr,
        "overall_strr_in_context": full_strr_in_context,
        "overall_tpc": full_tpc,
    }


def compute_dataset_fragmentation(
    tokenizer_id: str,
    items: List[Dict[str, Any]],
    hf_token: Optional[str] = None,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Tokenizer-only computation: Computes fragmentation across dataset without downloading model weights."""
    logger.info(f"Loading tokenizer '{tokenizer_id}' for dataset fragmentation profiling...")
    if tokenizer_id == "mock-model":
        class MockTokenizer:
            def __init__(self): self.name_or_path = "mock-model"
            def encode(self, t, add_special_tokens=False): return t.split()
            def __len__(self): return 32000
        tok = MockTokenizer()
        vocab_size = 32000
        tok_class = "MockTokenizer"
    else:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(tokenizer_id, token=hf_token)
        vocab_size = len(tok)
        tok_class = type(tok).__name__

    logger.info(f"Tokenizer: {tok_class} | Vocab Size: {vocab_size:,}")

    records = []
    for it in items:
        frag = compute_fragmentation_metrics(tok, it.get("question", ""), it.get("options", []))
        records.append({
            "question_id": it.get("question_id", ""),
            "subject": it.get("subject", ""),
            "overall_tpw": frag["overall_tpw"],
            "overall_strr_isolated": frag["overall_strr"],
            "overall_strr_in_context": frag["overall_strr_in_context"],
            "overall_tpc": frag["overall_tpc"],
        })

    frag_df = pd.DataFrame(records)
    summary = {
        "tokenizer_id": tokenizer_id,
        "vocab_size": vocab_size,
        "tokenizer_class": tok_class,
        "mean_tpw": float(frag_df["overall_tpw"].mean()),
        "mean_strr_isolated": float(frag_df["overall_strr_isolated"].mean()),
        "mean_strr_in_context": float(frag_df["overall_strr_in_context"].mean()),
        "mean_tpc": float(frag_df["overall_tpc"].mean()),
    }
    logger.info(f"Summary: Mean TPW={summary['mean_tpw']:.2f} | Mean STRR={summary['mean_strr_isolated']:.3f} | Mean TPC={summary['mean_tpc']:.3f}")
    return frag_df, summary


## Section 7: Uncertainty Quantification & Calibration Metrics
Implements exact mathematical formulations for calibration:
- **Expected Calibration Error (ECE):** Uniform equal-width and quantile adaptive binning.
- **Maximum Calibration Error (MCE):** Worst-case bin calibration gap.
- **Top-1 Brier Score & Full Multiclass Brier Score:** Evaluates mean squared probability deviation.
- **Negative Log-Likelihood (NLL):** True-class cross-entropy with $\epsilon$-stabilization.


In [ ]:
def _filter_valid_pairs(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
) -> Tuple[np.ndarray, np.ndarray]:
    """Filter out NaN, None, or infinite values from confidence and correctness arrays."""
    conf = np.asarray(confidences, dtype=np.float64)
    corr = np.asarray(corrects, dtype=np.float64)

    if len(conf) != len(corr):
        raise ValueError(f"Length mismatch: confidences={len(conf)}, corrects={len(corr)}")

    valid_mask = np.isfinite(conf) & np.isfinite(corr)
    conf_clean = np.clip(conf[valid_mask], 0.0, 1.0)
    corr_clean = (corr[valid_mask] > 0.5).astype(np.float64)

    return conf_clean, corr_clean


def expected_calibration_error(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    n_bins: int = 10,
    strategy: str = "uniform",
) -> float:
    """Calculate Expected Calibration Error (ECE)."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    n = len(conf)
    if n == 0:
        return float("nan")

    if strategy == "quantile":
        quantiles = np.linspace(0, 1, n_bins + 1)
        bins = np.percentile(conf, quantiles * 100)
        bins = np.unique(bins)
        if len(bins) < 2:
            bins = np.array([0.0, 1.0])
    else:
        bins = np.linspace(0.0, 1.0, n_bins + 1)

    actual_n_bins = len(bins) - 1
    ece = 0.0

    for i in range(actual_n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        bin_count = np.sum(mask)
        if bin_count == 0:
            continue

        bin_conf = np.mean(conf[mask])
        bin_acc = np.mean(corr[mask])
        ece += (bin_count / n) * abs(bin_conf - bin_acc)

    return float(ece)


def max_calibration_error(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    n_bins: int = 10,
) -> float:
    """Calculate Maximum Calibration Error (MCE)."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    if len(conf) == 0:
        return float("nan")

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    worst_error = 0.0
    has_bin = False

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        bin_count = np.sum(mask)
        if bin_count == 0:
            continue

        has_bin = True
        bin_conf = np.mean(conf[mask])
        bin_acc = np.mean(corr[mask])
        worst_error = max(worst_error, abs(bin_conf - bin_acc))

    return float(worst_error) if has_bin else float("nan")


def brier_score(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
) -> float:
    """Calculate Top-1 / MSP-based Brier Score."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    if len(conf) == 0:
        return float("nan")
    return float(np.mean((conf - corr) ** 2))


def multiclass_brier_score(
    probabilities: Union[List[List[float]], np.ndarray],
    gold_indices: Union[List[int], np.ndarray],
    num_classes: int = 4,
) -> float:
    """Calculate Full Multiclass Brier Score across probability vectors."""
    probs = np.asarray(probabilities, dtype=np.float64)
    golds = np.asarray(gold_indices, dtype=np.int64)

    if len(probs) != len(golds) or len(probs) == 0:
        return float("nan")

    valid_mask = [
        row is not None and len(row) == num_classes and np.all(np.isfinite(row))
        for row in probs
    ]
    if not any(valid_mask):
        return float("nan")

    valid_probs = probs[valid_mask]
    valid_golds = golds[valid_mask]

    n = len(valid_probs)
    one_hot = np.zeros((n, num_classes), dtype=np.float64)
    for i, g in enumerate(valid_golds):
        if 0 <= g < num_classes:
            one_hot[i, g] = 1.0

    return float(np.mean(np.sum((valid_probs - one_hot) ** 2, axis=1)))


def negative_log_likelihood(
    probabilities: Union[List[List[float]], np.ndarray],
    gold_indices: Union[List[int], np.ndarray],
    eps: float = 1e-12,
) -> float:
    """Calculate Negative Log-Likelihood (NLL) on the true class."""
    probs = np.asarray(probabilities, dtype=np.float64)
    golds = np.asarray(gold_indices, dtype=np.int64)

    if len(probs) != len(golds) or len(probs) == 0:
        return float("nan")

    nll_vals = []
    for p_vec, g in zip(probs, golds):
        if p_vec is None or not (0 <= g < len(p_vec)):
            continue
        prob_gold = float(p_vec[g])
        if not np.isfinite(prob_gold):
            continue
        prob_gold = np.clip(prob_gold, eps, 1.0)
        nll_vals.append(-np.log(prob_gold))

    if not nll_vals:
        return float("nan")
    return float(np.mean(nll_vals))


def compute_calibration_bins(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    n_bins: int = 10,
) -> List[Dict[str, Any]]:
    """Compute detailed bin statistics for calibration reliability diagrams."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    n = len(conf)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_records = []

    for i in range(n_bins):
        lo, hi = float(bins[i]), float(bins[i + 1])
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        count = int(np.sum(mask))
        if count > 0:
            mean_conf = float(np.mean(conf[mask]))
            accuracy = float(np.mean(corr[mask]))
            gap = float(abs(mean_conf - accuracy))
        else:
            mean_conf = float((lo + hi) / 2.0)
            accuracy = 0.0
            gap = 0.0

        bin_records.append({
            "bin_index": i,
            "bin_lower": lo,
            "bin_upper": hi,
            "sample_count": count,
            "sample_fraction": float(count / n) if n > 0 else 0.0,
            "mean_confidence": mean_conf,
            "accuracy": accuracy,
            "calibration_gap": gap,
        })

    return bin_records


def compute_all_calibration_metrics(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    probabilities: Optional[Union[List[List[float]], np.ndarray]] = None,
    gold_indices: Optional[Union[List[int], np.ndarray]] = None,
    n_bins: int = 10,
) -> Dict[str, Any]:
    """Compute a consolidated dictionary of all calibration metrics."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    total_valid = len(conf)

    if total_valid == 0:
        return {
            "sample_size": 0,
            "accuracy": float("nan"),
            "mean_confidence": float("nan"),
            f"ece_uniform_{n_bins}": float("nan"),
            f"ece_quantile_{n_bins}": float("nan"),
            "mce": float("nan"),
            "brier_score": float("nan"),
            "multiclass_brier_score": float("nan"),
            "nll": float("nan"),
            "bins": [],
        }

    acc = float(np.mean(corr))
    mean_conf = float(np.mean(conf))
    ece_u = expected_calibration_error(conf, corr, n_bins=n_bins, strategy="uniform")
    ece_q = expected_calibration_error(conf, corr, n_bins=n_bins, strategy="quantile")
    mce = max_calibration_error(conf, corr, n_bins=n_bins)
    bs = brier_score(conf, corr)

    mc_bs = float("nan")
    nll = float("nan")
    if probabilities is not None and gold_indices is not None:
        mc_bs = multiclass_brier_score(probabilities, gold_indices)
        nll = negative_log_likelihood(probabilities, gold_indices)

    bins = compute_calibration_bins(conf, corr, n_bins=n_bins)

    return {
        "sample_size": total_valid,
        "accuracy": acc,
        "mean_confidence": mean_conf,
        f"ece_uniform_{n_bins}": ece_u,
        f"ece_quantile_{n_bins}": ece_q,
        "mce": mce,
        "brier_score": bs,
        "multiclass_brier_score": mc_bs,
        "nll": nll,
        "bins": bins,
    }


## Section 8: Standardized Prompt Templates & Natural Token Continuation
Ensures prompt standardization across all models and languages. Built via a single unified builder (`_build_prompt`), both Bangla and English templates end with `:\n` in the `"plain"` variant, ensuring that candidate option letters ($A, B, C, D$) are the natural next tokens.

The `"answer_cue"` variant appends a language-appropriate cue (`উত্তর:` for Bangla, `Answer:` for English) after the instruction line, which can steer instruction-tuned models to concentrate more probability mass on the option tokens.

In [ ]:
def _build_prompt(
    question_prefix: str,
    question: str,
    options_prefix: str,
    options: List[str],
    instruction: str,
) -> str:
    """Single shared builder for multiple-choice prompts ensuring structural equivalence."""
    opts = list(options)
    if len(opts) < 4:
        opts.extend([""] * (4 - len(opts)))

    opt_block = "\n".join([f"{letter}) {opts[i]}" for i, letter in enumerate(OPTION_LETTERS)])
    return (
        f"{question_prefix}: {question}\n\n"
        f"{options_prefix}:\n"
        f"{opt_block}\n\n"
        f"{instruction}:\n"
    )


def format_bnmmlu_baseline_prompt(question: str, options: List[str]) -> str:
    """Standardized Bangla BnMMLU prompt template."""
    return _build_prompt(
        question_prefix="\u09aa\u09cd\u09b0\u09b6\u09cd\u09a8",
        question=question,
        options_prefix="\u09ac\u09bf\u0995\u09b2\u09cd\u09aa\u09b8\u09ae\u09c2\u09b9",
        options=options,
        instruction="\u09b8\u09b0\u09be\u09b8\u09b0\u09bf \u09b8\u09a0\u09bf\u0995 \u0989\u09a4\u09cd\u09a4\u09b0 \u09a6\u09bf\u09a8\u0964 \u09b6\u09c1\u09a7\u09c1 A, B, C, \u09ac\u09be D \u09b2\u09bf\u0996\u09c1\u09a8\u0964 \u0985\u09a8\u09cd\u09af \u0995\u09bf\u099b\u09c1 \u09b2\u09bf\u0996\u09ac\u09c7\u09a8 \u09a8\u09be",
    )


def format_english_mmlu_prompt(question: str, options: List[str]) -> str:
    """Matched English MMLU prompt template."""
    return _build_prompt(
        question_prefix="Question",
        question=question,
        options_prefix="Options",
        options=options,
        instruction="Answer directly with only the letter (A, B, C, or D). Do not write anything else",
    )



_FEWSHOT_EXEMPLARS_CACHE: Dict[str, Tuple[Optional[float], Dict[str, Any]]] = {}  # path -> (mtime, data)

def load_fewshot_exemplars(path: Optional[str] = None) -> Dict[str, Any]:
    """Load and cache fewshot_exemplars.json.

    Raises RuntimeError if file does not exist or cannot be parsed.
    """
    _cfg = globals().get("CONFIG")
    base_dir = getattr(_cfg, "output_dir", None) or OUTPUT_DIR
    exemplar_path = path or os.path.join(base_dir, "fewshot_exemplars.json")
    _mtime = os.path.getmtime(exemplar_path) if os.path.exists(exemplar_path) else None
    _cached = _FEWSHOT_EXEMPLARS_CACHE.get(exemplar_path)
    if _cached is not None and _cached[0] == _mtime:
        return _cached[1]

    if not os.path.exists(exemplar_path):
        raise RuntimeError(
            f"Few-shot exemplar file missing: '{exemplar_path}'. "
            "Please run prepare_research_subsets() or ensure fewshot_exemplars.json exists in the output directory."
        )
    with open(exemplar_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    _FEWSHOT_EXEMPLARS_CACHE[exemplar_path] = (_mtime, data)
    return data
def format_prompt(
    question: str,
    options: List[str],
    language: str = "bn",
    variant: str = "plain",
    n_shots: int = 4,
    exemplar_file: Optional[str] = None,
) -> str:
    """Dispatcher for prompt templates based on language, variant, and n_shots.

    Args:
        variant: "plain", "answer_cue", or "few_shot".
        n_shots: Number of exemplars for few_shot ({4, 8}).
    """
    if variant not in VALID_PROMPT_VARIANTS:
        raise ValueError(f"Unknown prompt variant '{variant}'. Allowed: {VALID_PROMPT_VARIANTS}")

    if language.lower() in ["en", "english"]:
        base = format_english_mmlu_prompt(question, options)
    else:
        base = format_bnmmlu_baseline_prompt(question, options)

    if variant == "plain":
        return base

    lang_key = "en" if language.lower() in ["en", "english"] else "bn"
    cue = ANSWER_CUES[lang_key]

    if variant == "answer_cue":
        return base.rstrip() + "\n" + cue

    # few_shot
    if n_shots not in {4, 8}:
        raise ValueError(f"Invalid n_shots={n_shots} for few_shot. Allowed: {{4, 8}}")

    exemplars_data = load_fewshot_exemplars(exemplar_file)
    exemplars = exemplars_data.get(lang_key, [])
    if len(exemplars) < n_shots:
        raise RuntimeError(f"Expected at least {n_shots} exemplars for {lang_key}, found {len(exemplars)}")

    blocks = []
    for ex in exemplars[:n_shots]:
        ex_base = (
            format_english_mmlu_prompt(ex["question"], ex["options"])
            if lang_key == "en"
            else format_bnmmlu_baseline_prompt(ex["question"], ex["options"])
        )
        gold = str(ex["correct_answer"]).upper()
        # Each example is the full answer_cue prompt with " " + gold letter appended after the cue
        ex_prompt = ex_base.rstrip() + "\n" + cue + " " + gold
        blocks.append(ex_prompt)

    # Final test item block ends with bare cue and NO trailing newline
    test_block = base.rstrip() + "\n" + cue
    blocks.append(test_block)

    return "\n\n".join(blocks)


## Section 9: Multi-Backend Inference Engine & Option-Token Extraction
Implements:
1. **Option Token Resolver (`_resolve_option_token_ids`):** Resolves candidate letters across single-token variants (`"A"`, `" A"`). Raises `RuntimeError` if any letter lacks a single-token representation.
2. **Logsumexp Normalization & Option Mass:** Full vocabulary log-softmax followed by logsumexp over variant tokens. Calculates `option_mass` = total full-vocabulary mass on candidate options.
3. **Finite Float32 Logits & NaN Guards:** Casts logits to float32 before softmax; on non-finite logits returns `predicted_letter=None` and flags `probability_source="nan_logits"`.
4. **Chat Template Support (`use_chat_template`):** Applies `tokenizer.apply_chat_template` with generation prompt when enabled.
5. **Left-Padding Batched Inference Support:** Safely handles batched queries with left padding.


In [ ]:
import zlib


@dataclass
class InferenceResult:
    predicted_letter: Optional[str]
    confidence: Optional[float]
    probabilities: Optional[List[float]]
    raw_response: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    probability_source: str = "none"
    option_mass: float = 1.0
    chat_template_used: bool = False


class BaseInferenceBackend:
    """Abstract base inference engine."""
    def __init__(self, model_id: str, **kwargs):
        self.model_id = model_id
        self.tokenizer = None

    def predict_option_probabilities(self, prompt: str) -> InferenceResult:
        raise NotImplementedError

    def generate_text(self, prompt: str, max_tokens: int = 64) -> InferenceResult:
        raise NotImplementedError


class MockInferenceBackend(BaseInferenceBackend):
    """Deterministic mock backend for testing and offline environments."""
    def __init__(self, model_id: str = "mock-model", seed: int = DEFAULT_SEED, **kwargs):
        super().__init__(model_id)
        self.rng = np.random.RandomState(seed)

        class MockTokenizer:
            def __init__(self):
                self.name_or_path = model_id
            def encode(self, text, add_special_tokens=False):
                words = text.split()
                toks = []
                for w in words:
                    is_bangla = any(ord(c) >= 0x0980 and ord(c) <= 0x09FF for c in w)
                    factor = max(1, len(w) // 2) if is_bangla else max(1, len(w) // 4)
                    toks.extend([zlib.crc32((w + str(i)).encode("utf-8")) % 50000 for i in range(factor)])
                return toks or [1]
            def tokenize(self, text):
                return [str(t) for t in self.encode(text)]

        self.tokenizer = MockTokenizer()

    def predict_option_probabilities(self, prompt: str) -> InferenceResult:
        raw_logits = self.rng.randn(4)
        exps = np.exp(raw_logits - np.max(raw_logits))
        probs = (exps / np.sum(exps)).tolist()
        pred_idx = int(np.argmax(probs))
        pred_letter = OPTION_LETTERS[pred_idx]
        confidence = float(probs[pred_idx])
        simulated_mass = float(self.rng.uniform(0.65, 0.95))

        return InferenceResult(
            predicted_letter=pred_letter,
            confidence=confidence,
            probabilities=probs,
            raw_response=pred_letter,
            prompt_tokens=len(self.tokenizer.encode(prompt)),
            completion_tokens=1,
            probability_source="mock_logits",
            option_mass=simulated_mass,
            chat_template_used=False,
        )

    def generate_text(self, prompt: str, max_tokens: int = 64) -> InferenceResult:
        res = self.predict_option_probabilities(prompt)
        res.raw_response = f"সঠিক উত্তর: {res.predicted_letter}"
        res.probability_source = "mock_text_generation"
        return res


class HuggingFaceBackend(BaseInferenceBackend):
    """HuggingFace Transformers inference engine with logsumexp option extraction."""
    def __init__(
        self,
        model_id: str,
        device: Optional[str] = None,
        dtype: str = "float16",
        hf_token: Optional[str] = None,
        use_chat_template: bool = False,
        **kwargs,
    ):
        super().__init__(model_id)
        if not HAS_TORCH:
            raise ImportError("PyTorch is required for HuggingFaceBackend.")
        from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
        import torch

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.use_chat_template = use_chat_template
        logger.info(f"Loading HuggingFace model '{model_id}' on {self.device} (dtype={dtype})...")

        token_arg = {"token": hf_token} if hf_token else {}
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, **token_arg)

        # Ensure left padding for batched causal inference
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Architecture + tokenizer facts are logged so they can be reported (and checked against the registry)
        model_cfg = AutoConfig.from_pretrained(model_id, **token_arg)
        self.model_type = getattr(model_cfg, "model_type", "unknown")
        self.vocab_size = len(self.tokenizer)
        logger.info(f"Architecture: model_type='{self.model_type}' | tokenizer={type(self.tokenizer).__name__} | "
                    f"tokenizer size={self.vocab_size:,}")

        # Safe dtype selection
        if dtype == "float16":
            torch_dtype = torch.float16
        elif dtype == "float32":
            torch_dtype = torch.float32
        else:
            torch_dtype = torch.float16 if self.device == "cuda" else torch.float32

        if self.device == "cpu" and torch_dtype == torch.float16:
            torch_dtype = torch.float32          # fp16 is not reliable on CPU

        if self.model_type.startswith("gemma") and self.device == "cuda" and torch_dtype != torch.float32:
            logger.info(f"Gemma-family architecture ('{self.model_type}') detected: forcing torch.float32 "
                        "(fp16 overflows and the T4 has no native bf16).")
            torch_dtype = torch.float32

        self.model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch_dtype, **token_arg)
        self.model.to(self.device)
        self.model.eval()

        # Log memory at load
        if self.device == "cuda":
            alloc_mb = torch.cuda.memory_allocated() / (1024 ** 2)
            max_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            logger.info(f"Model loaded. VRAM allocated: {alloc_mb:.1f} MB (Peak: {max_mb:.1f} MB)")

        # Resolve option variant token IDs
        self.variant_ids = self._resolve_option_token_ids()

    def _resolve_option_token_ids(self) -> Dict[str, List[int]]:
        """Resolve single-token IDs for candidate letters A, B, C, D across variants."""
        variant_ids: Dict[str, List[int]] = {}
        for letter in OPTION_LETTERS:
            candidates = [letter, f" {letter}"]
            ids = []
            for cand in candidates:
                toks = self.tokenizer.encode(cand, add_special_tokens=False)
                if len(toks) == 1:
                    ids.append(toks[0])
            unique_ids = list(dict.fromkeys(ids))
            if not unique_ids:
                raise RuntimeError(
                    f"Tokenizer for model '{self.model_id}' cannot encode option letter '{letter}' "
                    f"into any single-token variant (tested: {candidates}). Logit-level extraction requires single tokens."
                )
            variant_ids[letter] = unique_ids

        logger.info(f"Resolved Option Token IDs for '{self.model_id}':")
        for letter, ids in variant_ids.items():
            tokens_decoded = [self.tokenizer.decode([i]) for i in ids]
            logger.info(f"   Option {letter}: IDs={ids} -> tokens={tokens_decoded}")
        return variant_ids

    def predict_option_probabilities(self, prompt: str) -> InferenceResult:
        """Extract softmax probabilities over candidate option tokens via logsumexp."""
        torch = self.torch
        chat_used = False
        formatted_prompt = prompt

        if self.use_chat_template and getattr(self.tokenizer, "chat_template", None):
            messages = [{"role": "user", "content": prompt}]
            try:
                formatted_prompt = self.tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
                chat_used = True
            except Exception as e:
                logger.warning(f"Chat template application failed: {e}. Falling back to standard prompt.")

        inputs = self.tokenizer(
            formatted_prompt,
            return_tensors="pt",
            add_special_tokens=False if chat_used else True
        ).to(self.device)

        try:
            with torch.no_grad():
                outputs = self.model(**inputs)
                next_logits = outputs.logits[0, -1, :].float()

                if not torch.isfinite(next_logits).all():
                    logger.warning("Non-finite logits detected in model forward pass.")
                    return InferenceResult(
                        predicted_letter=None,
                        confidence=None,
                        probabilities=None,
                        raw_response="",
                        prompt_tokens=int(inputs["input_ids"].shape[1]),
                        completion_tokens=0,
                        probability_source="nan_logits",
                        option_mass=0.0,
                        chat_template_used=chat_used,
                    )

                log_probs = torch.log_softmax(next_logits, dim=-1)

                option_logsumexps = []
                all_option_ids = []
                for letter in OPTION_LETTERS:
                    ids = self.variant_ids[letter]
                    all_option_ids.extend(ids)
                    sub_log_probs = log_probs[ids]
                    lse = torch.logsumexp(sub_log_probs, dim=0)
                    option_logsumexps.append(lse)

                option_logsumexps_tensor = torch.stack(option_logsumexps)
                probs_tensor = torch.softmax(option_logsumexps_tensor, dim=0)
                probs = [float(p) for p in probs_tensor.cpu().numpy()]

                pred_idx = int(torch.argmax(probs_tensor).item())
                pred_letter = OPTION_LETTERS[pred_idx]
                confidence = float(probs[pred_idx])

                unique_all_ids = list(set(all_option_ids))
                option_mass = float(torch.sum(torch.exp(log_probs[unique_all_ids])).item())

                return InferenceResult(
                    predicted_letter=pred_letter,
                    confidence=confidence,
                    probabilities=probs,
                    raw_response=pred_letter,
                    prompt_tokens=int(inputs["input_ids"].shape[1]),
                    completion_tokens=1,
                    probability_source="option_logits_softmax",
                    option_mass=option_mass,
                    chat_template_used=chat_used,
                )
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" in str(e).lower() or isinstance(e, getattr(torch.cuda, "OutOfMemoryError", ())):
                logger.error(f"💥 GPU Out-Of-Memory (OOM) encountered: {e}")
                if self.device == "cuda":
                    torch.cuda.empty_cache()
                raise
            raise

    def generate_text(self, prompt: str, max_tokens: int = 64) -> InferenceResult:
        """Baseline text generation with regex extraction."""
        torch = self.torch
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=False,
            )
            prompt_len = inputs["input_ids"].shape[1]
            gen_ids = out_ids[0, prompt_len:]
            gen_text = self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            pred_letter = extract_answer_regex(gen_text)

            return InferenceResult(
                predicted_letter=pred_letter,
                confidence=None,
                probabilities=None,
                raw_response=gen_text,
                prompt_tokens=int(prompt_len),
                completion_tokens=len(gen_ids),
                probability_source="text_generation_heuristic",
                option_mass=0.0,
                chat_template_used=False,
            )


class VLLMBackend(BaseInferenceBackend):
    """High-throughput vLLM backend with candidate logprob extraction."""
    def __init__(self, model_id: str, **kwargs):
        super().__init__(model_id)
        from vllm import LLM, SamplingParams
        logger.info(f"Starting vLLM engine for '{model_id}'...")
        self.llm = LLM(model=model_id, gpu_memory_utilization=0.85, max_model_len=1024)
        self.tokenizer = self.llm.get_tokenizer()
        self.SamplingParams = SamplingParams

    def predict_option_probabilities(self, prompt: str) -> InferenceResult:
        sp = self.SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
        outputs = self.llm.generate([prompt], sp)
        gen_out = outputs[0].outputs[0]
        pred_text = (gen_out.text or "").strip().upper()

        probs = None
        confidence = None
        pred_letter = None
        option_mass = 0.0

        if gen_out.logprobs and len(gen_out.logprobs) > 0:
            first_pos = gen_out.logprobs[0]
            opt_logprobs = {}
            for tok_id, logprob_obj in first_pos.items():
                tok_str = logprob_obj.decoded_token.strip().upper()
                if tok_str in OPTION_LETTERS and tok_str not in opt_logprobs:
                    opt_logprobs[tok_str] = logprob_obj.logprob

            if len(opt_logprobs) == 4:
                unnorm = [np.exp(opt_logprobs[l]) for l in OPTION_LETTERS]
                norm_sum = sum(unnorm)
                if norm_sum > 0:
                    probs = [p / norm_sum for p in unnorm]
                    pred_idx = int(np.argmax(probs))
                    pred_letter = OPTION_LETTERS[pred_idx]
                    confidence = float(probs[pred_idx])
                    option_mass = float(norm_sum)

        if pred_letter is None:
            pred_letter = extract_answer_regex(pred_text)

        return InferenceResult(
            predicted_letter=pred_letter,
            confidence=confidence,
            probabilities=probs,
            raw_response=pred_text,
            prompt_tokens=len(outputs[0].prompt_token_ids) if outputs[0].prompt_token_ids else 0,
            completion_tokens=1,
            probability_source="vllm_logprobs" if probs is not None else "vllm_generation_heuristic",
            option_mass=option_mass,
            chat_template_used=False,
        )

    def generate_text(self, prompt: str, max_tokens: int = 64) -> InferenceResult:
        sp = self.SamplingParams(max_tokens=max_tokens)
        outputs = self.llm.generate([prompt], sp)
        gen_text = (outputs[0].outputs[0].text or "").strip()
        pred_letter = extract_answer_regex(gen_text)
        return InferenceResult(
            predicted_letter=pred_letter,
            confidence=None,
            probabilities=None,
            raw_response=gen_text,
            prompt_tokens=len(outputs[0].prompt_token_ids) if outputs[0].prompt_token_ids else 0,
            completion_tokens=len(outputs[0].outputs[0].token_ids) if hasattr(outputs[0].outputs[0], "token_ids") else 0,
            probability_source="vllm_generation_baseline",
            option_mass=0.0,
            chat_template_used=False,
        )


def create_inference_backend(backend_type: str, model_id: str, **kwargs) -> BaseInferenceBackend:
    """Factory function for inference backends. Prohibits mock in STRICT_REAL_MODE."""
    b_lower = backend_type.lower()
    if b_lower in ["transformers", "hf", "huggingface"]:
        return HuggingFaceBackend(model_id, **kwargs)
    elif b_lower in ["vllm"]:
        return VLLMBackend(model_id, **kwargs)
    elif b_lower in ["mock", "dummy", "test"]:
        if STRICT_REAL_MODE:
            raise RuntimeError(
                f"Mock backend type '{backend_type}' is strictly prohibited when STRICT_REAL_MODE=True. "
                "Only actual 'transformers' inference backend is permitted."
            )
        return MockInferenceBackend(model_id, **kwargs)
    else:
        raise ValueError(f"Unknown backend type: '{backend_type}'. Choose 'transformers'.")


## Section 10: Benchmark Evaluator with Checkpointing & Positional Bias Diagnostics
Core evaluation engine featuring:
- **Frequent Checkpointing:** Flushes results to disk every 25 items (`CHECKPOINT_FREQ`), ensuring session interrupts or 12h timeouts lose at most 25 items.
- **Independent Directory Hierarchy:** Writes outputs to model-specific directories `results/<model_key>/<lang>_results.csv`.
- **Positional Bias Diagnostics:** Validates predictions; flags runs as `INVALID` if one option exceeds 80% or mean `option_mass < 0.5`.


In [ ]:
# Patterns detecting cross-referencing options exempt from permutation debiasing
CROSS_REFERENCE_PATTERNS = [
    # English patterns
    r"\b(?:both|all|none|neither|any)\s+of\s+the\s+above\b",
    r"\b(?:all|both|neither)\s+(?:of\s+)?(?:them|these|the\s+options)\b",
    r"\b(?:both|either)\s+[A-D]\s+(?:and|or)\s+[A-D]\b",
    r"\b[A-D]\s+and\s+[A-D]\b",
    r"\b[A-D]\s+or\s+[A-D]\b",
    r"\boptions?\s+[A-D]\b",
    r"\bstatements?\s+[A-D]\b",
    # Standalone uppercase letters referring to options
    r"^\s*[A-D]\s*$",
    r"\b[A-D]\s*[,&]\s*[A-D]\b",

    # Bangla patterns
    r"উপরের\s+(?:সবগুলো|কোনটিই\s+নয়|কোনটাই\s+নয়|উভয়|সবকটি|সব)",
    r"(?:সবগুলো|কোনটিই\s+নয়|কোনটাই\s+নয়|সবকটি|উভয়ই|উভয়)",
    r"[ক-ঘ]\s+ও\s+[ক-ঘ]",
    r"[ক-ঘ]\s+এবং\s+[ক-ঘ]",
    r"[ক-ঘ]\s+বা\s+[ক-ঘ]",
    r"[A-D]\s+এবং\s+[A-D]",
    r"[A-D]\s+ও\s+[A-D]",
    r"[A-D]\s+বা\s+[A-D]",
]

def is_option_cross_referencing(options: List[str]) -> bool:
    """Return True if any option text references other options or relative positions."""
    if not options:
        return False
    combined_patterns = [re.compile(p, re.IGNORECASE) for p in CROSS_REFERENCE_PATTERNS]
    for opt in options:
        text = str(opt).strip()
        for pat in combined_patterns:
            if pat.search(text):
                return True
    return False

def extract_answer_regex(text: Optional[str]) -> Optional[str]:
    """Extract A, B, C, or D from generated text with Bengali numeral handling."""
    if not text:
        return None

    text_clean = str(text).strip()
    bengali_to_english = {
        "১": "1", "২": "2", "৩": "3", "৪": "4",
        "এক": "1", "দুই": "2", "তিন": "3", "চার": "4",
    }
    number_to_letter = {"1": "A", "2": "B", "3": "C", "4": "D"}

    for bengali, english in bengali_to_english.items():
        text_clean = text_clean.replace(bengali, english)

    text_upper = text_clean.upper()

    patterns = [
        r"^([ABCD])\b",
        r"\b([ABCD])\s*$",
        r"\b([ABCD])\)",
        r"উত্তর:\s*([ABCD])",
        r"ANSWER:\s*([ABCD])",
        r"\b([ABCD])\b",
        r"^([1-4])\b",
        r"\b([1-4])\s*$",
        r"\b([1-4])\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, text_upper)
        if match:
            ans = match.group(1)
            if ans in OPTION_LETTERS:
                return ans
            elif ans in number_to_letter:
                return number_to_letter[ans]

    nums = re.findall(r"\b([1-4])\b", text_clean)
    if len(nums) == 1 and nums[0] in number_to_letter:
        return number_to_letter[nums[0]]

    letters = re.findall(r"\b([ABCD])\b", text_upper)
    if len(letters) == 1:
        return letters[0]

    return None


class BenchmarkEvaluator:
    """Core evaluation engine supporting checkpointing and positional bias checks."""
    def __init__(
        self,
        backend: BaseInferenceBackend,
        model_key: str,
        mode: str = "extended",
        output_dir: str = OUTPUT_DIR,
        language: str = "bn",
        benchmark_name: str = "BnMMLU",
        resume: bool = True,
        prompt_variant: Optional[str] = None,
        collapse_threshold: float = MAX_COLLAPSE_THRESHOLD,
        settings: Optional[dict] = None,
    ):
        self.backend = backend
        self.model_key = model_key
        self.mode = mode.lower()
        self.language = language
        self.benchmark_name = benchmark_name
        # Resolved at call time (not at definition time) so later CONFIG changes are honoured
        self.prompt_variant = prompt_variant if prompt_variant is not None else CONFIG.prompt_variant
        if self.prompt_variant not in VALID_PROMPT_VARIANTS:
            raise ValueError(f"Unknown prompt_variant '{self.prompt_variant}'. Allowed: {sorted(VALID_PROMPT_VARIANTS)}")
        self.collapse_threshold = collapse_threshold
        # Effective settings dict (used for predict_item and inference_signature)
        self._settings = settings if settings is not None else {
            "prompt_variant": self.prompt_variant,
            "n_shots": 0,
            "use_chat_template": getattr(backend, "use_chat_template", False),
            "debias": "none",
            "collapse_threshold": collapse_threshold,
            "collapse_gate_overridden": False,
        }
        self.inference_signature = _make_inference_signature(self._settings)
        self.lock = threading.Lock()

        # Dedicated per-model directory: results/<model_key>/
        self.model_dir = os.path.join(output_dir, model_key)
        os.makedirs(self.model_dir, exist_ok=True)

        self.results_csv = os.path.join(self.model_dir, f"{language}_results.csv")
        self.results_jsonl = os.path.join(self.model_dir, f"{language}_results.jsonl")
        self.summary_json = os.path.join(self.model_dir, f"{language}_summary.json")

        self.processed_ids: set = set()
        self.buffer: List[Dict[str, Any]] = []
        self._initialize_outputs()

    def _initialize_outputs(self):
        """Prepare output CSV files with updated schema headers."""
        if not os.path.exists(self.results_csv):
            if self.mode == "baseline":
                columns = [
                    "question_id", "correct_answer", "llm_answer",
                    "original_llm_response", "model_name", "timestamp", "is_correct",
                ]
            else:
                columns = [
                    "question_id", "benchmark", "language", "subject", "subject_group", "has_markup", "model_key",
                    "model_id", "tokenizer_id", "gold_answer", "predicted_answer",
                    "is_correct", "confidence", "prob_A", "prob_B", "prob_C", "prob_D",
                    "option_mass", "chat_template_used", "calibration_error_item",
                    "question_token_count", "question_word_count", "question_char_count",
                    "question_tpw", "question_strr", "question_strr_in_context", "question_tpc",
                    "options_token_count", "options_word_count", "options_tpw",
                    "options_strr", "options_strr_in_context", "total_token_count",
                    "total_word_count", "overall_tpw", "overall_strr",
                    "overall_strr_in_context", "overall_tpc", "prompt_tokens", "probability_source",
                    "prompt_variant", "raw_response", "timestamp",
                    "inference_signature", "pred_raw", "confidence_raw", "perm_applied",
                ]
            df = pd.DataFrame(columns=columns)
            df.to_csv(self.results_csv, index=False, encoding="utf-8")

        # Load existing results for resume support.
        # ONLY the file read is wrapped in try/except. The schema guards run afterwards, outside the try,
        # so their RuntimeError actually propagates instead of being downgraded to a warning.
        existing = None
        if os.path.exists(self.results_csv):
            try:
                existing = pd.read_csv(self.results_csv, encoding="utf-8")
            except Exception as e:
                logger.warning(f"Could not read existing results CSV for resume: {e}")

        if existing is not None and not existing.empty:
            if self.mode != "baseline":
                if "prompt_tokens" not in existing.columns:
                    raise RuntimeError(
                        f"{self.results_csv} was written with an older schema (no 'prompt_tokens' column). "
                        "Delete that model folder (or point CONFIG.output_dir elsewhere) and re-run, otherwise the "
                        "length-controlled econometric models cannot be estimated."
                    )
                if "subject_group" not in existing.columns:
                    raise RuntimeError(
                        f"{self.results_csv} was written with an older schema (no subject_group column). "
                        "Delete that model folder (or point CONFIG.output_dir elsewhere) and re-run, otherwise the "
                        "subject-paired analysis would silently lose its keys."
                    )
                if "prompt_variant" not in existing.columns:
                    raise RuntimeError(
                        f"{self.results_csv} has no 'prompt_variant' column (written by an older version). "
                        "Delete that model folder (or change CONFIG.output_dir) and re-run, otherwise rows produced "
                        "with different prompts would be mixed."
                    )
                found_variants = set(existing["prompt_variant"].astype(str))
                if found_variants != {self.prompt_variant}:
                    raise RuntimeError(
                        f"Prompt-variant mismatch in {self.results_csv}: existing rows use {sorted(found_variants)} "
                        f"but this run uses '{self.prompt_variant}'. Delete that model folder (or change "
                        "CONFIG.output_dir) so rows from different prompt variants are never mixed."
                    )
                # v3a: also check inference_signature so debias/chat changes are caught
                if "inference_signature" in existing.columns:
                    found_sigs = set(existing["inference_signature"].dropna().astype(str))
                    if found_sigs and found_sigs != {self.inference_signature}:
                        raise RuntimeError(
                            f"Inference-signature mismatch in {self.results_csv}: existing rows have "
                            f"{sorted(found_sigs)} but this run has '{self.inference_signature}'. "
                            "Delete the model folder or change CONFIG.output_dir."
                        )
            if "question_id" in existing.columns:
                self.processed_ids = set(existing["question_id"].astype(str))
                logger.info(f"Resume loaded {len(self.processed_ids)} already-processed items from: {self.results_csv}")

    def evaluate_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        """Evaluate a single MCQ item in either baseline or extended mode."""
        qid = str(item.get("question_id", ""))
        gold = str(item.get("correct_answer", "")).upper()
        subject = str(item.get("subject", "general"))
        question = str(item.get("question", ""))
        options = item.get("options", [])

        # Baseline mode always keeps the plain prompt; the variant only applies to extended mode
        if self.mode == "baseline":
            prompt = format_prompt(question=question, options=options, language=self.language, variant="plain")
            inf_res = self.backend.generate_text(prompt)
            pred = inf_res.predicted_letter
            is_correct = bool(pred == gold) if pred and gold else False

            row = {
                "question_id": qid,
                "correct_answer": gold,
                "llm_answer": pred,
                "original_llm_response": inf_res.raw_response,
                "model_name": self.backend.model_id,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "is_correct": is_correct,
            }
            return row
        else:
            pr = predict_item(self.backend, item, self.language, self._settings)
            pred = pr.predicted_letter
            inf_res = pr  # PredictResult is duck-type compatible
            is_correct = bool(pred == gold) if pred and gold else False
            conf = pr.confidence

            calib_err = abs(conf - float(is_correct)) if conf is not None else float("nan")

            frag = compute_fragmentation_metrics(
                tokenizer=self.backend.tokenizer,
                question=question,
                options=options,
            )

            probs = inf_res.probabilities or [float("nan")] * 4

            row = {
                "question_id": qid,
                "benchmark": item.get("benchmark", self.benchmark_name),
                "language": self.language,
                "subject": subject,
                "subject_group": item.get("subject_group", ""),
                "has_markup": bool(item.get("has_markup", False)),
                "model_key": self.model_key,
                "model_id": self.backend.model_id,
                "tokenizer_id": getattr(self.backend.tokenizer, "name_or_path", self.backend.model_id),
                "gold_answer": gold,
                "predicted_answer": pred,
                "is_correct": is_correct,
                "confidence": conf if conf is not None else float("nan"),
                "prob_A": probs[0] if len(probs) > 0 else float("nan"),
                "prob_B": probs[1] if len(probs) > 1 else float("nan"),
                "prob_C": probs[2] if len(probs) > 2 else float("nan"),
                "prob_D": probs[3] if len(probs) > 3 else float("nan"),
                "option_mass": inf_res.option_mass,
                "chat_template_used": inf_res.chat_template_used,
                "prompt_variant": self.prompt_variant,
                "calibration_error_item": calib_err,
                "question_token_count": frag["question_token_count"],
                "question_word_count": frag["question_word_count"],
                "question_char_count": frag["question_char_count"],
                "question_tpw": frag["question_tpw"],
                "question_strr": frag["question_strr"],
                "question_strr_in_context": frag["question_strr_in_context"],
                "question_tpc": frag["question_tpc"],
                "options_token_count": frag["options_token_count"],
                "options_word_count": frag["options_word_count"],
                "options_tpw": frag["options_tpw"],
                "options_strr": frag["options_strr"],
                "options_strr_in_context": frag["options_strr_in_context"],
                "total_token_count": frag["total_token_count"],
                "total_word_count": frag["total_word_count"],
                "overall_tpw": frag["overall_tpw"],
                "overall_strr": frag["overall_strr"],
                "overall_strr_in_context": frag["overall_strr_in_context"],
                "overall_tpc": frag["overall_tpc"],
                "prompt_tokens": pr.prompt_tokens,
                "probability_source": inf_res.probability_source,
                "raw_response": inf_res.raw_response,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "inference_signature": self.inference_signature,
                "pred_raw": pr.pred_raw,
                "confidence_raw": pr.confidence_raw,
                "perm_applied": pr.perm_applied,
            }
            return row

    def flush_buffer(self) -> None:
        """Write all buffered rows to CSV/JSONL and clear the buffer."""
        with self.lock:
            if not self.buffer:
                return
            columns = list(pd.read_csv(self.results_csv, nrows=0, encoding="utf-8").columns)
            df = pd.DataFrame(self.buffer).reindex(columns=columns)
            df.to_csv(self.results_csv, mode="a", header=False, index=False, encoding="utf-8")
            with open(self.results_jsonl, "a", encoding="utf-8") as f:
                for r in self.buffer:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
            self.buffer.clear()

    def save_item_result(self, row: Dict[str, Any]) -> None:
        """Buffer one result and checkpoint to disk every CHECKPOINT_FREQ items."""
        with self.lock:
            self.buffer.append(row)
            self.processed_ids.add(str(row["question_id"]))
            need_flush = len(self.buffer) >= CHECKPOINT_FREQ
        if need_flush:
            self.flush_buffer()

    def evaluate_all(
        self,
        items: List[Dict[str, Any]],
        resume: bool = True,
        show_progress: bool = True,
    ) -> Dict[str, Any]:
        """Run evaluation with frequent checkpointing and periodic memory logging."""
        to_process = []
        for it in items:
            qid = str(it.get("question_id", ""))
            if resume and qid in self.processed_ids:
                continue
            to_process.append(it)

        logger.info(f"Starting {self.mode.upper()} evaluation: {len(to_process)} remaining ({len(self.processed_ids)} already done)")

        iterator = tqdm(to_process, desc=f"Evaluating {self.model_key} [{self.language}]") if show_progress else to_process
        try:
            for idx, item in enumerate(iterator):
                row = self.evaluate_item(item)
                self.save_item_result(row)

                # Periodic memory check
                if idx > 0 and idx % MEM_LOG_FREQ == 0 and HAS_TORCH and torch.cuda.is_available():
                    alloc_mb = torch.cuda.memory_allocated() / (1024 ** 2)
                    logger.info(f"[{idx}/{len(to_process)}] Active GPU VRAM: {alloc_mb:.1f} MB")
        finally:
            # Always persist whatever is buffered, even if the loop crashed or was interrupted
            self.flush_buffer()

        all_df = pd.read_csv(self.results_csv, encoding="utf-8")
        if "perm_applied" in all_df.columns and self._settings.get("debias") == "cyclic4":
            n_exempt = int((all_df["perm_applied"] == False).sum())
            logger.info(f"Permutation debiasing (cyclic4): {n_exempt}/{len(all_df)} items were exempted from permutation.")
        summary = self.generate_summary(all_df)

        with open(self.summary_json, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)

        logger.info("Evaluation complete. Results saved:")
        logger.info(f"   CSV:   {self.results_csv}")
        logger.info(f"   JSON:  {self.summary_json}")
        return summary

    def generate_summary(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Generate summary statistics and positional bias diagnostics."""
        total_items = len(df)
        if total_items == 0:
            return {"total_items": 0}

        is_correct_series = df["is_correct"].astype(bool)
        accuracy = float(is_correct_series.mean() * 100.0)

        pred_col = "predicted_answer" if "predicted_answer" in df.columns else "llm_answer"
        gold_col = "gold_answer" if "gold_answer" in df.columns else "correct_answer"

        pred_dist = df[pred_col].value_counts(dropna=False).to_dict()
        gold_dist = df[gold_col].value_counts(dropna=False).to_dict()
        n_nan = int(df["confidence"].isna().sum()) if "confidence" in df.columns else 0
        mean_mass = float(df["option_mass"].mean()) if "option_mass" in df.columns else 1.0

        max_pred_pct = max([c / total_items for c in pred_dist.values()]) if total_items > 0 else 0.0
        validity_flag = "VALID"
        validity_reasons = []

        if max_pred_pct > self.collapse_threshold:
            validity_flag = "INVALID"
            validity_reasons.append(f"Predicted letter collapse: single option exceeds {self.collapse_threshold*100:.0f}% ({max_pred_pct*100:.1f}%)")
        if mean_mass < MIN_OPTION_MASS_THRESHOLD:
            validity_flag = "INVALID"
            validity_reasons.append(f"Insufficient option mass: mean option mass={mean_mass:.4f} (< 0.50)")

        if validity_flag == "INVALID":
            logger.warning(f"⚠️ EVALUATION RUN FLAGGED AS INVALID: {'; '.join(validity_reasons)}")

        collapse_gate_overridden = self._settings.get("collapse_gate_overridden", False)
        if collapse_gate_overridden:
            logger.warning(f"⚠️ collapse_gate_overridden=True for '{self.model_key}' [{self.language}]: "
                           f"collapse_threshold={self.collapse_threshold:.2f} (not the global {MAX_COLLAPSE_THRESHOLD:.2f})")

        summary: Dict[str, Any] = {
            "model_key": self.model_key,
            "model_id": self.backend.model_id,
            "mode": self.mode,
            "benchmark": self.benchmark_name,
            "language": self.language,
            "total_items": total_items,
            "accuracy_percent": accuracy,
            "validity_flag": validity_flag,
            "validity_reasons": validity_reasons,
            "collapse_gate_overridden": collapse_gate_overridden,
            "inference_signature": self.inference_signature,
            "diagnostics": {
                "prediction_distribution": {str(k): int(v) for k, v in pred_dist.items()},
                "gold_distribution": {str(k): int(v) for k, v in gold_dist.items()},
                "n_nan": n_nan,
                "mean_option_mass": mean_mass,
                "n_perm_exempt": (int((df["perm_applied"] == False).sum()) if self._settings.get("debias") == "cyclic4" else None) if "perm_applied" in df.columns else None,
            }
        }

        # Raw calibration & prediction distribution (when pred_raw/confidence_raw exist)
        if "pred_raw" in df.columns and "confidence_raw" in df.columns and df["pred_raw"].notna().any():
            raw_pred_dist = df["pred_raw"].value_counts(dropna=False).to_dict()
            summary["raw_prediction_distribution"] = {str(k): int(v) for k, v in raw_pred_dist.items()}
            raw_top_share = max([c / total_items for c in raw_pred_dist.values()]) if total_items > 0 else 0.0
            logger.info(f"Raw top-letter share (before debiasing): {raw_top_share*100:.1f}%")

            if self.mode != "baseline":
                raw_conf = df["confidence_raw"].values
                raw_corr = (df["pred_raw"].astype(str).str.upper() == df[gold_col].astype(str).str.upper()).values
                summary["calibration_metrics_raw"] = compute_all_calibration_metrics(
                    confidences=raw_conf,
                    corrects=raw_corr,
                    probabilities=None,
                    gold_indices=None,
                    n_bins=10,
                )

        # Subject breakdown
        if "subject" in df.columns:
            subj_stats = {}
            for subj, sub_df in df.groupby("subject"):
                subj_acc = float(sub_df["is_correct"].astype(bool).mean() * 100.0)
                subj_stats[str(subj)] = {"count": len(sub_df), "accuracy_percent": subj_acc}
            summary["subject_accuracy"] = subj_stats

        # Extended calibration metrics
        if self.mode != "baseline" and "confidence" in df.columns:
            confidences = df["confidence"].values
            corrects = is_correct_series.values

            probs_matrix = None
            gold_indices = None
            if all(c in df.columns for c in ["prob_A", "prob_B", "prob_C", "prob_D", "gold_answer"]):
                probs_matrix = df[["prob_A", "prob_B", "prob_C", "prob_D"]].values
                gold_indices = [
                    OPTION_LETTERS.index(g) if g in OPTION_LETTERS else -1
                    for g in df["gold_answer"].astype(str).str.upper()
                ]

            summary["calibration_metrics"] = compute_all_calibration_metrics(
                confidences=confidences,
                corrects=corrects,
                probabilities=probs_matrix,
                gold_indices=gold_indices,
                n_bins=10,
            )

            frag_cols = [c for c in ["overall_tpw", "overall_strr", "overall_strr_in_context", "overall_tpc"] if c in df.columns]
            summary["fragmentation_stats"] = {
                col: {
                    "mean": float(df[col].mean()),
                    "std": float(df[col].std()),
                    "median": float(df[col].median()),
                }
                for col in frag_cols
            }

        return summary

def resolve_run_settings(
    model_key: str,
    config: "ExperimentConfig",
) -> dict:
    """Resolve effective run settings for model_key, with MODEL_REGISTRY overriding CONFIG globals.

    Returns a dict with keys:
        prompt_variant, n_shots, use_chat_template, debias, collapse_threshold,
        collapse_gate_overridden

    Raises ValueError for unknown/invalid values.
    Raises NotImplementedError for debias='cyclic4' (implemented in v3b).
    """
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_key '{model_key}'. Registered: {list(MODEL_REGISTRY.keys())}")
    m_info = dict(MODEL_REGISTRY[model_key])

    # Precedence: model_settings.json > MODEL_REGISTRY > CONFIG
    persisted = load_model_settings(getattr(config, "output_dir", OUTPUT_DIR)).get(model_key, {})
    persisted_core = {k: persisted[k] for k in ("prompt_variant", "n_shots", "use_chat_template", "debias") if k in persisted}
    settings_source = "model_settings.json" if persisted_core else (
        "MODEL_REGISTRY" if any(k in m_info for k in ("prompt_variant", "n_shots", "debias")) else "CONFIG")
    m_info.update(persisted_core)

    pv = m_info.get("prompt_variant", config.prompt_variant)
    if pv not in VALID_PROMPT_VARIANTS:
        raise ValueError(f"Invalid prompt_variant='{pv}' for '{model_key}'. Allowed: {sorted(VALID_PROMPT_VARIANTS)}")

    n_shots = m_info.get("n_shots", getattr(config, "n_shots", 0))
    if pv == "few_shot":
        if n_shots not in {4, 8}:
            raise ValueError(f"Invalid n_shots={n_shots} for '{model_key}' with prompt_variant='few_shot'. Allowed: {{4, 8}}")
    else:
        if n_shots not in {0, 4, 8}:
            raise ValueError(f"Invalid n_shots={n_shots} for '{model_key}' with prompt_variant='{pv}'. Allowed: {{0, 4, 8}}")

    use_chat = bool(m_info.get("use_chat_template", False))

    debias = m_info.get("debias", getattr(config, "debias", "none"))
    if debias not in ("none", "cyclic4"):
        raise ValueError(f"Invalid debias='{debias}' for '{model_key}'. Allowed: ('none', 'cyclic4')")

    ct_override = m_info.get("collapse_threshold_override", None)
    collapse_gate_overridden = ct_override is not None
    collapse_threshold = float(ct_override) if collapse_gate_overridden else MAX_COLLAPSE_THRESHOLD

    return {
        "prompt_variant": pv,
        "n_shots": n_shots,
        "effective_n_shots": int(n_shots) if pv == "few_shot" else 0,
        "use_chat_template": use_chat,
        "debias": debias,
        "collapse_threshold": collapse_threshold,
        "collapse_gate_overridden": collapse_gate_overridden,
        "settings_source": settings_source,
    }


def _make_inference_signature(settings: dict) -> str:
    """Build a deterministic string that uniquely identifies the inference configuration."""
    pv = settings.get("prompt_variant", "plain")
    shots_val = settings.get("n_shots", 0) if pv == "few_shot" else 0
    chat = settings.get("use_chat_template", False)
    debias = settings.get("debias", "none")
    return f"{pv}|shots={shots_val}|chat={int(bool(chat))}|debias={debias}"


class PredictResult:
    """Thin wrapper returned by predict_item."""
    __slots__ = (
        "predicted_letter", "confidence", "probabilities", "option_mass",
        "prompt_tokens", "probability_source", "raw_response", "chat_template_used",
        "pred_raw", "confidence_raw", "perm_applied",
    )

    def __init__(self, inf_res, *, pred_raw, confidence_raw, perm_applied: bool):
        self.predicted_letter = inf_res.predicted_letter
        self.confidence = inf_res.confidence
        self.probabilities = inf_res.probabilities
        self.option_mass = inf_res.option_mass
        self.prompt_tokens = inf_res.prompt_tokens
        self.probability_source = inf_res.probability_source
        self.raw_response = inf_res.raw_response
        self.chat_template_used = getattr(inf_res, "chat_template_used", False)
        self.pred_raw = pred_raw
        self.confidence_raw = confidence_raw
        self.perm_applied = perm_applied


def predict_item(
    backend: "BaseInferenceBackend",
    item: dict,
    language: str,
    settings: dict,
) -> PredictResult:
    """Build prompt and call backend; return a PredictResult.

    Supports debias='none' and debias='cyclic4'.
    When debias='cyclic4':
      - Checks if options are cross-referencing. If so, exempts (s=0 only, perm_applied=False).
      - Otherwise, runs 4 passes with cyclic rotation shift s in {0, 1, 2, 3}.
      - Position j receives original option (j + s) % 4.
      - Maps back each pass's probabilities to original options and averages them.
      - predicted_letter, confidence (MSP), probabilities, option_mass refer to averaged result.
      - probability_source = "option_logits_cyclic4".
      - Identity pass (s=0) yields pred_raw and confidence_raw.
      - perm_applied = True for permuted items.
    """
    question = str(item.get("question", ""))
    options = item.get("options", [])
    pv = settings.get("prompt_variant", "plain")
    debias = settings.get("debias", "none")

    if debias not in ("none", "cyclic4"):
        raise ValueError(f"Invalid debias='{debias}'. Allowed: ('none', 'cyclic4')")

    n_shots = settings.get("n_shots", 4)
    ex_file = settings.get("exemplar_file")
    # Pass 0: Identity prompt (s = 0)
    prompt_0 = format_prompt(question=question, options=options, language=language, variant=pv, n_shots=n_shots, exemplar_file=ex_file)
    inf_res_0 = backend.predict_option_probabilities(prompt_0)
    pred_raw = inf_res_0.predicted_letter
    confidence_raw = inf_res_0.confidence

    # If debias is none, or fewer than 4 options, return identity pass
    if debias == "none" or len(options) != 4:
        return PredictResult(
            inf_res_0,
            pred_raw=pred_raw,
            confidence_raw=confidence_raw,
            perm_applied=False,
        )

    # Check cross-referencing exemption
    if is_option_cross_referencing(options):
        return PredictResult(
            inf_res_0,
            pred_raw=pred_raw,
            confidence_raw=confidence_raw,
            perm_applied=False,
        )

    # debias == "cyclic4" with 4 options and no exemption: run 4 cyclic passes
    pass_results = [inf_res_0]
    for s in (1, 2, 3):
        # Permuted position j gets original option (j + s) % 4
        perm_options = [options[(j + s) % 4] for j in range(4)]
        p_prompt = format_prompt(question=question, options=perm_options, language=language, variant=pv, n_shots=n_shots, exemplar_file=ex_file)
        p_res = backend.predict_option_probabilities(p_prompt)
        pass_results.append(p_res)

    # Map each pass's probabilities back to original options
    mapped_probs_all = []
    option_masses = []
    prompt_tokens_all = []

    for s, p_res in enumerate(pass_results):
        probs_s = p_res.probabilities or [0.25, 0.25, 0.25, 0.25]
        # At shift s, permuted position j corresponds to original option orig_idx = (j + s) % 4
        # Conversely, original option orig_idx is located at permuted position j = (orig_idx - s) % 4
        mapped = [probs_s[(orig_idx - s) % 4] for orig_idx in range(4)]
        mapped_probs_all.append(mapped)
        if p_res.option_mass is not None:
            option_masses.append(p_res.option_mass)
        if p_res.prompt_tokens is not None:
            prompt_tokens_all.append(p_res.prompt_tokens)

    # Average mapped distributions across the 4 passes
    avg_probs = [float(np.mean([mapped_probs_all[s][i] for s in range(4)])) for i in range(4)]
    best_idx = int(np.argmax(avg_probs))
    predicted_letter = OPTION_LETTERS[best_idx]
    confidence = float(avg_probs[best_idx])
    mean_option_mass = float(np.mean(option_masses)) if option_masses else inf_res_0.option_mass
    mean_prompt_tokens = int(round(np.mean(prompt_tokens_all))) if prompt_tokens_all else inf_res_0.prompt_tokens

    # Construct combined inference result
    avg_inf_res = InferenceResult(
        predicted_letter=predicted_letter,
        confidence=confidence,
        probabilities=avg_probs,
        option_mass=mean_option_mass,
        prompt_tokens=mean_prompt_tokens,
        probability_source="option_logits_cyclic4",
        raw_response=inf_res_0.raw_response,
        chat_template_used=getattr(inf_res_0, "chat_template_used", False),
    )

    return PredictResult(
        avg_inf_res,
        pred_raw=pred_raw,
        confidence_raw=confidence_raw,
        perm_applied=True,
    )


## Section 11: Single-Model Pipeline Execution Engine & Smoke Test
Provides:
1. **`smoke_test`:** Evaluates items (default `CONFIG.smoke_n=30`), performs an option permutation shuffle test on 10 items, and verifies that the model does not suffer from positional collapse before full evaluation. When the mean-mass gate fails, the top-8 next tokens for the first 3 smoke prompts are logged to help diagnose the problem (applying chat templates when enabled).
2. **`run_one_model`:** Fully executes one model across Bangla and English, threads resolved per-model settings (`prompt_variant`, `n_shots`, `use_chat_template`, `debias`) through `BenchmarkEvaluator`, manages GPU memory cleanup in a `finally` block, writes `status.json` (including `prompt_variant`, `chat_template_used`, and `debias`), and updates progress.
3. **`status_table`:** Scans all completed runs and displays a consolidated progress dashboard.
4. **`diagnose_prompt_variants`:** Diagnostic tool evaluating the full grid of prompt variants, chat templates, and debiasing schemes on-GPU using validity gates only (Section 11b).

> **Prompt variant threading:** `format_prompt` receives the variant and exemplars explicitly. All per-model choices must be selected via validity gates and documented in the paper's methodology and limitations.

In [ ]:
import dataclasses
def _log_top_next_tokens(backend: BaseInferenceBackend, prompts: List[str], k: int = 8) -> str:
    """Log top-k next-token probabilities for diagnostic prompts.

    Silently does nothing if the backend has no .model attribute (e.g. mock backends in tests).
    Applies the tokenizer's chat template when backend.use_chat_template is on, so the logged
    prompt matches the real input passed to the model.
    Returns the diagnostic string (also logged).
    """
    model = getattr(backend, "model", None)
    if model is None:
        return ""

    tokenizer = backend.tokenizer
    torch = getattr(backend, "torch", None)
    if torch is None:
        return ""

    device = getattr(backend, "device", "cpu")
    lines = []
    use_chat = getattr(backend, "use_chat_template", False)
    has_chat_tpl = getattr(tokenizer, "chat_template", None) is not None

    for i, prompt in enumerate(prompts):
        try:
            formatted_prompt = prompt
            if use_chat and has_chat_tpl and hasattr(tokenizer, "apply_chat_template"):
                try:
                    formatted_prompt = tokenizer.apply_chat_template(
                        [{"role": "user", "content": prompt}],
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                except Exception as ex:
                    logger.warning(f"Failed to apply chat template in _log_top_next_tokens: {ex}")
                    formatted_prompt = prompt

            inputs = tokenizer(formatted_prompt, return_tensors="pt", add_special_tokens=True).to(device)
            with torch.no_grad():
                logits = model(**inputs).logits[0, -1, :].float()
            probs = torch.softmax(logits, dim=-1)
            topk = torch.topk(probs, k)
            tok_strs = [tokenizer.decode([tid]) for tid in topk.indices.tolist()]
            tok_probs = topk.values.tolist()
            line = f"  Prompt {i+1} top-{k}: " + ", ".join(
                f"{repr(t)}={p:.4f}" for t, p in zip(tok_strs, tok_probs)
            )
            lines.append(line)
            logger.info(line)
        except Exception as e:
            lines.append(f"  Prompt {i+1}: error getting top tokens: {e}")
    return "\n".join(lines)


def _build_backend(
    model_key: str,
    config: ExperimentConfig,
    use_chat_template: Optional[bool] = None,
) -> BaseInferenceBackend:
    """Create the inference backend for a registered model."""
    m_info = MODEL_REGISTRY[model_key]
    backend = create_inference_backend(
        backend_type=config.backend,
        model_id=m_info["hf_id"],
        dtype=m_info["dtype"],
        hf_token=config.hf_token,
        use_chat_template=(m_info["use_chat_template"] if use_chat_template is None else bool(use_chat_template)),
        seed=config.seed,
    )
    found_type = getattr(backend, "model_type", None)
    expected_type = m_info.get("expected_model_type")
    if found_type and expected_type and found_type != expected_type:
        logger.warning(f"Registry expects model_type='{expected_type}' for '{model_key}' but the Hub config says '{found_type}'. "
                       "Update tokenizer_family/notes in MODEL_REGISTRY so the paper describes the model correctly.")
    return backend


def _run_smoke_checks(
    model_key: str,
    items: List[Dict[str, Any]],
    n: int,
    config: ExperimentConfig,
    backend: BaseInferenceBackend,
    settings: Optional[dict] = None,
) -> bool:
    """Pre-run diagnostics on an already-loaded backend, including an option permutation check."""
    logger.info("=" * 65)
    logger.info(f"🔍 SMOKE TEST DIAGNOSTICS: model_key='{model_key}' ({n} items)")
    logger.info("=" * 65)

    # Random (seeded) sample: the dataset may be ordered by subject, and the first rows would then be one subject
    _rng = np.random.RandomState(config.seed)
    test_slice = [items[i] for i in _rng.permutation(len(items))[:n]]
    results = []

    _smoke_settings = dict(settings) if settings else {
        "prompt_variant": config.prompt_variant,
        "n_shots": getattr(config, "n_shots", 0),
        "use_chat_template": getattr(backend, "use_chat_template", False),
        "debias": getattr(config, "debias", "none"),
        "collapse_threshold": MAX_COLLAPSE_THRESHOLD,
        "collapse_gate_overridden": False,
    }
    for i, it in enumerate(test_slice):
        lang = it.get("language", "bn")
        res = predict_item(backend, it, lang, _smoke_settings)
        gold = it.get("correct_answer", "")
        correct = (res.predicted_letter == gold) if res.predicted_letter and gold else False

        results.append({
            "qid": it.get("question_id", str(i)),
            "gold": gold,
            "pred": res.predicted_letter,
            "conf": res.confidence,
            "correct": correct,
            "mass": res.option_mass,
            "tokens": res.prompt_tokens,
        })

    df_res = pd.DataFrame(results)
    pred_counts = df_res["pred"].value_counts().to_dict()
    mean_mass = df_res["mass"].mean()
    acc = df_res["correct"].mean() * 100.0

    logger.info("SMOKE TEST RESULTS TABLE (First 10 items):")
    logger.info("\n" + df_res[["qid", "gold", "pred", "conf", "correct", "mass"]].head(10).to_string(index=False))
    logger.info(f"Prediction Distribution: {pred_counts}")
    logger.info(f"Mean Accuracy: {acc:.1f}% | Mean Option Mass: {mean_mass:.4f}")

    # Check validity gates
    max_pct = max(df_res["pred"].value_counts(normalize=True)) if len(df_res) > 0 else 0.0
    if max_pct > MAX_COLLAPSE_THRESHOLD:
        collapse_msg = (f"Single-letter prediction collapse in the smoke sample ({max_pct*100:.1f}% > 80% on one option). "
                        "Small / weak models on Bangla often do this; the full run will still flag the run INVALID if it persists.")
        if config.strict_smoke:
            raise RuntimeError(f"Smoke test ABORTED for '{model_key}': {collapse_msg}")
        logger.warning(f"⚠️ {collapse_msg} (continuing because CONFIG.strict_smoke=False)")

    if mean_mass < MIN_OPTION_MASS_THRESHOLD:
        # Log top-8 next tokens for first 3 prompts to help diagnose
        diag_prompts = [
            format_prompt(it["question"], it["options"], language=it.get("language", "bn"),
                           variant=_smoke_settings["prompt_variant"],
                           n_shots=_smoke_settings.get("n_shots", 4),
                           exemplar_file=_smoke_settings.get("exemplar_file"))
            for it in test_slice[:3]
        ]
        diag_info = _log_top_next_tokens(backend, diag_prompts, k=8)
        raise RuntimeError(
            f"Smoke test ABORTED for '{model_key}': Low option mass ({mean_mass:.4f} < {MIN_OPTION_MASS_THRESHOLD}). "
            "Model does not allocate sufficient probability to candidate option tokens. "
            "For instruction-tuned models try use_chat_template=True in MODEL_REGISTRY "
            "(and keep it documented in the paper).\n"
            f"Top next-token diagnostics:\n{diag_info}\n"
            f"Hint: run diagnose_prompt_variants('{model_key}') to compare prompt variants on-GPU."
        )

    if df_res["conf"].isna().any():
        raise RuntimeError(f"Smoke test ABORTED for '{model_key}': Non-finite logits (NaN/Inf) detected.")

    # Option permutation sanity check on 10 items
    logger.info("Checking option permutation sensitivity on 10 items...")
    perm_rng = np.random.RandomState(config.seed)
    perm_preds = []
    for it in test_slice[:10]:
        opts = list(it["options"])
        # Permute options
        perm = perm_rng.permutation(4)
        shuffled_opts = [opts[p] for p in perm]
        new_prompt = format_prompt(it["question"], shuffled_opts, language=it.get("language", "bn"),
                                    variant=_smoke_settings["prompt_variant"],
                                    n_shots=_smoke_settings.get("n_shots", 4),
                                    exemplar_file=_smoke_settings.get("exemplar_file"))
        p_res = backend.predict_option_probabilities(new_prompt)
        perm_preds.append(p_res.predicted_letter)

    perm_share = max(pd.Series(perm_preds).value_counts(normalize=True))
    logger.info(f"Permutation Positional-Bias Score: {perm_share:.2f}")

    logger.info("✅ Smoke test PASSED all criteria.")
    logger.info("=" * 65)
    return True


def smoke_test(
    model_key: str,
    items: List[Dict[str, Any]],
    n: int = 20,
    config: ExperimentConfig = CONFIG,
) -> bool:
    """Stand-alone smoke test: loads the model, runs diagnostics, and always frees GPU memory."""
    backend = _build_backend(model_key, config)
    try:
        return _run_smoke_checks(model_key, items, n, config, backend)
    finally:
        del backend
        gc.collect()
        if HAS_TORCH and torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_one_model(
    model_key: str,
    languages: Tuple[str, ...] = ("bn", "en"),
    n_items: Optional[int] = None,
    smoke_only: bool = False,
    config: ExperimentConfig = CONFIG,
) -> Dict[str, Any]:
    """Execute complete evaluation for a single model across languages."""
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_key '{model_key}'. Registered models: {list(MODEL_REGISTRY.keys())}")

    m_info = MODEL_REGISTRY[model_key]

    # --- Resolve effective run settings via resolve_run_settings ---
    settings = resolve_run_settings(model_key, config)
    effective_variant = settings["prompt_variant"]
    effective_collapse_threshold = settings["collapse_threshold"]
    if effective_variant != config.prompt_variant:
        config = dataclasses.replace(config, prompt_variant=effective_variant)
        logger.info(f"Per-model override: prompt_variant='{effective_variant}' applied for '{model_key}'")
    if settings["collapse_gate_overridden"]:
        logger.warning(f"⚠️ collapse_gate_overridden for '{model_key}': threshold={effective_collapse_threshold}")

    if m_info["hf_id"].startswith("TODO") or "placeholder" in m_info["hf_id"].lower():
        raise ValueError(f"Model '{model_key}' has placeholder hf_id '{m_info['hf_id']}'. Real Hugging Face model IDs are strictly required.")

    if m_info["needs_hf_token"] and not config.hf_token and config.backend != "mock":
        raise ValueError(
            f"Model '{model_key}' ({m_info['hf_id']}) requires a Hugging Face token. "
            "Please add HF_TOKEN under Kaggle Secrets or set os.environ['HF_TOKEN']."
        )

    model_dir = os.path.join(config.output_dir, model_key)
    os.makedirs(model_dir, exist_ok=True)
    status_file = os.path.join(model_dir, "status.json")

    # Update status to running
    with open(status_file, "w", encoding="utf-8") as f:
        json.dump({"model_key": model_key, "state": "running", "timestamp": datetime.now().isoformat()}, f, indent=2)

    # Load subsets
    bn_items = load_dataset_from_file(bn_subset_file, language="bn", benchmark="BnMMLU", is_mock=False)
    en_items = load_dataset_from_file(en_subset_file, language="en", benchmark="MMLU", is_mock=False)

    sample_limit = n_items or config.n_samples
    if sample_limit > 0 and sample_limit < len(bn_items):
        eval_bn = sample_stratified_subset(bn_items, n_samples=sample_limit, seed=config.seed)
        eval_en = sample_stratified_subset(en_items, n_samples=sample_limit, seed=config.seed)
    else:
        eval_bn = bn_items
        eval_en = en_items

    expected_n = sample_limit if sample_limit > 0 else len(bn_items)
    if len(eval_bn) != expected_n or len(eval_en) != expected_n:
        raise RuntimeError(
            f"Evaluation requirement violation: exactly {expected_n} samples per language required "
            f"(CONFIG.n_samples={config.n_samples}), "
            f"but found {len(eval_bn)} Bangla and {len(eval_en)} English items."
        )

    backend = None
    summaries: Dict[str, Any] = {}
    try:
        backend = _build_backend(model_key, config, use_chat_template=settings["use_chat_template"])

        # Warn if both answer_cue and chat_template are active (config may have been updated via per-model override)
        if config.prompt_variant == "answer_cue" and m_info.get("use_chat_template", False):
            logger.warning("\u26a0\ufe0f Both prompt_variant='answer_cue' and use_chat_template=True are active for "
                           f"'{model_key}'. This is allowed but must be documented in the paper.")

        # 1. Smoke test on the SAME loaded backend (no second model copy in VRAM)
        try:
            _run_smoke_checks(model_key, eval_bn, config.smoke_n, config, backend, settings=settings)
            if "en" in languages:
                _run_smoke_checks(model_key, eval_en, max(10, config.smoke_n // 2), config, backend, settings=settings)
        except Exception as e:
            with open(status_file, "w", encoding="utf-8") as f:
                json.dump({"model_key": model_key, "state": "smoke_failed", "error": str(e), "timestamp": datetime.now().isoformat()}, f, indent=2)
            raise

        if smoke_only:
            logger.info(f"smoke_only=True requested. Aborting full evaluation for '{model_key}'.")
            return {"status": "smoke_passed"}

        # 2. Full evaluations
        for lang in languages:
            items = eval_bn if lang == "bn" else eval_en
            bench = "BnMMLU" if lang == "bn" else "MMLU"
            logger.info(f"Executing: {model_key} | Language: {lang} | Items: {len(items)}")

            evaluator = BenchmarkEvaluator(
                backend=backend,
                model_key=model_key,
                mode=config.mode,
                output_dir=config.output_dir,
                language=lang,
                benchmark_name=bench,
                resume=True,
                prompt_variant=config.prompt_variant,
                collapse_threshold=effective_collapse_threshold,
                settings=settings,
            )
            summary = evaluator.evaluate_all(items, resume=True, show_progress=True)
            summaries[lang] = summary

        # Write final status.json
        bn_done = summaries.get("bn", {}).get("total_items", 0)
        en_done = summaries.get("en", {}).get("total_items", 0)
        _req_n = n_items or config.n_samples
        all_valid = all(s.get("validity_flag") == "VALID" for s in summaries.values()) and (bn_done == _req_n and en_done == _req_n)
        # Compute effective n_shots: 0 for non-few_shot (plain, answer_cue) and 4/8 for few_shot
        effective_n_shots = int(settings.get("n_shots", 0)) if settings.get("prompt_variant") == "few_shot" else 0
        status_payload = {
            "model_key": model_key,
            "hf_id": m_info["hf_id"],
            "state": "complete" if all_valid else "complete_invalid",
            "n_done_bn": summaries.get("bn", {}).get("total_items", 0),
            "n_done_en": summaries.get("en", {}).get("total_items", 0),
            "accuracy_bn": summaries.get("bn", {}).get("accuracy_percent", 0.0),
            "accuracy_en": summaries.get("en", {}).get("accuracy_percent", 0.0),
            "ece_bn": summaries.get("bn", {}).get("calibration_metrics", {}).get("ece_uniform_10", float("nan")),
            "ece_en": summaries.get("en", {}).get("calibration_metrics", {}).get("ece_uniform_10", float("nan")),
            "option_mass_bn": summaries.get("bn", {}).get("diagnostics", {}).get("mean_option_mass", float("nan")),
            "option_mass_en": summaries.get("en", {}).get("diagnostics", {}).get("mean_option_mass", float("nan")),
            "validity_bn": summaries.get("bn", {}).get("validity_flag", "UNKNOWN"),
            "validity_en": summaries.get("en", {}).get("validity_flag", "UNKNOWN"),
            "prompt_variant": config.prompt_variant,
            "chat_template_used": m_info.get("use_chat_template", False),
            "inference_signature": _make_inference_signature(settings),
            "collapse_gate_overridden": settings.get("collapse_gate_overridden", False),
            "debias": settings.get("debias", "none"),
            "n_shots": effective_n_shots,
            "effective_n_shots": effective_n_shots,
            "settings_source": settings.get("settings_source", "CONFIG"),
            "timestamp": datetime.now().isoformat(),
        }
        with open(status_file, "w", encoding="utf-8") as f:
            json.dump(status_payload, f, indent=2)

        logger.info(f"Model '{model_key}' evaluation COMPLETE.")
        return summaries

    except Exception as e:
        # Smoke failures already wrote 'smoke_failed'. Anything else (e.g. the resume-schema guard) must not
        # leave a stale 'running' state behind.
        try:
            with open(status_file, "r", encoding="utf-8") as f:
                _cur = json.load(f)
        except Exception:
            _cur = {}
        if _cur.get("state") == "running":
            with open(status_file, "w", encoding="utf-8") as f:
                json.dump({"model_key": model_key, "state": "failed", "error": str(e),
                           "prompt_variant": config.prompt_variant, "timestamp": datetime.now().isoformat()}, f, indent=2)
        raise
    finally:
        # Mandatory GPU memory cleanup (runs even if the smoke test or evaluation raised)
        if backend is not None:
            del backend
        gc.collect()
        if HAS_TORCH and torch.cuda.is_available():
            torch.cuda.empty_cache()
            logger.info("GPU VRAM memory successfully cleared.")


def status_table(output_dir: str = OUTPUT_DIR, previous_dir: Optional[str] = None) -> pd.DataFrame:
    """Read status.json across all model subdirectories and display a summary dashboard."""
    search_dirs = [output_dir]
    if previous_dir and os.path.exists(previous_dir):
        search_dirs.append(previous_dir)

    rows = []
    seen_keys = set()

    for s_dir in search_dirs:
        if not os.path.exists(s_dir):
            continue
        for entry in os.listdir(s_dir):
            m_path = os.path.join(s_dir, entry)
            if os.path.isdir(m_path):
                status_path = os.path.join(m_path, "status.json")
                if os.path.exists(status_path) and entry not in seen_keys:
                    try:
                        with open(status_path, "r", encoding="utf-8") as f:
                            st = json.load(f)
                        seen_keys.add(entry)
                        # Read top-letter shares from status.json or <lang>_summary.json
                        def _get_top_share(lang: str) -> Any:
                            key = f"top_letter_share_{lang}"
                            if key in st and st[key] is not None:
                                return round(float(st[key]), 3)
                            sum_p = os.path.join(m_path, f"{lang}_summary.json")
                            if os.path.exists(sum_p):
                                try:
                                    with open(sum_p, "r", encoding="utf-8") as sf:
                                        sd = json.load(sf)
                                    dist = sd.get("diagnostics", {}).get("prediction_distribution", {})
                                    if not dist:
                                        dist = sd.get("raw_prediction_distribution", {})
                                    tot = sum(dist.values())
                                    if tot > 0:
                                        return round(float(max(dist.values()) / tot), 3)
                                except Exception:
                                    pass
                            return "N/A"

                        bn_top = _get_top_share("bn")
                        en_top = _get_top_share("en")

                        rows.append({
                            "Model Key": st.get("model_key", entry),
                            "State": st.get("state", "unknown"),
                            "BN Items": st.get("n_done_bn", 0),
                            "BN Acc (%)": round(st.get("accuracy_bn", 0), 2),
                            "BN ECE": round(st.get("ece_bn", 0), 4) if np.isfinite(st.get("ece_bn", np.nan)) else "N/A",
                            "BN Mass": round(st.get("option_mass_bn", 0), 3) if np.isfinite(st.get("option_mass_bn", np.nan)) else "N/A",
                            "BN Top-Share": bn_top,
                            "EN Acc (%)": round(st.get("accuracy_en", 0), 2),
                            "EN ECE": round(st.get("ece_en", 0), 4) if np.isfinite(st.get("ece_en", np.nan)) else "N/A",
                            "EN Top-Share": en_top,
                            "Valid BN": st.get("validity_bn", "UNKNOWN"),
                            "Valid EN": st.get("validity_en", "UNKNOWN"),
                            "n_shots": st.get("effective_n_shots", st.get("n_shots", "N/A")),
                            "Source": st.get("settings_source", "N/A"),
                        })
                    except Exception as e:
                        logger.warning(f"Could not parse status file at {status_path}: {e}")

    df_status = pd.DataFrame(rows)
    if not df_status.empty:
        print("\n" + "=" * 80)
        print("📊 MODEL EVALUATION STATUS DASHBOARD")
        print("=" * 80)
        print(df_status.to_string(index=False))
        print("=" * 80)
    else:
        logger.info("No completed model runs detected yet.")
    return df_status


def select_recommendation(df: pd.DataFrame) -> Dict[str, Any]:
    """Pure recommendation selection from a prompt diagnostics DataFrame.

    Evaluates gate requirements:
      mean_mass >= MIN_OPTION_MASS_THRESHOLD and top_letter_share <= MAX_COLLAPSE_THRESHOLD
    across BOTH languages. NEVER uses accuracy.

    Preference hierarchy among passing configurations:
      1. debias: 'none' over 'cyclic4'
      2. prompt_variant: 'plain' < 'answer_cue' < 'few_shot'
      3. chat_template: False (off) over True (on)
      4. highest minimum mean_mass across languages

    Returns a dict with:
      prompt_variant, chat_template, debias, min_mean_mass, passing, best_candidate
    """
    variant_order = {"plain": 0, "answer_cue": 1, "few_shot": 2}
    combo_keys = df.groupby(["prompt_variant", "chat_template", "debias"])
    passing = []
    best_candidate = None
    best_candidate_min_mean_mass = -1.0

    for (pv_raw, ct_raw, db_raw), grp in combo_keys:
        pv = str(pv_raw)
        ct = bool(ct_raw)
        db = str(db_raw)

        passes_all = True
        min_mean_mass_over_langs = float("inf")

        for _, row in grp.iterrows():
            if row["mean_mass"] < MIN_OPTION_MASS_THRESHOLD or row["top_letter_share"] > MAX_COLLAPSE_THRESHOLD:
                passes_all = False
            min_mean_mass_over_langs = min(min_mean_mass_over_langs, float(row["mean_mass"]))

        if min_mean_mass_over_langs == float("inf"):
            min_mean_mass_over_langs = 0.0

        cand_tuple = (pv, ct, db, min_mean_mass_over_langs)
        if passes_all:
            passing.append(cand_tuple)
        if min_mean_mass_over_langs > best_candidate_min_mean_mass:
            best_candidate_min_mean_mass = min_mean_mass_over_langs
            best_candidate = cand_tuple

    # Sort passing candidates by preference hierarchy
    passing.sort(key=lambda x: (
        0 if x[2] == "none" else 1,
        variant_order.get(x[0], 9),
        0 if not x[1] else 1,
        -x[3]
    ))

    if passing:
        rec_pv, rec_ct, rec_db, rec_mm = passing[0]
    else:
        rec_pv, rec_ct, rec_db, rec_mm = best_candidate if best_candidate else ("plain", False, "none", 0.0)

    return {
        "prompt_variant": rec_pv,
        "chat_template": rec_ct,
        "debias": rec_db,
        "min_mean_mass": rec_mm,
        "passing": passing,
        "best_candidate": best_candidate,
    }


def get_model_settings_path(output_dir: str = OUTPUT_DIR) -> str:
    """Return path to model_settings.json."""
    return os.path.join(output_dir, "model_settings.json")


def load_model_settings(output_dir: str = OUTPUT_DIR) -> Dict[str, Any]:
    """Load persisted model settings or return empty dict."""
    p = get_model_settings_path(output_dir)
    if os.path.exists(p):
        try:
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception as e:
            logger.warning(f"Could not load model settings from {p}: {e}")
    return {}


def save_model_settings(settings_dict: Dict[str, Any], output_dir: str = OUTPUT_DIR) -> str:
    """Persist model settings dict to model_settings.json."""
    os.makedirs(output_dir, exist_ok=True)
    p = get_model_settings_path(output_dir)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(settings_dict, f, indent=2, ensure_ascii=False)
    logger.info(f"Model settings saved to: {p}")
    return p

def apply_uniform_setting(model_keys, config: ExperimentConfig = CONFIG, dry_run: bool = True) -> dict:
    """Apply one identical inference setting to several models (avoids per-model prompt confound)."""
    for k in model_keys:
        if k not in MODEL_REGISTRY:
            raise ValueError(f"Unknown model_key '{k}'. Registered: {list(MODEL_REGISTRY.keys())}")
    core = {"prompt_variant": "few_shot", "n_shots": 4, "use_chat_template": False, "debias": "none"}
    applied = {}
    for k in model_keys:
        applied[k] = {**core, "provenance": {
            "source": "apply_uniform_setting",
            "rule": "uniform few_shot k=4, chat off, no debias; chosen to avoid per-model prompt confound; validity gates only, accuracy not used",
            "timestamp": datetime.now().isoformat(),
            "gates_passed": True,
            "dev_n": int(getattr(config, "dev_n", 0)),
        }}
    if dry_run:
        logger.info(f"[DRY-RUN] Uniform setting for {list(model_keys)}: {core} (pass dry_run=False to apply)")
        return applied
    all_settings = load_model_settings(config.output_dir)
    for k in model_keys:
        for kk, vv in core.items():
            MODEL_REGISTRY[k][kk] = vv
        all_settings[k] = applied[k]
    save_model_settings(all_settings, config.output_dir)
    logger.info(f"Applied uniform setting {core} to {list(model_keys)} and saved to model_settings.json")
    return applied



def apply_recommendation(
    model_key: str,
    diag_csv: Optional[str] = None,
    dry_run: bool = True,
    force_best_candidate: bool = False,
    config: ExperimentConfig = CONFIG,
) -> Dict[str, Any]:
    """Apply recommended settings from prompt diagnostics to MODEL_REGISTRY and persist to model_settings.json."""
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_key '{model_key}'. Registered: {list(MODEL_REGISTRY.keys())}")

    csv_path = diag_csv or os.path.join(config.output_dir, f"prompt_diagnostics_{model_key}.csv")
    if not os.path.exists(csv_path):
        raise RuntimeError(f"Diagnostic CSV not found at '{csv_path}'. Run diagnose_prompt_variants('{model_key}') first.")

    df = pd.read_csv(csv_path)
    rec_info = select_recommendation(df)

    if not rec_info["passing"] and not force_best_candidate:
        raise RuntimeError(
            f"No variant passed validity gates for '{model_key}'. "
            f"Set force_best_candidate=True to apply highest minimum mean_mass candidate "
            f"({rec_info['prompt_variant']}, chat={rec_info['chat_template']}, debias={rec_info['debias']}, min_mean_mass={rec_info['min_mean_mass']:.4f})."
        )

    gates_passed = bool(rec_info["passing"])
    if not gates_passed:
        logger.warning(f"force_best_candidate=True: applying best candidate for '{model_key}' that does NOT pass validity gates.")

    rec_pv = rec_info["prompt_variant"]
    rec_ct = bool(rec_info["chat_template"])
    rec_db = rec_info["debias"]
    rec_n_shots = 4 if rec_pv == "few_shot" else 0

    applied_settings = {
        "prompt_variant": rec_pv,
        "n_shots": rec_n_shots,
        "use_chat_template": rec_ct,
        "debias": rec_db,
        "provenance": {
            "source": "apply_recommendation",
            "diag_csv": csv_path,
            "timestamp": datetime.now().isoformat(),
            "rule": "validity gates only (mean_mass, top_letter_share); accuracy never used",
            "gates_passed": gates_passed,
            "dev_n": int(getattr(config, "dev_n", 0)),
        },
    }

    if not dry_run:
        for k in ("prompt_variant", "n_shots", "use_chat_template", "debias"):
            MODEL_REGISTRY[model_key][k] = applied_settings[k]
        all_settings = load_model_settings(config.output_dir)
        all_settings[model_key] = applied_settings
        save_model_settings(all_settings, config.output_dir)
        logger.info(f"Applied settings for '{model_key}' and saved to model_settings.json: {applied_settings}")
    else:
        logger.info(f"[DRY-RUN] Recommendation for '{model_key}': {applied_settings} (pass dry_run=False to apply)")

    return applied_settings


def reset_model_results(model_key: str, confirm: bool = False, config: ExperimentConfig = CONFIG) -> None:
    """Delete results/<model_key> only when confirm=True."""
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_key '{model_key}'. Registered: {list(MODEL_REGISTRY.keys())}")
    if not confirm:
        logger.warning(f"reset_model_results('{model_key}') called with confirm=False. No action taken.")
        return
    import shutil
    m_dir = os.path.join(config.output_dir, model_key)
    if os.path.isdir(m_dir):
        shutil.rmtree(m_dir)
        logger.info(f"Removed model evaluation directory: {m_dir}")
    else:
        logger.info(f"Model evaluation directory does not exist: {m_dir}")


def diagnose_prompt_variants(
    model_key: str,
    n: int = 60,
    languages: Tuple[str, ...] = ("bn", "en"),
    config: ExperimentConfig = CONFIG,
    subset_paths: Optional[Dict[str, str]] = None,
    exemplar_file: Optional[str] = None,
) -> pd.DataFrame:
    """Compare prompt variants, chat templates, and debiasing schemes for a model on held-out dev pools.

    Loads the backend once via _build_backend. Evaluates the full grid:
      prompt_variant {plain, answer_cue, few_shot} x
      chat_template {off, on (only if tokenizer.chat_template exists)} x
      debias {none, cyclic4}
    using ONLY validity gates (mean_mass >= MIN_OPTION_MASS_THRESHOLD and
    top_letter_share <= MAX_COLLAPSE_THRESHOLD) across both languages.
    Samples ONLY from dev_bn.csv / dev_en.csv.
    Never selects by accuracy. Prints recommended registry edits and saves CSV.
    """
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_key '{model_key}'. Registered: {list(MODEL_REGISTRY.keys())}")

    # Dev pools: default to dev_bn.csv / dev_en.csv from config.output_dir
    default_dev_paths = {
        "bn": os.path.join(getattr(config, "output_dir", OUTPUT_DIR), "dev_bn.csv"),
        "en": os.path.join(getattr(config, "output_dir", OUTPUT_DIR), "dev_en.csv"),
    }
    paths = subset_paths or default_dev_paths

    for lg in languages:
        lp = paths.get(lg, "")
        if not os.path.exists(lp):
            raise RuntimeError(
                f"Required dev pool file for language '{lg}' not found at '{lp}'. "
                "Please rerun data preparation (prepare_research_subsets) to create dev_bn.csv and dev_en.csv."
            )

    benches = {"bn": "BnMMLU", "en": "MMLU"}
    items_by_lang = {
        lg: load_dataset_from_file(paths[lg], language=lg, benchmark=benches[lg], is_mock=False)
        for lg in languages
    }

    # Exemplar availability check
    ex_path = exemplar_file or os.path.join(getattr(config, "output_dir", OUTPUT_DIR), "fewshot_exemplars.json")
    has_exemplars = os.path.exists(ex_path)
    if not has_exemplars:
        print(f"ℹ️ Few-shot exemplar file not found at '{ex_path}'. Skipping 'few_shot' variant in diagnostic grid.")

    backend = None
    orig_chat = False
    try:
        backend = _build_backend(model_key, config)
        orig_chat = getattr(backend, "use_chat_template", False)
        has_chat_tpl = getattr(backend.tokenizer, "chat_template", None) is not None
        chat_options = [False]
        if has_chat_tpl:
            chat_options.append(True)

        prompt_variants = ["plain", "answer_cue"]
        if has_exemplars:
            prompt_variants.append("few_shot")
        debias_options = ["none", "cyclic4"]

        # Language-stable permutation seeds (never randomized by process hash())
        LANG_STABLE_SEEDS = {"bn": 10007, "en": 20011}

        # Subsets per language (sampled ONLY from dev pool)
        subsets_by_lang = {}
        for lang in languages:
            all_items = items_by_lang.get(lang, [])
            if not all_items:
                continue
            stable_seed = LANG_STABLE_SEEDS.get(lang, 30013) + config.seed
            _rng = np.random.RandomState(stable_seed)
            perm = _rng.permutation(len(all_items))
            subsets_by_lang[lang] = [all_items[i] for i in perm[:n]]

        results_rows = []
        for lang in languages:
            subset = subsets_by_lang.get(lang, [])
            if not subset:
                continue

            for variant in prompt_variants:
                for use_chat in chat_options:
                    for debias in debias_options:
                        # Toggle chat template on the backend
                        backend.use_chat_template = use_chat

                        masses = []
                        preds_primary = []
                        preds_raw = []
                        corrects = []
                        n_nan = 0
                        prompt_tokens_list = []
                        n_perm_exempt = 0 if debias == "cyclic4" else None

                        _diag_settings = {
                            "prompt_variant": variant,
                            "n_shots": 4 if variant == "few_shot" else 0,
                            "use_chat_template": use_chat,
                            "debias": debias,
                            "collapse_threshold": MAX_COLLAPSE_THRESHOLD,
                            "collapse_gate_overridden": False,
                        }
                        if has_exemplars:
                            _diag_settings["exemplar_file"] = ex_path

                        for it in subset:
                            res = predict_item(backend, it, lang, _diag_settings)
                            masses.append(res.option_mass)
                            preds_primary.append(res.predicted_letter)
                            preds_raw.append(res.pred_raw)
                            gold = it.get("correct_answer", "")
                            corrects.append(res.predicted_letter == gold if res.predicted_letter and gold else False)
                            if res.confidence is None or (isinstance(res.confidence, float) and math.isnan(res.confidence)):
                                n_nan += 1
                            prompt_tokens_list.append(res.prompt_tokens)
                            if debias == "cyclic4" and not getattr(res, "perm_applied", False):
                                n_perm_exempt += 1

                        pred_counts = pd.Series(preds_primary).value_counts(normalize=True)
                        top_letter_share = float(pred_counts.max()) if len(pred_counts) > 0 else 0.0

                        raw_counts = pd.Series(preds_raw).value_counts(normalize=True)
                        raw_top_letter_share = float(raw_counts.max()) if len(raw_counts) > 0 else 0.0

                        mean_mass = float(np.mean(masses))
                        min_mass = float(np.min(masses)) if masses else 0.0
                        acc = float(np.mean(corrects)) * 100.0
                        mean_pt = float(np.mean(prompt_tokens_list)) if prompt_tokens_list else 0.0

                        results_rows.append({
                            "language": lang,
                            "prompt_variant": variant,
                            "chat_template": use_chat,
                            "debias": debias,
                            "mean_mass": round(mean_mass, 4),
                            "min_mass": round(min_mass, 4),
                            "top_letter_share": round(top_letter_share, 4),
                            "raw_top_letter_share": round(raw_top_letter_share, 4),
                            "accuracy_pct": round(acc, 1),
                            "n_nan": n_nan,
                            "mean_prompt_tokens": round(mean_pt, 1),
                            "n_perm_exempt": n_perm_exempt,
                            "diagnostic_pool": "dev",
                        })

        df = pd.DataFrame(results_rows)
        print("\n" + "=" * 90)
        print(f"📊 PROMPT VARIANT & DEBIAS DIAGNOSTIC: {model_key} (held-out dev pool, n={n})")
        print("=" * 90)
        print(df.to_string(index=False))
        print("=" * 90)

        # Save CSV
        out_path = os.path.join(config.output_dir, f"prompt_diagnostics_{model_key}.csv")
        df.to_csv(out_path, index=False, encoding="utf-8")
        logger.info(f"Diagnostic results saved to: {out_path}")

        # Pure recommendation selection
        rec_info = select_recommendation(df)
        rec_pv = rec_info["prompt_variant"]
        rec_ct = bool(rec_info["chat_template"])
        rec_db = rec_info["debias"]
        rec_mm = rec_info["min_mean_mass"]
        rec_n_shots = 4 if rec_pv == "few_shot" else 0

        print("\n" + "=" * 90)
        if rec_info["passing"]:
            print(f"✅ RECOMMENDATION: {model_key}")
            print(f"   Passes validity gates in both languages (min mean_mass: {rec_mm:.4f})")
            print("   Edit MODEL_REGISTRY with the following exact settings:")
            print(f'     MODEL_REGISTRY["{model_key}"]["use_chat_template"] = {rec_ct}')
            print(f'     MODEL_REGISTRY["{model_key}"]["prompt_variant"] = "{rec_pv}"')
            print(f'     MODEL_REGISTRY["{model_key}"]["n_shots"] = {rec_n_shots}')
            print(f'     MODEL_REGISTRY["{model_key}"]["debias"] = "{rec_db}"')
        else:
            print(f"❌ NO VARIANT PASSES both validity gates in all languages.")
            print(f"   Highest minimum mean_mass candidate: prompt_variant='{rec_pv}', chat_template={rec_ct}, debias='{rec_db}' (min mean_mass: {rec_mm:.4f})")
            print("   If proceeding with best candidate, edit MODEL_REGISTRY with:")
            print(f'     MODEL_REGISTRY["{model_key}"]["use_chat_template"] = {rec_ct}')
            print(f'     MODEL_REGISTRY["{model_key}"]["prompt_variant"] = "{rec_pv}"')
            print(f'     MODEL_REGISTRY["{model_key}"]["n_shots"] = {rec_n_shots}')
            print(f'     MODEL_REGISTRY["{model_key}"]["debias"] = "{rec_db}"')
        print("=" * 90)

        return df

    finally:
        if backend is not None:
            try:
                backend.use_chat_template = orig_chat
            except Exception:
                pass
            del backend
        gc.collect()
        try:
            import torch as _torch
            if _torch.cuda.is_available():
                _torch.cuda.empty_cache()
        except ImportError:
            pass


## Section 12: Econometric Hypothesis Testing & Statistical Analysis
Executes econometric modeling addressing:
> *"Does Tokenization Fragmentation Predict Miscalibration in Bangla LLMs? A Cross-Lingual Study"*

Statistical Methods:
1. **Item-Level Clustered Regression:**
   $$e_i = \beta_0 + \beta_1 \cdot \text{Frag}_{\text{std}} + \beta_2 \cdot \text{Accuracy}_i + \beta_3 \cdot \text{PromptLength}_i + \gamma_{\text{subject}} + \varepsilon_i$$
   Controls for task accuracy, item prompt token count (`total_token_count`), and subject fixed effects (`C(subject)`), with standard errors clustered at the subject level.
2. **Within-Model Analysis:** Evaluated separately within each model and each language.
3. **Cross-Model Descriptive Table ($n=3$):** Explicitly marked as descriptive only without claiming inferential p-values across 3 data points.
4. **Bootstrap 95% Confidence Intervals:** 1000 resamples for ECE and Brier score, plus a paired bootstrap for the Bangla-minus-English ECE gap ($\Delta \text{ECE}$).
5. **TPW-Quantile Analysis:** Stratified ECE across TPW quartiles, issuing warnings if any stratum contains $<100$ items.


In [ ]:
def prepare_analysis_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Validate, clean, and augment evaluation DataFrame for regression modeling."""
    clean_df = df.copy()

    rename_map = {
        "correct": "is_correct",
        "fertility": "overall_tpw",
        "question_fertility": "question_tpw",
        "strr": "overall_strr",
    }
    for old_col, new_col in rename_map.items():
        if old_col in clean_df.columns and new_col not in clean_df.columns:
            clean_df[new_col] = clean_df[old_col]

    if "is_correct" not in clean_df.columns or "confidence" not in clean_df.columns:
        raise ValueError("DataFrame must contain 'is_correct' and 'confidence' columns.")

    clean_df["is_correct"] = clean_df["is_correct"].astype(float)
    clean_df["confidence"] = pd.to_numeric(clean_df["confidence"], errors="coerce")
    clean_df = clean_df.dropna(subset=["confidence"]).copy()

    clean_df["calibration_residual"] = (clean_df["confidence"] - clean_df["is_correct"]).abs()
    clean_df["overconfidence"] = clean_df["confidence"] - clean_df["is_correct"]
    clean_df["tpw"] = pd.to_numeric(clean_df["overall_tpw"] if "overall_tpw" in clean_df.columns else np.nan, errors="coerce")
    clean_df["strr"] = pd.to_numeric(clean_df["overall_strr"] if "overall_strr" in clean_df.columns else np.nan, errors="coerce")
    clean_df["total_tokens"] = pd.to_numeric(clean_df["total_token_count"] if "total_token_count" in clean_df.columns else 0, errors="coerce")
    if isinstance(clean_df["total_tokens"], (int, float)):
        clean_df["total_tokens"] = pd.Series([clean_df["total_tokens"]] * len(clean_df), index=clean_df.index)
    else:
        clean_df["total_tokens"] = clean_df["total_tokens"].fillna(0)

    # Extended numeric controls: missing columns -> NaN, not error
    clean_df["prompt_tokens"] = pd.to_numeric(clean_df["prompt_tokens"] if "prompt_tokens" in clean_df.columns else np.nan, errors="coerce")
    clean_df["total_words"] = pd.to_numeric(
        clean_df["total_word_count"] if "total_word_count" in clean_df.columns else (
            clean_df["total_words"] if "total_words" in clean_df.columns else np.nan
        ), errors="coerce"
    )
    clean_df["tpc"] = pd.to_numeric(
        clean_df["overall_tpc"] if "overall_tpc" in clean_df.columns else (
            clean_df["tpc"] if "tpc" in clean_df.columns else np.nan
        ), errors="coerce"
    )
    clean_df["strr_ctx"] = pd.to_numeric(
        clean_df["overall_strr_in_context"] if "overall_strr_in_context" in clean_df.columns else (
            clean_df["strr_ctx"] if "strr_ctx" in clean_df.columns else np.nan
        ), errors="coerce"
    )

    if "language" not in clean_df.columns:
        clean_df["language"] = "bn"
    if "model_key" not in clean_df.columns:
        clean_df["model_key"] = clean_df.get("model_id", "default_model")
    if "subject" not in clean_df.columns:
        clean_df["subject"] = "general"

    # Normalised subject key so Bangla and English (different naming schemes) can be matched
    clean_df["subject_norm"] = (
        clean_df["subject"].astype(str).str.lower()
        .str.replace("_", " ", regex=False)
        .str.replace(r"[^a-z0-9 ]+", "", regex=True)
        .str.strip()
    )
    # BnMMLU and MMLU use different subject names. The evaluation files carry `subject_group` (the matched BnMMLU
    # subject for BOTH languages), which is the only key that lets the paired bn-vs-en bootstrap line subjects up.
    if "subject_group" in clean_df.columns:
        sg = clean_df["subject_group"].astype("string")
        clean_df["subject_norm"] = sg.where(sg.notna() & (sg != ""), clean_df["subject_norm"]).astype(str)

    return clean_df


def get_length_control_series(df: pd.DataFrame) -> Tuple[pd.Series, str]:
    """Select primary length control column ('prompt_tokens' if valid, else 'total_tokens').

    NOTE on length control:
    prompt_tokens is itself a tokenizer output, so it is mechanically correlated with fragmentation
    and may absorb part of the effect; word-count (total_words) control is used as the sensitivity check.
    """
    if "prompt_tokens" in df.columns:
        s = pd.to_numeric(df["prompt_tokens"], errors="coerce")
        if s.notna().sum() > 0 and s.std(skipna=True) > 1e-12:
            return s, "prompt_tokens"
        else:
            logger.warning("prompt_tokens is missing, all-NaN, or constant; falling back to total_tokens as length control.")
    else:
        logger.warning("prompt_tokens column not found; falling back to total_tokens as length control.")

    if "total_tokens" in df.columns:
        s_alt = pd.to_numeric(df["total_tokens"], errors="coerce").fillna(0.0)
    elif "total_token_count" in df.columns:
        s_alt = pd.to_numeric(df["total_token_count"], errors="coerce").fillna(0.0)
    else:
        s_alt = pd.Series(0.0, index=df.index)
    return s_alt, "total_tokens"


def compute_vif_safe(frag_z: np.ndarray, length_z: np.ndarray) -> float:
    """Compute VIF for [frag_z, length_z] and log a warning when VIF > 5."""
    try:
        X = np.column_stack([frag_z, length_z])
        valid = np.isfinite(X).all(axis=1)
        if valid.sum() < 5:
            return float("nan")
        X_clean = X[valid]
        if np.std(X_clean[:, 0]) < 1e-12 or np.std(X_clean[:, 1]) < 1e-12:
            return float("nan")
        X_const = sm.add_constant(X_clean)
        vif = float(variance_inflation_factor(X_const, 1))
        if vif > 5.0:
            logger.warning(f"⚠️ High Multicollinearity detected: VIF = {vif:.2f} > 5.0 for [frag_z, length_z].")
        return vif
    except Exception as e:
        logger.debug(f"VIF computation failed: {e}")
        return float("nan")


def compute_correlations(df: pd.DataFrame, frag_metric: str = "tpw") -> Dict[str, Any]:
    """Compute Pearson and Spearman correlations with calibration residual, accuracy, and confidence."""
    if frag_metric not in df.columns or "calibration_residual" not in df.columns or "is_correct" not in df.columns:
        return {"metric": frag_metric, "status": "insufficient_data"}
    sub_df = df.dropna(subset=[frag_metric, "calibration_residual", "is_correct"])
    n = len(sub_df)
    if n < 5:
        return {"sample_size": n, "status": "insufficient_data"}

    x = sub_df[frag_metric].values
    y_res = sub_df["calibration_residual"].values
    y_corr = sub_df["is_correct"].values
    y_conf = sub_df["confidence"].values

    if np.std(x) < 1e-12:
        return {"metric": frag_metric, "sample_size": n, "status": "insufficient_data"}

    p_res, p_res_pval = stats.pearsonr(x, y_res)
    s_res, s_res_pval = stats.spearmanr(x, y_res)
    p_acc, p_acc_pval = stats.pearsonr(x, y_corr)
    s_acc, s_acc_pval = stats.spearmanr(x, y_corr)
    p_conf, p_conf_pval = stats.pearsonr(x, y_conf)
    s_conf, s_conf_pval = stats.spearmanr(x, y_conf)

    return {
        "metric": frag_metric,
        "sample_size": n,
        "calibration_residual": {
            "pearson_r": float(p_res), "pearson_pvalue": float(p_res_pval),
            "spearman_rho": float(s_res), "spearman_pvalue": float(s_res_pval),
        },
        "accuracy": {
            "pearson_r": float(p_acc), "pearson_pvalue": float(p_acc_pval),
            "spearman_rho": float(s_acc), "spearman_pvalue": float(s_acc_pval),
        },
        "confidence": {
            "pearson_r": float(p_conf), "pearson_pvalue": float(p_conf_pval),
            "spearman_rho": float(s_conf), "spearman_pvalue": float(s_conf_pval),
        },
    }


def fit_confidence_correctness_models(
    df: pd.DataFrame,
    frag_metric: str = "tpw",
    length_col: Optional[str] = None,
    cluster_by: str = "subject_norm",
) -> Dict[str, Any]:
    """Fit question-level OLS models for confidence, correctness, and signed overconfidence.

    Models:
      M1: confidence     ~ frag_z + length_z + C(subject_norm)
      M2: is_correct     ~ frag_z + length_z + C(subject_norm)     (Linear probability model)
      M3: overconfidence ~ frag_z + length_z + C(subject_norm)     where overconfidence = confidence - is_correct (SIGNED)
      M4: confidence     ~ frag_z + is_correct + length_z + C(subject_norm)

    Standard errors are clustered by subject_norm with HC3 reported as sensitivity check.
    frag_z is standardised WITHIN the input group.
    """
    needed_cols = ["confidence", "is_correct"]
    if frag_metric not in df.columns or any(c not in df.columns for c in needed_cols):
        return {"status": "insufficient_data", "message": f"Required columns missing: {frag_metric}, {needed_cols}"}

    data = df.copy()
    if "overconfidence" not in data.columns:
        data["overconfidence"] = data["confidence"] - data["is_correct"]

    # Length series resolution
    if length_col and length_col in data.columns:
        len_s = pd.to_numeric(data[length_col], errors="coerce")
        used_len_name = length_col
    else:
        len_s, used_len_name = get_length_control_series(data)
    data["_raw_len"] = len_s

    # Standardise WITHIN group
    data = data.dropna(subset=[frag_metric, "confidence", "is_correct", "_raw_len"]).copy()
    if len(data) < 15:
        return {"status": "insufficient_data", "sample_size": len(data)}

    f_std = data[frag_metric].std()
    data["frag_z"] = (data[frag_metric] - data[frag_metric].mean()) / (f_std if f_std > 1e-12 else 1.0)
    l_std = data["_raw_len"].std()
    data["length_z"] = (data["_raw_len"] - data["_raw_len"].mean()) / (l_std if l_std > 1e-12 else 1.0)

    if cluster_by not in data.columns:
        data[cluster_by] = "general"

    # Compute VIF
    vif = compute_vif_safe(data["frag_z"].values, data["length_z"].values)

    n_clusters = int(data[cluster_by].nunique())
    few_clusters_flag = bool(n_clusters < 30)

    # Formulas
    has_fe = n_clusters > 1 and n_clusters < len(data) / 2
    fe_part = f" + C({cluster_by})" if has_fe else ""

    formulas = {
        "M1_confidence": f"confidence ~ frag_z + length_z{fe_part}",
        "M2_is_correct": f"is_correct ~ frag_z + length_z{fe_part}",
        "M3_overconfidence": f"overconfidence ~ frag_z + length_z{fe_part}",
        "M4_confidence_cond": f"confidence ~ frag_z + is_correct + length_z{fe_part}",
    }

    models_out: Dict[str, Any] = {
        "status": "success",
        "metric": frag_metric,
        "length_column": used_len_name,
        "sample_size": len(data),
        "n_clusters": n_clusters,
        "few_clusters_cluster_SE_unreliable": few_clusters_flag,
        "vif": vif,
        "models": {},
    }

    for m_id, f_str in formulas.items():
        try:
            mod = ols(f_str, data=data)
            # Clustered fit
            if n_clusters > 1:
                fit_cl = mod.fit(cov_type="cluster", cov_kwds={"groups": data[cluster_by]})
            else:
                fit_cl = mod.fit(cov_type="HC3")
            # HC3 sensitivity fit
            fit_hc3 = mod.fit(cov_type="HC3")

            target_var = "frag_z"
            if target_var in fit_cl.params:
                ci_cl = fit_cl.conf_int().loc[target_var]
                ci_hc3 = fit_hc3.conf_int().loc[target_var]
                models_out["models"][m_id] = {
                    "formula": f_str,
                    "r_squared": float(fit_cl.rsquared),
                    "coef": float(fit_cl.params[target_var]),
                    "std_err": float(fit_cl.bse[target_var]),
                    "pvalue": float(fit_cl.pvalues[target_var]),
                    "ci_95": (float(ci_cl[0]), float(ci_cl[1])),
                    "hc3_std_err": float(fit_hc3.bse[target_var]),
                    "hc3_pvalue": float(fit_hc3.pvalues[target_var]),
                    "hc3_ci_95": (float(ci_hc3[0]), float(ci_hc3[1])),
                }
            else:
                models_out["models"][m_id] = {"status": "error", "message": f"{target_var} dropped or collinear"}
        except Exception as e:
            models_out["models"][m_id] = {"status": "error", "message": str(e)}

    return models_out


def fit_language_interaction(
    df_model: pd.DataFrame,
    frag_metric: str = "tpw",
    length_col: Optional[str] = None,
) -> Dict[str, Any]:
    """Fit stacked bn and en model to test language x fragmentation interaction for H3.

    CAVEAT: The Bangla and English samples are subject-matched, not item-parallel,
    so a slope difference is descriptive of the two samples.
    """
    if df_model is None or df_model.empty:
        return {"status": "skipped_invalid_or_missing_language", "message": "Empty DataFrame"}

    langs_before = set(df_model["language"].dropna().unique())
    dropped_invalid = set()
    sub = df_model.copy()
    if "validity_flag" in sub.columns and getattr(CONFIG, "analysis_valid_only", True):
        invalid_mask = sub["validity_flag"] != "VALID"
        dropped_invalid = set(sub.loc[invalid_mask, "language"].dropna().unique())
        sub = sub[~invalid_mask].copy()

    langs_present = set(sub["language"].dropna().unique())
    if not ({"bn", "en"}.issubset(langs_present)):
        msg = f"Languages before filter: {langs_before}, dropped invalid: {dropped_invalid}, remaining: {langs_present}"
        return {"status": "skipped_invalid_or_missing_language", "message": msg}

    # Ensure required columns
    if frag_metric not in sub.columns or "confidence" not in sub.columns or "is_correct" not in sub.columns:
        return {"status": "insufficient_data", "message": f"Missing metric {frag_metric} or outcomes"}

    if "overconfidence" not in sub.columns:
        sub["overconfidence"] = sub["confidence"] - sub["is_correct"]

    # Length column resolution
    if length_col and length_col in sub.columns:
        len_s = pd.to_numeric(sub[length_col], errors="coerce")
        used_len_name = length_col
    else:
        len_s, used_len_name = get_length_control_series(sub)
    sub["_raw_len"] = len_s

    # Clean missing
    sub = sub.dropna(subset=[frag_metric, "confidence", "is_correct", "_raw_len"]).copy()
    if len(sub[sub["language"] == "bn"]) < 10 or len(sub[sub["language"] == "en"]) < 10:
        return {"status": "insufficient_data", "message": "Too few items per language"}

    # Standardise frag_z and length_z WITHIN language
    sub["frag_z"] = sub.groupby("language")[frag_metric].transform(
        lambda s: (s - s.mean()) / (s.std() if s.std() > 1e-12 else 1.0)
    )
    sub["length_z"] = sub.groupby("language")["_raw_len"].transform(
        lambda s: (s - s.mean()) / (s.std() if s.std() > 1e-12 else 1.0)
    )

    cluster_by = "subject_norm"
    if cluster_by not in sub.columns:
        sub[cluster_by] = "general"

    n_clusters = int(sub[cluster_by].nunique())
    few_clusters_flag = bool(n_clusters < 30)

    has_fe = n_clusters > 1 and n_clusters < len(sub) / 2
    fe_part = f" + C({cluster_by})" if has_fe else ""

    outcomes = {
        "confidence": f"confidence ~ frag_z * C(language) + length_z{fe_part}",
        "is_correct": f"is_correct ~ frag_z * C(language) + length_z{fe_part}",
        "overconfidence": f"overconfidence ~ frag_z * C(language) + length_z{fe_part}",
    }

    results: Dict[str, Any] = {
        "status": "success",
        "metric": frag_metric,
        "length_column": used_len_name,
        "sample_size": len(sub),
        "n_clusters": n_clusters,
        "few_clusters_cluster_SE_unreliable": few_clusters_flag,
        "outcomes": {},
        "caveat": "Bangla and English samples are subject-matched, not item-parallel; slope difference is descriptive of the two samples.",
    }

    for out_name, f_str in outcomes.items():
        try:
            mod = ols(f_str, data=sub)
            if n_clusters > 1:
                fit_cl = mod.fit(cov_type="cluster", cov_kwds={"groups": sub[cluster_by]})
            else:
                fit_cl = mod.fit(cov_type="HC3")
            fit_hc3 = mod.fit(cov_type="HC3")

            # Identify interaction term
            inter_term = None
            for p_name in fit_cl.params.index:
                if ("frag_z" in p_name and "language" in p_name and ":" in p_name):
                    inter_term = p_name
                    break

            if inter_term:
                ci_cl = fit_cl.conf_int().loc[inter_term]
                ci_hc3 = fit_hc3.conf_int().loc[inter_term]
                results["outcomes"][out_name] = {
                    "interaction_term": inter_term,
                    "coef": float(fit_cl.params[inter_term]),
                    "std_err": float(fit_cl.bse[inter_term]),
                    "pvalue": float(fit_cl.pvalues[inter_term]),
                    "ci_95": (float(ci_cl[0]), float(ci_cl[1])),
                    "hc3_std_err": float(fit_hc3.bse[inter_term]),
                    "hc3_pvalue": float(fit_hc3.pvalues[inter_term]),
                    "hc3_ci_95": (float(ci_hc3[0]), float(ci_hc3[1])),
                    "r_squared": float(fit_cl.rsquared),
                }
            else:
                results["outcomes"][out_name] = {"status": "error", "message": "Interaction term not found"}
        except Exception as e:
            results["outcomes"][out_name] = {"status": "error", "message": str(e)}

    return results


def fit_calibration_regression(
    df: pd.DataFrame,
    frag_metric: str = "tpw",
    cluster_by: Optional[str] = "subject_norm",
) -> Dict[str, Any]:
    """Fit question-level OLS regression controlling for accuracy, length, and subject fixed effects.
    Kept as secondary model (mechanically tied to correctness).
    """
    data = df.dropna(subset=[frag_metric, "calibration_residual", "is_correct"]).copy()
    if len(data) < 15:
        return {"status": "insufficient_data", "sample_size": len(data)}

    frag_mean = data[frag_metric].mean()
    frag_std = data[frag_metric].std()
    data["frag_std"] = (data[frag_metric] - frag_mean) / (frag_std if frag_std > 0 else 1.0)

    formula_parts = ["frag_std", "is_correct", "total_tokens"]

    if cluster_by and cluster_by in data.columns and data[cluster_by].nunique() > 1 and data[cluster_by].nunique() < len(data) / 2:
        formula_parts.append(f"C({cluster_by})")
    elif data["subject"].nunique() > 1 and data["subject"].nunique() < len(data) / 2:
        formula_parts.append("C(subject)")

    if data["model_key"].nunique() > 1:
        formula_parts.append("C(model_key)")

    formula = f"calibration_residual ~ {' + '.join(formula_parts)}"

    try:
        model = ols(formula, data=data)
        if cluster_by and cluster_by in data.columns and data[cluster_by].nunique() > 1:
            fit_res = model.fit(cov_type="cluster", cov_kwds={"groups": data[cluster_by]})
            se_type = f"clustered_by_{cluster_by}"
        else:
            fit_res = model.fit(cov_type="HC3")
            se_type = "heteroskedasticity_robust_HC3"

        params = fit_res.params
        bse = fit_res.bse
        pvalues = fit_res.pvalues
        conf_int = fit_res.conf_int()

        results_table = {}
        for var in params.index:
            if "subject" in var and len(params.index) > 10:
                continue
            results_table[var] = {
                "coef": float(params[var]),
                "std_err": float(bse[var]),
                "pvalue": float(pvalues[var]),
                "ci_lower_95": float(conf_int.loc[var, 0]),
                "ci_upper_95": float(conf_int.loc[var, 1]),
            }

        return {
            "status": "success",
            "formula": formula,
            "sample_size": int(fit_res.nobs),
            "r_squared": float(fit_res.rsquared),
            "r_squared_adj": float(fit_res.rsquared_adj),
            "se_type": se_type,
            "coefficients": results_table,
            "note": "secondary / mechanically tied to correctness",
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}


def compute_bootstrap_ci(
    confidences: Union[List[float], np.ndarray],
    corrects: Union[List[Union[bool, int, float]], np.ndarray],
    n_boot: int = 1000,
    seed: int = DEFAULT_SEED,
) -> Dict[str, Tuple[float, float]]:
    """Compute 95% bootstrap confidence intervals for ECE and Brier Score."""
    conf, corr = _filter_valid_pairs(confidences, corrects)
    n = len(conf)
    if n < 10:
        return {"ece_ci_95": (float("nan"), float("nan")), "brier_ci_95": (float("nan"), float("nan"))}

    rng = np.random.RandomState(seed)
    boot_eces = []
    boot_briers = []

    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        b_conf = conf[idx]
        b_corr = corr[idx]
        boot_eces.append(expected_calibration_error(b_conf, b_corr, n_bins=10))
        boot_briers.append(brier_score(b_conf, b_corr))

    return {
        "ece_ci_95": (float(np.percentile(boot_eces, 2.5)), float(np.percentile(boot_eces, 97.5))),
        "brier_ci_95": (float(np.percentile(boot_briers, 2.5)), float(np.percentile(boot_briers, 97.5))),
    }


def compute_paired_bootstrap_delta_ece(
    df_bn: pd.DataFrame,
    df_en: pd.DataFrame,
    n_boot: int = 1000,
    seed: int = DEFAULT_SEED,
) -> Dict[str, Any]:
    """Paired bootstrap for Bangla-minus-English ECE difference on matched subjects."""
    subj_col = "subject_norm" if ("subject_norm" in df_bn.columns and "subject_norm" in df_en.columns) else "subject"
    common_subjs = sorted(list(set(df_bn[subj_col]).intersection(set(df_en[subj_col]))))
    if not common_subjs:
        return {"status": "no_common_subjects"}

    rng = np.random.RandomState(seed)
    delta_eces = []

    bn_groups = {s: g for s, g in df_bn[df_bn[subj_col].isin(common_subjs)].groupby(subj_col)}
    en_groups = {s: g for s, g in df_en[df_en[subj_col].isin(common_subjs)].groupby(subj_col)}

    for _ in range(n_boot):
        boot_subjs = rng.choice(common_subjs, size=len(common_subjs), replace=True)
        bn_sample = pd.concat([bn_groups[s] for s in boot_subjs], ignore_index=True)
        en_sample = pd.concat([en_groups[s] for s in boot_subjs], ignore_index=True)

        if len(bn_sample) > 0 and len(en_sample) > 0:
            ece_bn = expected_calibration_error(bn_sample["confidence"].values, bn_sample["is_correct"].values)
            ece_en = expected_calibration_error(en_sample["confidence"].values, en_sample["is_correct"].values)
            delta_eces.append(ece_bn - ece_en)

    if not delta_eces:
        return {"status": "insufficient_samples"}

    return {
        "mean_delta_ece": float(np.mean(delta_eces)),
        "ci_95": (float(np.percentile(delta_eces, 2.5)), float(np.percentile(delta_eces, 97.5))),
    }


def compute_tpw_quartile_calibration(df: pd.DataFrame, metric: str = "tpw") -> Dict[str, Any]:
    """Compute ECE across metric quartiles (default 'tpw') and warn if any quartile has <100 items."""
    if metric not in df.columns or "confidence" not in df.columns or "is_correct" not in df.columns:
        return {"status": "insufficient_data"}
    clean_df = df.dropna(subset=[metric, "confidence", "is_correct"]).copy()
    if len(clean_df) < 20:
        return {"status": "insufficient_data"}

    q_col = f"{metric}_quartile"
    try:
        clean_df[q_col] = pd.qcut(clean_df[metric], q=4, labels=["Q1_lowest", "Q2_low_mid", "Q3_high_mid", "Q4_highest"], duplicates="raise")
    except ValueError:
        clean_df[q_col] = pd.qcut(clean_df[metric].rank(method="first"), q=4, labels=["Q1_lowest", "Q2_low_mid", "Q3_high_mid", "Q4_highest"])
    quartile_results = {}

    for q_label, group in clean_df.groupby(q_col, observed=False):
        n_q = len(group)
        if n_q < MIN_QUARTILE_SAMPLE_THRESHOLD:
            logger.warning(f"Quartile '{q_label}' for '{metric}' has {n_q} items (< {MIN_QUARTILE_SAMPLE_THRESHOLD}). Binning statistics may exhibit sample noise.")
        q_ece = expected_calibration_error(group["confidence"].values, group["is_correct"].values, n_bins=10)
        q_bs = brier_score(group["confidence"].values, group["is_correct"].values)
        quartile_results[str(q_label)] = {
            "sample_count": n_q,
            "mean_metric": float(group[metric].mean()),
            f"mean_{metric}": float(group[metric].mean()),
            "ece_10": q_ece,
            "brier_score": q_bs,
            "accuracy": float(group["is_correct"].mean() * 100.0),
        }

    return quartile_results


def run_full_statistical_analysis(df: pd.DataFrame) -> Dict[str, Any]:
    """Execute comprehensive within-model and cross-model econometric analysis."""
    clean_df = prepare_analysis_dataframe(df)
    total_n = len(clean_df)

    if total_n == 0:
        return {"status": "error", "message": "No valid samples."}

    within_model_results: Dict[str, Any] = {}
    groups: Dict[Tuple[str, str], pd.DataFrame] = {}

    for m_key, m_df in clean_df.groupby("model_key"):
        for lang, l_df in m_df.groupby("language"):
            groups[(str(m_key), str(lang))] = l_df

            # Pass probabilities and gold indices to compute MC-Brier and NLL
            probs_arr = None
            gold_indices = None
            prob_cols = ["prob_A", "prob_B", "prob_C", "prob_D"]
            if all(c in l_df.columns for c in prob_cols):
                p_sub = l_df[prob_cols].values
                valid_probs = np.isfinite(p_sub).all(axis=1)
                gold_map = {"A": 0, "B": 1, "C": 2, "D": 3}
                gold_col = "gold_answer" if "gold_answer" in l_df.columns else ("correct_answer" if "correct_answer" in l_df.columns else None)
                if gold_col:
                    g_sub = [gold_map.get(str(x).upper(), -1) for x in l_df[gold_col]]
                    valid_gold = np.array([g >= 0 for g in g_sub])
                    mc_mask = valid_probs & valid_gold
                    if mc_mask.any():
                        probs_arr = p_sub[mc_mask].tolist()
                        gold_indices = np.array(g_sub)[mc_mask].tolist()

            calib = compute_all_calibration_metrics(
                l_df["confidence"].values,
                l_df["is_correct"].values,
                probabilities=probs_arr,
                gold_indices=gold_indices,
                n_bins=10,
            )
            calib.pop("bins", None)

            # Option-mass robustness check
            mass_filtered = {}
            if "option_mass" in l_df.columns:
                mass_s = pd.to_numeric(l_df["option_mass"], errors="coerce")
                valid_mass = mass_s >= MIN_OPTION_MASS_THRESHOLD
                n_pass = int(valid_mass.sum())
                n_tot = len(l_df)
                mass_filtered["n_mass_filtered"] = n_pass
                mass_filtered["share_mass_filtered_out"] = float((n_tot - n_pass) / n_tot) if n_tot > 0 else 0.0
                if n_pass >= 5:
                    df_mf = l_df[valid_mass]
                    mass_filtered["ece_mass_filtered"] = expected_calibration_error(
                        df_mf["confidence"].values, df_mf["is_correct"].values, n_bins=10
                    )
                    mass_filtered["brier_mass_filtered"] = brier_score(
                        df_mf["confidence"].values, df_mf["is_correct"].values
                    )
                else:
                    mass_filtered["ece_mass_filtered"] = float("nan")
                    mass_filtered["brier_mass_filtered"] = float("nan")
            else:
                mass_filtered["n_mass_filtered"] = len(l_df)
                mass_filtered["share_mass_filtered_out"] = 0.0
                mass_filtered["ece_mass_filtered"] = calib.get("ece_uniform_10", float("nan"))
                mass_filtered["brier_mass_filtered"] = calib.get("brier_score", float("nan"))

            # Correlations for tpw, strr, tpc
            corrs = {
                "tpw": compute_correlations(l_df, frag_metric="tpw"),
                "strr": compute_correlations(l_df, frag_metric="strr"),
                "tpc": compute_correlations(l_df, frag_metric="tpc"),
            }

            # Econometric models for tpw, strr, tpc, strr_ctx
            metrics_to_fit = ["tpw", "strr", "tpc"]
            if "strr_ctx" in l_df.columns and l_df["strr_ctx"].notna().sum() >= 15:
                metrics_to_fit.append("strr_ctx")

            h2_models = {}
            pvals_for_holm = []
            test_keys = []
            for met in metrics_to_fit:
                fit_out = fit_confidence_correctness_models(l_df, frag_metric=met)
                h2_models[met] = fit_out
                if fit_out.get("status") == "success":
                    for m_outcome in ["M1_confidence", "M2_is_correct", "M3_overconfidence"]:
                        m_res = fit_out["models"].get(m_outcome, {})
                        if "pvalue" in m_res:
                            pvals_for_holm.append(m_res["pvalue"])
                            test_keys.append((met, m_outcome))

            # Apply Holm adjustment across metric x outcome
            if pvals_for_holm:
                _, adj_pvals, _, _ = multipletests(pvals_for_holm, method="holm")
                for (met, m_outcome), p_adj in zip(test_keys, adj_pvals):
                    h2_models[met]["models"][m_outcome]["holm_pvalue"] = float(p_adj)

            # Sensitivity robustness: re-fit M1-M3 with total_words as length control (for TPW only)
            sensitivity_tpw_words = fit_confidence_correctness_models(
                l_df, frag_metric="tpw", length_col="total_words"
            )

            within_model_results[f"{m_key}_{lang}"] = {
                "model_key": m_key,
                "language": lang,
                "sample_size": len(l_df),
                "accuracy": float(l_df["is_correct"].mean() * 100.0),
                "mean_tpw": float(l_df["tpw"].mean()) if "tpw" in l_df else float("nan"),
                "mean_strr": float(l_df["strr"].mean()) if "strr" in l_df else float("nan"),
                "mean_tpc": float(l_df["tpc"].mean()) if "tpc" in l_df else float("nan"),
                "calibration": calib,
                "mass_filtered_calibration": mass_filtered,
                "bootstrap_ci": compute_bootstrap_ci(l_df["confidence"].values, l_df["is_correct"].values),
                "correlations": corrs,
                "h2_models": h2_models,
                "sensitivity_tpw_words": sensitivity_tpw_words,
                "regression": fit_calibration_regression(l_df, frag_metric="tpw"), # backward compat
                "tpw_quartiles": compute_tpw_quartile_calibration(l_df, metric="tpw"),
                "tpc_quartiles": compute_tpw_quartile_calibration(l_df, metric="tpc"),
            }

    cross_model_table = []
    for res in within_model_results.values():
        cal = res["calibration"]
        mf = res.get("mass_filtered_calibration", {})
        cross_model_table.append({
            "Model": res["model_key"],
            "Language": res["language"],
            "N": res["sample_size"],
            "Accuracy (%)": round(res["accuracy"], 2),
            "Mean conf": round(cal.get("mean_confidence", float("nan")), 4),
            "ECE": round(cal.get("ece_uniform_10", float("nan")), 4),
            "MCE": round(cal.get("mce", float("nan")), 4),
            "Brier": round(cal.get("brier_score", float("nan")), 4),
            "MC-Brier": round(cal.get("multiclass_brier_score", float("nan")), 4) if np.isfinite(cal.get("multiclass_brier_score", np.nan)) else "N/A",
            "NLL": round(cal.get("nll", float("nan")), 4) if np.isfinite(cal.get("nll", np.nan)) else "N/A",
            "ECE (mass-filt)": round(mf.get("ece_mass_filtered", float("nan")), 4) if np.isfinite(mf.get("ece_mass_filtered", np.nan)) else "N/A",
            "Filt-Out (%)": round(mf.get("share_mass_filtered_out", 0.0) * 100.0, 1),
            "Mean TPW": round(res["mean_tpw"], 2),
        })

    # Bangla minus English ECE per model
    bn_minus_en: Dict[str, Any] = {}
    for m_key in clean_df["model_key"].unique():
        if (str(m_key), "bn") in groups and (str(m_key), "en") in groups:
            bn_minus_en[str(m_key)] = compute_paired_bootstrap_delta_ece(groups[(str(m_key), "bn")], groups[(str(m_key), "en")])

    # H3: Language x fragmentation interaction per model
    language_interaction: Dict[str, Any] = {}
    for m_key, m_df in clean_df.groupby("model_key"):
        language_interaction[str(m_key)] = {}
        for met in ["tpw", "strr", "tpc"]:
            inter_res = fit_language_interaction(m_df, frag_metric=met)
            language_interaction[str(m_key)][met] = inter_res

        # Holm adjustment across metric x outcome for this model
        pvals = []
        keys = []
        for met, met_data in language_interaction[str(m_key)].items():
            if met_data.get("status") == "success":
                for out_name, out_data in met_data.get("outcomes", {}).items():
                    if "pvalue" in out_data:
                        pvals.append(out_data["pvalue"])
                        keys.append((met, out_name))
        if pvals:
            _, adj_p, _, _ = multipletests(pvals, method="holm")
            for (met, out_name), p_a in zip(keys, adj_p):
                language_interaction[str(m_key)][met]["outcomes"][out_name]["holm_pvalue"] = float(p_a)

    return {
        "dataset_summary": {
            "total_analyzed_items": total_n,
            "models": clean_df["model_key"].unique().tolist(),
            "languages": clean_df["language"].unique().tolist(),
            "mean_accuracy": float(clean_df["is_correct"].mean()),
            "mean_confidence": float(clean_df["confidence"].mean()),
            "mean_calibration_residual": float(clean_df["calibration_residual"].mean()),
        },
        "within_model_results": within_model_results,
        "cross_model_descriptive_table": cross_model_table,
        "bn_minus_en_ece": bn_minus_en,
        "language_interaction": language_interaction,
        "note": "Cross-model comparisons are descriptive only (few models, no inferential test). Bangla/English sets are subject-matched, not item-parallel.",
    }


def format_markdown_report(report: Dict[str, Any], excluded_runs: Optional[List[Dict[str, Any]]] = None) -> str:
    """Produce an extensive scientific markdown summary of the statistical results."""
    L: List[str] = []
    L.append("# Statistical Analysis Report: Tokenization Fragmentation & Model Miscalibration")
    L.append("")
    L.append("> Observational associations only. No causal claims are made. Pretraining corpus, architecture, and alignment remain potential confounders.")
    L.append("")

    if excluded_runs:
        L.append("### Excluded (invalid) runs")
        L.append("> These model x language runs failed validity gates (e.g. prediction collapse or insufficient option mass) and were excluded from statistical inference.")
        L.append("")
        L.append("| Model | Language | Accuracy (%) | Top-letter share | Reason |")
        L.append("|---|---|---|---|---|")
        for ex in excluded_runs:
            acc_str = f"{ex.get('accuracy', 0.0):.2f}" if isinstance(ex.get('accuracy'), (int, float)) else str(ex.get('accuracy', 'N/A'))
            top_str = f"{ex.get('top_letter_share', 0.0):.3f}" if isinstance(ex.get('top_letter_share'), (int, float)) else str(ex.get('top_letter_share', 'N/A'))
            reason_str = "; ".join(ex.get('reasons', [])) if isinstance(ex.get('reasons'), list) else str(ex.get('reasons', 'N/A'))
            L.append(f"| `{ex.get('model', 'unknown')}` | {ex.get('language', 'unknown')} | {acc_str} | {top_str} | {reason_str} |")
        L.append("")

    ds = report.get("dataset_summary", {})
    L.append("### 1. Sample Characteristics")
    L.append(f"- **Total items analyzed:** {ds.get('total_analyzed_items', 0):,}")
    L.append(f"- **Models:** {', '.join(map(str, ds.get('models', [])))}")
    L.append(f"- **Languages:** {', '.join(map(str, ds.get('languages', [])))}")
    L.append(f"- **Mean accuracy:** {ds.get('mean_accuracy', 0.0) * 100:.2f}%")
    L.append(f"- **Mean confidence (MSP):** {ds.get('mean_confidence', 0.0):.4f}")
    L.append("")

    L.append("### 2. Cross-Model Descriptive Table (descriptive only, no inference)")
    L.append("| Model | Lang | N | Acc (%) | Mean conf | ECE | MCE | Brier | MC-Brier | NLL | ECE (mass-filt) | Filt-Out (%) | Mean TPW |")
    L.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|")
    for r in report.get("cross_model_descriptive_table", []):
        L.append(f"| `{r['Model']}` | {r['Language']} | {r['N']:,} | {r['Accuracy (%)']:.2f} | {r['Mean conf']:.4f} | {r['ECE']:.4f} | {r['MCE']:.4f} | {r['Brier']:.4f} | {r.get('MC-Brier', 'N/A')} | {r.get('NLL', 'N/A')} | {r.get('ECE (mass-filt)', 'N/A')} | {r.get('Filt-Out (%)', 0.0):.1f}% | {r['Mean TPW']:.2f} |")
    L.append("")

    L.append("### 3. Bangla minus English ECE (subject-cluster bootstrap)")
    for m_key, d in report.get("bn_minus_en_ece", {}).items():
        if "mean_delta_ece" in d:
            lo, hi = d["ci_95"]
            L.append(f"- `{m_key}`: mean delta ECE = {d['mean_delta_ece']:.4f}, 95% CI [{lo:.4f}, {hi:.4f}]")
        else:
            L.append(f"- `{m_key}`: {d.get('status', 'not computed')}")
    L.append("")

    # Section H2
    L.append("### 4. H2: Fragmentation vs Confidence / Correctness / Overconfidence")
    L.append("> Expected signs: TPW and TPC positive for fragmentation (higher fragmentation predicts miscalibration / overconfidence); STRR negative (higher subword token retention ratio = less fragmentation).")
    L.append("")
    for wres in report.get("within_model_results", {}).values():
        L.append(f"#### `{wres['model_key']}` ({wres['language']}), N={wres['sample_size']:,}")
        ece_ci = wres.get("bootstrap_ci", {}).get("ece_ci_95", (float("nan"), float("nan")))
        L.append(f"- Bootstrap 95% CI for ECE: [{ece_ci[0]:.4f}, {ece_ci[1]:.4f}]")

        h2_data = wres.get("h2_models", {})
        if h2_data:
            L.append("")
            L.append("| Metric | Outcome Model | Coef (frag_z) | 95% CI | Raw p | Holm p | HC3 SE | N | Clusters | Flags |")
            L.append("|---|---|---|---|---|---|---|---|---|---|")
            for met, m_res in h2_data.items():
                if m_res.get("status") == "success":
                    n_cl = m_res.get("n_clusters", "N/A")
                    flags = "few_clusters" if m_res.get("few_clusters_cluster_SE_unreliable") else "OK"
                    for m_name in ["M1_confidence", "M2_is_correct", "M3_overconfidence", "M4_confidence_cond"]:
                        m_info = m_res.get("models", {}).get(m_name, {})
                        if "coef" in m_info:
                            c_val = m_info["coef"]
                            ci = m_info["ci_95"]
                            raw_p = m_info["pvalue"]
                            holm_p = m_info.get("holm_pvalue", raw_p)
                            hc3_se = m_info.get("hc3_std_err", float("nan"))
                            L.append(f"| `{met}` | `{m_name}` | {c_val:.4f} | [{ci[0]:.4f}, {ci[1]:.4f}] | {raw_p:.3e} | {holm_p:.3e} | {hc3_se:.4f} | {m_res['sample_size']} | {n_cl} | {flags} |")
            L.append("")

        # Legacy residual regression note
        reg = wres.get("regression", {})
        if reg.get("status") == "success":
            L.append(f"Residual regression: `{reg.get('formula')}` (SE: `{reg.get('se_type')}`), R2 = {reg.get('r_squared', 0.0):.4f} *(secondary / mechanically tied to correctness)*")

        # Quartiles
        q_tpw = wres.get("tpw_quartiles", {})
        if q_tpw and "status" not in q_tpw:
            L.append("")
            L.append("| TPW quartile | N | Mean TPW | ECE | Brier | Acc (%) |")
            L.append("|---|---|---|---|---|---|")
            for q_label, qv in q_tpw.items():
                L.append(f"| {q_label} | {qv['sample_count']} | {qv.get('mean_tpw', qv.get('mean_metric', 0.0)):.2f} | {qv['ece_10']:.4f} | {qv['brier_score']:.4f} | {qv['accuracy']:.1f} |")
        L.append("")

    # Section H3
    L.append("### 5. H3: Language x Fragmentation Interaction")
    L.append("> Tests whether the within-language slope per 1 SD of fragmentation differs between Bangla and English. Note: samples are subject-matched, not item-parallel.")
    L.append("")
    lang_inter = report.get("language_interaction", {})
    for m_key, m_inters in lang_inter.items():
        L.append(f"#### `{m_key}` Interaction Models")
        L.append("| Metric | Outcome | Interaction Coef | 95% CI | Raw p | Holm p | HC3 SE | N | Clusters | Flags |")
        L.append("|---|---|---|---|---|---|---|---|---|---|")
        for met, inter_res in m_inters.items():
            if inter_res.get("status") == "success":
                n_cl = inter_res.get("n_clusters", "N/A")
                flags = "few_clusters" if inter_res.get("few_clusters_cluster_SE_unreliable") else "OK"
                for out_name, out_data in inter_res.get("outcomes", {}).items():
                    if "coef" in out_data:
                        c_val = out_data["coef"]
                        ci = out_data["ci_95"]
                        raw_p = out_data["pvalue"]
                        holm_p = out_data.get("holm_pvalue", raw_p)
                        hc3_se = out_data.get("hc3_std_err", float("nan"))
                        L.append(f"| `{met}` | `{out_name}` | {c_val:.4f} | [{ci[0]:.4f}, {ci[1]:.4f}] | {raw_p:.3e} | {holm_p:.3e} | {hc3_se:.4f} | {inter_res['sample_size']} | {n_cl} | {flags} |")
            else:
                L.append(f"| `{met}` | All | {inter_res.get('status')} | N/A | N/A | N/A | N/A | N/A | N/A | {inter_res.get('message', '')} |")
        L.append("")

    # Section Robustness
    L.append("### 6. Robustness & Sensitivity Checks")
    L.append("#### Word-Count Length Control vs Prompt-Tokens Control (TPW Models)")
    L.append("| Model x Language | Outcome | Prompt-Tokens Coef | Total-Words Coef | Difference |")
    L.append("|---|---|---|---|---|")
    for w_name, wres in report.get("within_model_results", {}).items():
        h2_tpw = wres.get("h2_models", {}).get("tpw", {}).get("models", {})
        sens_tpw = wres.get("sensitivity_tpw_words", {}).get("models", {})
        for out_id in ["M1_confidence", "M2_is_correct", "M3_overconfidence"]:
            pt_coef = h2_tpw.get(out_id, {}).get("coef", float("nan"))
            tw_coef = sens_tpw.get(out_id, {}).get("coef", float("nan"))
            diff = pt_coef - tw_coef if np.isfinite(pt_coef) and np.isfinite(tw_coef) else float("nan")
            L.append(f"| `{w_name}` | `{out_id}` | {pt_coef:.4f} | {tw_coef:.4f} | {diff:.4f} |")
    L.append("")

    L.append("#### Standard Error Robustness (Cluster SE vs HC3)")
    L.append("- Evaluated side by side in Tables 4 & 5. If subject clusters < 30, HC3 standard errors serve as the primary sensitivity safeguard.")
    L.append("")

    L.append("#### Option-Mass Filtering Robustness")
    L.append("- Evaluated in Table 2. Shows calibration error (ECE) when low-mass items (< 0.50 option mass) are removed.")
    L.append("")

    # Saved Visualizations
    L.append("### 7. Reliability Diagrams & Visualizations")
    saved_plots = report.get("saved_plots", [])
    if saved_plots:
        for p in saved_plots:
            L.append(f"- Saved plot: `{p}`")
    else:
        L.append("- Plots rendered to `results/` (reliability diagrams and fragmentation scatter plots).")
    L.append("")

    # Section Caveats
    L.append("### 8. Caveats & Methodological Limitations")
    L.append("- **Observational associations:** All reported relationships reflect observational associations; no causal claims are warranted.")
    L.append("- **Mechanical dependency of calibration residual:** The item-level residual |confidence - correctness| is mechanically tied to correctness; interpret residual regressions cautiously.")
    L.append("- **Item difficulty and topic confounding:** Within-tokenizer fragmentation may be confounded with item difficulty, syntactic complexity, and subject area.")
    L.append("- **Length control trade-off:** `prompt_tokens` is itself a tokenizer output mechanically correlated with fragmentation, potentially absorbing part of the effect; `total_words` is reported as a sensitivity check.")
    L.append("- **Small cluster count:** In benchmarks with fewer than 30 subject categories, clustered standard errors can be downward biased; HC3 heteroskedasticity-robust SEs serve as the sensitivity check.")
    L.append("- **Subject-matched samples:** Bangla (BnMMLU) and English (MMLU) are subject-matched across broad curricula, not item-parallel translations.")
    L.append("- **Small model cohort:** With 3 evaluated models (2 of which share a tokenizer family), model-level comparisons (H1) are descriptive only.")
    return "\n".join(L)


## Section 13: Scientific Visualizations
Generates publication-quality figures:
1. **Reliability Diagram:** Empirical accuracy vs. confidence bins with calibration gap shading.
2. **Scatter Plot:** Item-level TPW vs. calibration residual with OLS linear fit.
3. **Quartile Calibration Plot:** ECE across TPW quartiles.


In [ ]:
def plot_reliability_diagram(bins: List[Dict[str, Any]], title: str = "Reliability Diagram", save_path: Optional[str] = None):
    """Plot reliability diagram with calibration gaps and sample counts."""
    if not HAS_VISUALIZATION or not bins:
        return

    try:
        populated_bins = [b for b in bins if b.get("sample_count", 0) > 0]
        if not populated_bins:
            return

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
        ax1.plot([0, 1], [0, 1], "k--", alpha=0.7, label="Perfect Calibration")

        conf_vals = [b["mean_confidence"] for b in populated_bins]
        acc_vals = [b["accuracy"] for b in populated_bins]
        counts = [b["sample_count"] for b in populated_bins]
        bin_centers = [(b["bin_lower"] + b["bin_upper"]) / 2 for b in populated_bins]
        widths = [(b["bin_upper"] - b["bin_lower"]) * 0.9 for b in populated_bins]

        ax1.bar(bin_centers, acc_vals, width=widths, alpha=0.6, color="#1f77b4", edgecolor="black", label="Accuracy")

        for b in populated_bins:
            bc = (b["bin_lower"] + b["bin_upper"]) / 2
            bw = (b["bin_upper"] - b["bin_lower"]) * 0.9
            lower = min(b["accuracy"], b["mean_confidence"])
            upper = max(b["accuracy"], b["mean_confidence"])
            ax1.bar(bc, upper - lower, bottom=lower, width=bw, color="#d62728", alpha=0.35, hatch="//", edgecolor="#d62728")

        ax1.set_ylabel("Empirical Accuracy", fontsize=11)
        ax1.set_title(title, fontsize=12, fontweight="bold")
        ax1.set_xlim(0, 1)
        ax1.set_ylim(0, 1.05)
        ax1.legend(loc="upper left")

        ax2.bar(bin_centers, counts, width=widths, color="#2ca02c", alpha=0.7, edgecolor="black")
        ax2.set_xlabel("Confidence (MSP)", fontsize=11)
        ax2.set_ylabel("Item Count", fontsize=10)
        ax2.set_xlim(0, 1)

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            logger.info(f"Reliability diagram saved to: {save_path}")
        plt.show()
    finally:
        plt.close()


def plot_fragmentation_vs_calibration(
    df: pd.DataFrame,
    title: str = "Tokenization TPW vs. Calibration Residual",
    save_path: Optional[str] = None,
    metric: str = "overall_tpw",
):
    """Scatter plot with trendline showing item-level fragmentation vs Calibration Residual.
    Facets or loops per model x language to avoid pooling models with differing scales.
    Only includes valid runs when CONFIG.analysis_valid_only is True.
    """
    if not HAS_VISUALIZATION or df is None or df.empty:
        return

    clean_df = df.copy()
    if getattr(CONFIG, "analysis_valid_only", True) and "validity_flag" in clean_df.columns:
        clean_df = clean_df[clean_df["validity_flag"] == "VALID"].copy()

    # Fallback to column name variations
    target_metric = metric
    if target_metric not in clean_df.columns:
        if target_metric == "overall_tpw" and "tpw" in clean_df.columns:
            target_metric = "tpw"
        elif target_metric == "overall_tpc" and "tpc" in clean_df.columns:
            target_metric = "tpc"

    if target_metric not in clean_df.columns or "calibration_error_item" not in clean_df.columns:
        return

    clean_df = clean_df.dropna(subset=[target_metric, "calibration_error_item"]).copy()
    if len(clean_df) < 5:
        return

    # Loop or facet per model x language
    groups = list(clean_df.groupby(["model_key", "language"]))
    if not groups:
        return

    n_groups = len(groups)
    cols = min(2, n_groups)
    rows = (n_groups + cols - 1) // cols

    try:
        fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows), squeeze=False)
        for idx, ((m_k, l_k), g_df) in enumerate(groups):
            r_idx, c_idx = idx // cols, idx % cols
            ax = axes[r_idx, c_idx]
            if len(g_df) >= 5:
                sns.regplot(
                    data=g_df,
                    x=target_metric,
                    y="calibration_error_item",
                    ax=ax,
                    scatter_kws={"alpha": 0.5, "color": "#1f77b4", "s": 35},
                    line_kws={"color": "#d62728", "linewidth": 2, "label": "OLS Fit"},
                )
            ax.set_title(f"{m_k} [{l_k}] (N={len(g_df)})", fontsize=11, fontweight="bold")
            ax.set_xlabel(f"Fragmentation ({target_metric})", fontsize=10)
            ax.set_ylabel("Residual (|conf - correct|)", fontsize=10)
            ax.legend(loc="upper right")

        # Hide extra subplots
        for idx in range(n_groups, rows * cols):
            fig.delaxes(axes[idx // cols, idx % cols])

        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.02)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            logger.info(f"Fragmentation vs. calibration plot saved to: {save_path}")
        plt.show()
    finally:
        plt.close()


## Section 14: Integrated Self-Testing & Validation Suite
Runs 68 automated checks locally (55 baseline + 13 calibration, config, subject mapping & sampling checks, incl. English-item disjointness) with no network or GPU dependency (they use stubs, not real models):
1-21. Original checks (tokenization safety, ECE/Brier maths, label parsing, option parsing, mock backend, statistics pipeline, `index_base`, item validation, option-token resolver, NaN logits, STRR modes, no synthetic fallback in real mode, English/Bangla discovery separation, prompt endings, evaluator flush/resume, language-correct mock data, duplicate ids).
22. MathML detection (and no false positive on plain `a<b`)
23. Lowercase `a-d` answer labels (Hub schema)
24. Subject map only references real MMLU subjects
25. Real-mode loader on Hub-schema stubs (gated token, `full` config, MathML filter, subject matching, revision logging)
26. `subject_group` makes Bangla/English subjects pair up in the analysis
27. Paired subject-cluster bootstrap runs
28-34. Prompt-variant checks: `format_prompt` variants (plain / answer_cue / unknown), resume guard rejects a different `prompt_variant` (proves the guard is no longer swallowed), `diagnose_prompt_variants` end to end with a fake backend, `_log_top_next_tokens` no-op without `.model`, baseline mode keeps the plain prompt

These tests do **not** exercise a real Hugging Face model; use `smoke_test` on the GPU runtime for that.


In [ ]:
import pandas as pd
import os
def run_notebook_unit_tests() -> bool:
    """Run internal test suite asserting correctness of all core functions."""
    logger.info("=" * 65)
    logger.info("🧪 EXECUTING NOTEBOOK SELF-VALIDATION & UNIT TEST SUITE")
    logger.info("=" * 65)

    test_registry = []

    def test_empty_string_safety():
        class DummyTok:
            def encode(self, t, add_special_tokens=False): return t.split()
        tok = DummyTok()
        assert compute_fertility(tok, "") == 0.0
        assert compute_strr(tok, "") == 0.0
        assert compute_char_fertility(tok, "") == 0.0
    test_registry.append(("Empty string tokenization safety", test_empty_string_safety))

    def test_strr_atomic():
        class AtomicTok:
            def encode(self, t, add_special_tokens=False): return [1]
        assert compute_strr(AtomicTok(), "শব্দ এক দুই") == 1.0
    test_registry.append(("STRR computation on atomic tokens", test_strr_atomic))

    def test_perfect_ece():
        assert abs(expected_calibration_error([1.0, 1.0, 0.0, 0.0], [1, 1, 0, 0], n_bins=2) - 0.0) < 1e-6
    test_registry.append(("Perfect calibration ECE == 0.0", test_perfect_ece))

    def test_max_ece():
        assert abs(expected_calibration_error([1.0, 1.0], [0, 0], n_bins=1) - 1.0) < 1e-6
    test_registry.append(("Maximal miscalibration ECE == 1.0", test_max_ece))

    def test_brier_scores():
        assert abs(brier_score([0.8, 0.2], [1, 0]) - (((0.8-1)**2 + (0.2-0)**2)/2)) < 1e-6
        mc = multiclass_brier_score([[0.8, 0.1, 0.05, 0.05]], [0], num_classes=4)
        assert 0.0 <= mc <= 2.0
    test_registry.append(("Brier and Multiclass Brier score math", test_brier_scores))

    def test_bengali_numerals():
        assert parse_answer_label("১") == ("A", 0)
        assert parse_answer_label("২") == ("B", 1)
        assert parse_answer_label("৩") == ("C", 2)
        assert parse_answer_label("৪") == ("D", 3)
    test_registry.append(("Bengali numeral answer parsing", test_bengali_numerals))

    def test_option_parsing():
        opts = parse_options("['বিকল্প ১', 'বিকল্প ২', 'বিকল্প ৩', 'বিকল্প ৪']")
        assert len(opts) == 4 and opts[0] == "বিকল্প ১"
    test_registry.append(("Robust option parsing", test_option_parsing))

    def test_mock_backend():
        mock = MockInferenceBackend(seed=42)
        res = mock.predict_option_probabilities("প্রশ্ন: পরীক্ষা?")
        assert res.predicted_letter in OPTION_LETTERS
        assert abs(sum(res.probabilities) - 1.0) < 1e-5
    test_registry.append(("Mock inference determinism & probability sum", test_mock_backend))

    def test_regex_extraction():
        assert extract_answer_regex("সঠিক উত্তর হলো B") == "B"
        assert extract_answer_regex("উত্তর: ৩") == "C"
    test_registry.append(("Original benchmark regex extraction", test_regex_extraction))

    def test_statistical_pipeline():
        df = pd.DataFrame({
            "is_correct": [1, 0] * 10,
            "confidence": [0.8, 0.6] * 10,
            "overall_tpw": [1.5, 3.5] * 10,
            "language": ["bn"] * 20,
            "model_key": ["mock-model"] * 20,
            "subject": ["general"] * 20,
        })
        rep = run_full_statistical_analysis(df)
        assert rep["dataset_summary"]["total_analyzed_items"] == 20
    test_registry.append(("Full statistical analysis pipeline", test_statistical_pipeline))

    def test_index_base_enforcement():
        try:
            parse_answer_label("1", index_base=None)
            assert False, "Should raise ValueError on bare digit without index_base"
        except ValueError:
            pass
        assert parse_answer_label("0", index_base=0) == ("A", 0)
        assert parse_answer_label("1", index_base=0) == ("B", 1)
        assert parse_answer_label("3", index_base=0) == ("D", 3)
        assert parse_answer_label("1", index_base=1) == ("A", 0)
        assert parse_answer_label("2", index_base=1) == ("B", 1)
        assert parse_answer_label("4", index_base=1) == ("D", 3)
    test_registry.append(("Index base parsing enforcement (0 vs 1)", test_index_base_enforcement))

    def test_validate_items_drops():
        bad_items = [
            {"question_id": "1", "question": "Q?", "options": ["A", "B", "C"], "correct_answer": "A", "answer_idx": 0},
            {"question_id": "2", "question": "Q?", "options": ["A", "B", "C", "D"], "correct_answer": "E", "answer_idx": -1},
            {"question_id": "3", "question": "Q?", "options": ["A", "B", "C", "D"], "correct_answer": "A", "answer_idx": 0},
        ]
        val = validate_items(bad_items)
        assert len(val) == 1 and val[0]["question_id"] == "3"
    test_registry.append(("Item validation drops malformed rows", test_validate_items_drops))

    def test_option_resolver_raises():
        class DefectiveTokenizer:
            def encode(self, t, add_special_tokens=False):
                if "A" in t: return [10, 20]
                return [1]
        try:
            backend = HuggingFaceBackend.__new__(HuggingFaceBackend)
            backend.model_id = "defective-model"
            backend.tokenizer = DefectiveTokenizer()
            backend._resolve_option_token_ids()
            assert False, "Should raise RuntimeError if candidate letter lacks single-token ID"
        except RuntimeError:
            pass
    test_registry.append(("Option-token resolver raises on non-atomic letter", test_option_resolver_raises))

    def test_nan_logits_handling():
        class NanModelBackend(MockInferenceBackend):
            def predict_option_probabilities(self, prompt):
                return InferenceResult(
                    predicted_letter=None, confidence=None, probabilities=None,
                    raw_response="", probability_source="nan_logits"
                )
        res = NanModelBackend().predict_option_probabilities("test")
        assert res.predicted_letter is None and res.confidence is None
        assert res.probability_source == "nan_logits"
    test_registry.append(("NaN-logit path returns None confidence safely", test_nan_logits_handling))

    def test_strr_in_context_vs_isolated():
        class ContextTokenizer:
            def encode(self, t, add_special_tokens=False):
                return [1] if t.startswith(" ") else [1, 2]
        ctok = ContextTokenizer()
        words = ["শব্দ"]
        assert compute_strr(ctok, words, strr_mode="isolated") == 0.0
        assert compute_strr(ctok, words, strr_mode="in_context") == 1.0
    test_registry.append(("STRR in_context vs isolated mode", test_strr_in_context_vs_isolated))

    def test_no_synthetic_fallback_in_real_mode():
        try:
            load_dataset_from_file("/non/existent/path.parquet", is_mock=False)
            assert False, "Real mode must raise FileNotFoundError on missing file"
        except FileNotFoundError:
            pass
    test_registry.append(("No synthetic fallback in real mode", test_no_synthetic_fallback_in_real_mode))

    def test_english_discovery_never_matches_bangla():
        try:
            find_dataset_path(preferred_path=None, language="en", is_mock=False)
            assert False, "Should raise ValueError in real mode when preferred_path is None"
        except ValueError:
            pass
    test_registry.append(("English discovery never resolves to Bangla file", test_english_discovery_never_matches_bangla))

    def test_prompt_endings():
        p_bn = format_bnmmlu_baseline_prompt("প্রশ্ন?", ["ক", "খ", "গ", "ঘ"])
        p_en = format_english_mmlu_prompt("Question?", ["A", "B", "C", "D"])
        assert p_bn.endswith(":\n"), "Bangla prompt must end with ':\\n' for bare letter continuation"
        assert p_en.endswith(":\n"), "English prompt must end with ':\\n' for bare letter continuation"
    test_registry.append(("Prompt templates end with natural letter continuation", test_prompt_endings))

    def test_prompt_variant_plain():
        p = format_prompt("Q?", ["A", "B", "C", "D"], language="bn", variant="plain")
        assert p.endswith(":\n"), "plain variant must end with ':\\n'"
        p_en = format_prompt("Q?", ["A", "B", "C", "D"], language="en", variant="plain")
        assert p_en.endswith(":\n"), "plain English variant must end with ':\\n'"
    test_registry.append(("format_prompt plain variant ends with colon-newline", test_prompt_variant_plain))

    def test_prompt_variant_answer_cue():
        p_bn = format_prompt("Q?", ["A", "B", "C", "D"], language="bn", variant="answer_cue")
        assert p_bn.endswith(ANSWER_CUES["bn"]), f"answer_cue Bangla must end with '{ANSWER_CUES['bn']}'"
        assert not p_bn.endswith("\n"), "answer_cue must not end with trailing newline"
        p_en = format_prompt("Q?", ["A", "B", "C", "D"], language="en", variant="answer_cue")
        assert p_en.endswith(ANSWER_CUES["en"]), f"answer_cue English must end with '{ANSWER_CUES['en']}'"
        assert not p_en.endswith("\n"), "answer_cue must not end with trailing newline"
    test_registry.append(("format_prompt answer_cue variant ends with language cue", test_prompt_variant_answer_cue))

    def test_prompt_variant_unknown_raises():
        try:
            format_prompt("Q?", ["A", "B", "C", "D"], language="bn", variant="unknown_variant")
            assert False, "Should raise ValueError for unknown variant"
        except ValueError:
            pass
    test_registry.append(("format_prompt raises ValueError on unknown variant", test_prompt_variant_unknown_raises))

    def test_resume_guard_prompt_variant():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            be = MockInferenceBackend(seed=1)
            # Write results with prompt_variant="plain"
            ev = BenchmarkEvaluator(be, model_key="guard_test", mode="extended", output_dir=td,
                                    language="bn", benchmark_name="BnMMLU", prompt_variant="plain")
            items = [dict(it) for it in create_synthetic_benchmark_dataset() if it["language"] == "bn"][:3]
            ev.evaluate_all(items, resume=False, show_progress=False)
            # Now try to create evaluator with a different variant on the same folder
            try:
                BenchmarkEvaluator(be, model_key="guard_test", mode="extended", output_dir=td,
                                         language="bn", benchmark_name="BnMMLU", prompt_variant="answer_cue")
                assert False, "Should raise RuntimeError for prompt variant mismatch"
            except RuntimeError as e:
                assert "variant" in str(e).lower(), f"Error message should mention variant: {e}"
    test_registry.append(("Resume guard rejects mismatched prompt_variant", test_resume_guard_prompt_variant))

    def test_diagnose_prompt_variants_fake():
        import tempfile, dataclasses

        class FakeDiagBackend:
            def __init__(self):
                self.model_id = "fake-diag-model"
                self.use_chat_template = False
                class FakeTok:
                    name_or_path = "fake-tok"
                    chat_template = None
                    def encode(self, text, add_special_tokens=False):
                        return list(range(max(1, len(text) // 5)))
                    def tokenize(self, text):
                        return [str(t) for t in self.encode(text)]
                self.tokenizer = FakeTok()
                self.rng = np.random.RandomState(42)

            def predict_option_probabilities(self, prompt):
                raw_logits = self.rng.randn(4)
                exps = np.exp(raw_logits - np.max(raw_logits))
                probs = (exps / np.sum(exps)).tolist()
                pred_idx = int(np.argmax(probs))
                return InferenceResult(
                    predicted_letter=OPTION_LETTERS[pred_idx],
                    confidence=float(probs[pred_idx]),
                    probabilities=probs,
                    raw_response=OPTION_LETTERS[pred_idx],
                    prompt_tokens=len(self.tokenizer.encode(prompt)),
                    completion_tokens=1,
                    probability_source="fake_logits",
                    option_mass=float(self.rng.uniform(0.55, 0.95)),
                    chat_template_used=self.use_chat_template,
                )

        with tempfile.TemporaryDirectory() as td:
            fake_be = FakeDiagBackend()
            original_build = globals().get("_build_backend")
            orig_bn_file = globals().get("bn_subset_file")
            orig_en_file = globals().get("en_subset_file")
            orig_output_dir = CONFIG.output_dir

            try:
                globals()["_build_backend"] = lambda mk, cfg: fake_be
                # Set temporary output_dir on CONFIG
                CONFIG.output_dir = td
                cfg = dataclasses.replace(CONFIG, output_dir=td)

                # Write tiny subsets and dev pools
                paths = {}
                for lg in ("bn", "en"):
                    rows = [dict(it, options=json.dumps(it["options"], ensure_ascii=False))
                            for it in create_synthetic_benchmark_dataset() if it["language"] == lg]
                    paths[lg] = os.path.join(td, f"{lg}_subset.csv")
                    pd.DataFrame(rows).to_csv(paths[lg], index=False, encoding="utf-8")
                    dev_p = os.path.join(td, f"dev_{lg}.csv")
                    pd.DataFrame(rows).to_csv(dev_p, index=False, encoding="utf-8")
                globals()["bn_subset_file"] = paths["bn"]
                globals()["en_subset_file"] = paths["en"]

                # Tiny synthetic exemplar file (8 balanced exemplars per language)
                ex_path = os.path.join(td, "fewshot_exemplars.json")
                synth_ex = {
                    "bn": [{"question": f"Q bn {i}", "options": ["A", "B", "C", "D"], "correct_answer": OPTION_LETTERS[i % 4]} for i in range(8)],
                    "en": [{"question": f"Q en {i}", "options": ["A", "B", "C", "D"], "correct_answer": OPTION_LETTERS[i % 4]} for i in range(8)],
                    "version": "test_v1",
                    "n_shots_max": 8,
                }
                with open(ex_path, "w", encoding="utf-8") as f:
                    json.dump(synth_ex, f)

                # Run diagnose_prompt_variants without explicit subset_paths or exemplar_file
                df = diagnose_prompt_variants("llama32_3b", n=5, languages=("bn", "en"), config=cfg)
                assert set(df["prompt_variant"]) == {"plain", "answer_cue", "few_shot"}, f"Got {set(df['prompt_variant'])}"
                assert set(df["debias"]) == {"none", "cyclic4"}, f"Got {set(df['debias'])}"
                assert set(df["language"]) == {"bn", "en"}
                assert isinstance(df, pd.DataFrame) and len(df) > 0, "Should produce non-empty DataFrame"
                expected_cols = [
                    "language", "prompt_variant", "chat_template", "debias",
                    "mean_mass", "min_mass", "top_letter_share", "raw_top_letter_share",
                    "accuracy_pct", "n_nan", "mean_prompt_tokens", "n_perm_exempt"
                ]
                for col in expected_cols:
                    assert col in df.columns, f"Missing expected column {col}"
                csv_path = os.path.join(td, "prompt_diagnostics_llama32_3b.csv")
                assert os.path.exists(csv_path), "Should save diagnostic CSV"
            finally:
                if original_build is not None:
                    globals()["_build_backend"] = original_build
                globals()["bn_subset_file"] = orig_bn_file
                globals()["en_subset_file"] = orig_en_file
                CONFIG.output_dir = orig_output_dir
    test_registry.append(("diagnose_prompt_variants runs end-to-end with fake backend", test_diagnose_prompt_variants_fake))

    def test_diagnose_recommendation_never_uses_accuracy():
        # Synthetic DataFrame simulating diagnostic results
        # Row 1: high accuracy (95%), but fails collapse gate (top_letter_share=0.85)
        # Row 2: moderate accuracy (90%), passes both gates, but debias=cyclic4
        # Row 3: lower accuracy (55%), passes both gates, debias=none, answer_cue
        # Assert recommendation selects Row 3 (debias 'none' over 'cyclic4', ignores accuracy)
        data = [
            # Candidate 1: High accuracy, FAILS collapse gate
            {"language": "bn", "prompt_variant": "plain", "chat_template": False, "debias": "none",
             "mean_mass": 0.85, "min_mass": 0.70, "top_letter_share": 0.85, "raw_top_letter_share": 0.85, "accuracy_pct": 95.0, "n_nan": 0, "mean_prompt_tokens": 50.0, "n_perm_exempt": 0},
            {"language": "en", "prompt_variant": "plain", "chat_template": False, "debias": "none",
             "mean_mass": 0.85, "min_mass": 0.70, "top_letter_share": 0.85, "raw_top_letter_share": 0.85, "accuracy_pct": 95.0, "n_nan": 0, "mean_prompt_tokens": 50.0, "n_perm_exempt": 0},

            # Candidate 2: Passes gates, debias=cyclic4, accuracy=90%
            {"language": "bn", "prompt_variant": "plain", "chat_template": False, "debias": "cyclic4",
             "mean_mass": 0.80, "min_mass": 0.65, "top_letter_share": 0.30, "raw_top_letter_share": 0.85, "accuracy_pct": 90.0, "n_nan": 0, "mean_prompt_tokens": 50.0, "n_perm_exempt": 0},
            {"language": "en", "prompt_variant": "plain", "chat_template": False, "debias": "cyclic4",
             "mean_mass": 0.80, "min_mass": 0.65, "top_letter_share": 0.30, "raw_top_letter_share": 0.85, "accuracy_pct": 90.0, "n_nan": 0, "mean_prompt_tokens": 50.0, "n_perm_exempt": 0},

            # Candidate 3: Passes gates, debias=none, answer_cue, accuracy=55% (lower than others!)
            {"language": "bn", "prompt_variant": "answer_cue", "chat_template": False, "debias": "none",
             "mean_mass": 0.75, "min_mass": 0.60, "top_letter_share": 0.35, "raw_top_letter_share": 0.35, "accuracy_pct": 55.0, "n_nan": 0, "mean_prompt_tokens": 55.0, "n_perm_exempt": 0},
            {"language": "en", "prompt_variant": "answer_cue", "chat_template": False, "debias": "none",
             "mean_mass": 0.75, "min_mass": 0.60, "top_letter_share": 0.35, "raw_top_letter_share": 0.35, "accuracy_pct": 55.0, "n_nan": 0, "mean_prompt_tokens": 55.0, "n_perm_exempt": 0},
        ]
        df = pd.DataFrame(data)

        # Call select_recommendation on synthetic table
        res = select_recommendation(df)
        assert len(res["passing"]) == 2, f"Expected 2 passing candidates, got {len(res['passing'])}"
        assert res["prompt_variant"] == "answer_cue"
        assert res["chat_template"] is False or not res["chat_template"]
        assert res["chat_template"] == False
        assert res["debias"] == "none"

        # Table 2: chat_template built with numpy bools (np.False_)
        df_np = df.copy()
        df_np["chat_template"] = np.array([np.bool_(False)] * len(df_np))
        res_np = select_recommendation(df_np)
        assert res_np["prompt_variant"] == "answer_cue"
        assert res_np["chat_template"] == False
        assert type(res_np["chat_template"]) is bool
        assert res_np["debias"] == "none"

        # Table 3: No passing candidates (both fail collapse gate)
        df_fail = df.copy()
        df_fail["top_letter_share"] = 0.99
        res_fail = select_recommendation(df_fail)
        assert len(res_fail["passing"]) == 0
        assert res_fail["best_candidate"] is not None
        # Should pick the candidate with highest minimum mean_mass (Candidate 1 with mean_mass=0.85)
        assert res_fail["prompt_variant"] == "plain"
        assert res_fail["chat_template"] == False
        assert type(res_fail["chat_template"]) is bool
        assert res_fail["min_mean_mass"] == 0.85
    test_registry.append(("Recommendation selects by validity gates and preference order, never accuracy", test_diagnose_recommendation_never_uses_accuracy))

    def test_diagnose_numpy_bool_print_registry():
        # Verifies that np.bool_ in dataframe flows through select_recommendation and prints plain True/False
        data = [
            {"language": "bn", "prompt_variant": "plain", "chat_template": np.bool_(True), "debias": "none",
             "mean_mass": 0.80, "min_mass": 0.65, "top_letter_share": 0.30, "raw_top_letter_share": 0.30, "accuracy_pct": 50.0, "n_nan": 0, "mean_prompt_tokens": 50.0, "n_perm_exempt": 0},
            {"language": "en", "prompt_variant": "plain", "chat_template": np.bool_(True), "debias": "none",
             "mean_mass": 0.80, "min_mass": 0.65, "top_letter_share": 0.30, "raw_top_letter_share": 0.30, "accuracy_pct": 50.0, "n_nan": 0, "mean_prompt_tokens": 50.0, "n_perm_exempt": 0},
        ]
        df = pd.DataFrame(data)
        res = select_recommendation(df)
        assert type(res["chat_template"]) is bool
        line = f'MODEL_REGISTRY["test_model"]["use_chat_template"] = {res["chat_template"]}'
        assert line == 'MODEL_REGISTRY["test_model"]["use_chat_template"] = True'
    test_registry.append(("diagnose_prompt_variants formats plain Python bools for MODEL_REGISTRY", test_diagnose_numpy_bool_print_registry))

    def test_diagnose_stable_seeds():
        LANG_STABLE_SEEDS = {"bn": 10007, "en": 20011}
        items = list(range(100))
        rng_bn1 = np.random.RandomState(LANG_STABLE_SEEDS["bn"] + 42)
        perm_bn1 = rng_bn1.permutation(len(items))[:30].tolist()
        rng_bn2 = np.random.RandomState(LANG_STABLE_SEEDS["bn"] + 42)
        perm_bn2 = rng_bn2.permutation(len(items))[:30].tolist()
        rng_en = np.random.RandomState(LANG_STABLE_SEEDS["en"] + 42)
        perm_en = rng_en.permutation(len(items))[:30].tolist()

        assert perm_bn1 == perm_bn2, "Deterministic across invocations"
        assert perm_bn1 != perm_en, "Distinct between languages"
    test_registry.append(("diagnose_prompt_variants uses stable per-language permutation seeds", test_diagnose_stable_seeds))

    def test_log_top_next_tokens_no_model():
        class NoModelBackend:
            model_id = "no-model"
            tokenizer = None
        be = NoModelBackend()
        # Should silently do nothing (no exception)
        result = _log_top_next_tokens(be, ["test prompt"], k=8)
        assert result == "", "Should return empty string for backend without .model"
    test_registry.append(("_log_top_next_tokens is no-op on backend without .model", test_log_top_next_tokens_no_model))

    def test_evaluator_flush_non_multiple_of_checkpoint():
        # Regression test: 7 items (< CHECKPOINT_FREQ) must all be written, and resume must skip them
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            be = MockInferenceBackend(seed=1)
            ev = BenchmarkEvaluator(be, model_key="unit_test_model", mode="extended", output_dir=td,
                                    language="bn", benchmark_name="BnMMLU")
            items = [dict(it) for it in create_synthetic_benchmark_dataset() if it["language"] == "bn"][:7]
            ev.evaluate_all(items, resume=False, show_progress=False)
            out = pd.read_csv(ev.results_csv)
            assert len(out) == 7 and out["question_id"].notna().all(), f"expected 7 rows, got {len(out)}"
            ev2 = BenchmarkEvaluator(be, model_key="unit_test_model", mode="extended", output_dir=td,
                                     language="bn", benchmark_name="BnMMLU")
            ev2.evaluate_all(items, resume=True, show_progress=False)
            assert len(pd.read_csv(ev2.results_csv)) == 7, "resume must not duplicate rows"
    test_registry.append(("Evaluator flushes partial buffer and resumes cleanly", test_evaluator_flush_non_multiple_of_checkpoint))

    def test_synthetic_fallback_respects_language():
        # The synthetic fixtures are only reachable with STRICT_REAL_MODE off; toggle it for this test only and restore it
        global STRICT_REAL_MODE
        _prev_strict = STRICT_REAL_MODE
        STRICT_REAL_MODE = False
        try:
            bn = load_dataset_from_file("FALLBACK_SYNTHETIC", language="bn", is_mock=True)
            en = load_dataset_from_file("FALLBACK_SYNTHETIC", language="en", is_mock=True)
        finally:
            STRICT_REAL_MODE = _prev_strict
        assert STRICT_REAL_MODE is _prev_strict
        assert bn and all(it["language"] == "bn" for it in bn)
        assert en and all(it["language"] == "en" for it in en)
    test_registry.append(("Synthetic mock data respects requested language", test_synthetic_fallback_respects_language))

    def test_duplicate_question_ids_made_unique():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            p = os.path.join(td, "dup.csv")
            pd.DataFrame({
                "question_id": ["X", "X"], "subject": ["s", "s"], "question": ["q1", "q2"],
                "options": [json.dumps(["a", "b", "c", "d"]), json.dumps(["a", "b", "c", "d"])],
                "correct_answer": ["A", "B"],
            }).to_csv(p, index=False)
            items = load_dataset_from_file(p, language="bn", is_mock=False)
            assert len({it["question_id"] for it in items}) == 2
    test_registry.append(("Duplicate question_ids are made unique", test_duplicate_question_ids_made_unique))

    def test_markup_detection():
        assert contains_markup("x <mfrac><mn>1</mn><mn>2</mn></mfrac>", ["a", "b", "c", "d"])
        assert contains_markup("q", ["<mi>x</mi>", "b", "c", "d"])
        assert not contains_markup("যদি a<b এবং c>d হয় তবে", ["1", "2", "3", "4"]), "plain inequalities must not be flagged"
    test_registry.append(("MathML markup detection (no false positive on a<b)", test_markup_detection))

    def test_hub_lowercase_labels():
        # The Hub version of BnMMLU stores correct_answer as lowercase a/b/c/d
        assert parse_answer_label("a") == ("A", 0)
        assert parse_answer_label("d") == ("D", 3)
    test_registry.append(("Lowercase a-d answer labels (Hub schema)", test_hub_lowercase_labels))

    def test_subject_map_valid():
        mmlu_57 = set("""abstract_algebra anatomy astronomy business_ethics clinical_knowledge college_biology college_chemistry
        college_computer_science college_mathematics college_medicine college_physics computer_security conceptual_physics
        econometrics electrical_engineering elementary_mathematics formal_logic global_facts high_school_biology
        high_school_chemistry high_school_computer_science high_school_european_history high_school_geography
        high_school_government_and_politics high_school_macroeconomics high_school_mathematics high_school_microeconomics
        high_school_physics high_school_psychology high_school_statistics high_school_us_history high_school_world_history
        human_aging human_sexuality international_law jurisprudence logical_fallacies machine_learning management marketing
        medical_genetics miscellaneous moral_disputes moral_scenarios nutrition philosophy prehistory professional_accounting
        professional_law professional_medicine professional_psychology public_relations security_studies sociology
        us_foreign_policy virology world_religions""".split())
        assert len(mmlu_57) == 57
        valid_norm = {normalize_subject(x) for x in mmlu_57}
        assert normalize_subject("Advanced_Mathematics") == "advanced mathematics"
        full_map = build_subject_map()
        for bn_s, targets in full_map.items():
            for t in targets:
                assert t in valid_norm, f"map entry '{bn_s}' -> '{t}' is not an MMLU subject"
        # Validate that every key in SUBJECT_MAP_EXTRA is present (normalised) and its values are valid MMLU subjects
        for k_extra, v_extra in SUBJECT_MAP_EXTRA.items():
            k_norm = normalize_subject(k_extra)
            assert k_norm in full_map, f"SUBJECT_MAP_EXTRA key '{k_norm}' missing from build_subject_map()"
            assert len(full_map[k_norm]) > 0, f"SUBJECT_MAP_EXTRA key '{k_norm}' has no targets"
            for t in v_extra:
                assert normalize_subject(t) in valid_norm, f"SUBJECT_MAP_EXTRA target '{t}' not in MMLU 57"
    test_registry.append(("BnMMLU->MMLU subject map and SUBJECT_MAP_EXTRA only reference real MMLU subjects", test_subject_map_valid))

    def test_prepare_subsets_hub_schema_stub():
        # Exercises the REAL-mode code path against stubs that mimic the Hub schemas (no network needed)
        import tempfile, types, dataclasses

        class _DS:
            def __init__(self, df): self._df = df
            def to_pandas(self): return self._df.copy()

        def _bn_df():
            rows = []
            subs = {"Advanced_Physics": 60, "Biology": 60, "Bengali": 60}
            sid = 0
            for s, n in subs.items():
                for k in range(n):
                    q = f"প্রশ্ন {s} {k}"
                    if s == "Advanced_Physics" and k % 5 == 0:
                        q += " <mfrac><mn>1</mn><mn>2</mn></mfrac>"      # MathML item that must be filtered
                    rows.append({"Unique_Serial": sid, "subject_name": s, "question": q, "correct_answer": "abcd"[k % 4],
                                 "options": np.array(["ক", "খ", "গ", "ঘ"], dtype=object), "question_char_count": len(q)})
                    sid += 1
            return pd.DataFrame(rows)

        mmlu_test_rows = []
        for s in ["high_school_physics", "college_physics", "high_school_biology", "college_biology", "philosophy"]:
            for k in range(50):
                mmlu_test_rows.append({"question": f"Q {s} {k}", "subject": s, "choices": ["w", "x", "y", "z"], "answer": k % 4})

        mmlu_dev_rows = []
        for s in ["high_school_physics", "college_physics", "high_school_biology", "college_biology"]:
            for k in range(10):
                mmlu_dev_rows.append({"question": f"Dev Q {s} {k}", "subject": s, "choices": ["a", "b", "c", "d"], "answer": k % 4})

        dev_should_raise = False

        def _fake_load_dataset(repo, cfg=None, split=None, token=None, **kw):
            if repo == "samanjoy2/BnMMLU":
                assert cfg == "full", "BnMMLU needs an explicit config"
                assert token == "tok", "gated dataset must receive the token"
                return {"train": _DS(_bn_df())}
            if repo == "cais/mmlu":
                if split == "dev":
                    if dev_should_raise:
                        raise RuntimeError("Simulated Hub dev split 404 / network error")
                    return mmlu_dev_rows
                return mmlu_test_rows
            raise AssertionError(repo)

        fake = types.ModuleType("datasets"); fake.load_dataset = _fake_load_dataset
        old_mod = sys.modules.get("datasets"); old_rev = globals().get("_hub_revision")
        orig_strict = globals().get("STRICT_REAL_MODE")
        sys.modules["datasets"] = fake; globals()["_hub_revision"] = lambda *a, **k: "test-sha"
        try:
            with tempfile.TemporaryDirectory() as td:
                cfg = dataclasses.replace(CONFIG, backend="transformers", hf_token="tok", bn_local_path=None,
                                          dev_n=10,
                                          exclude_markup=True, subject_match_mode="intersection",
                                          allow_fewshot_test_pool_fallback=False)
                bn_p, en_p = prepare_research_subsets(output_dir=td, n_items=60, seed=1, hf_token="tok", is_mock=False, config=cfg)
                bn = pd.read_csv(bn_p); en = pd.read_csv(en_p)
                assert len(bn) == len(en) > 0
                assert not bn["question"].str.contains("<mfrac>").any(), "MathML items must be removed"
                assert "bengali" not in set(bn["subject_group"]), "unmapped subjects are excluded in intersection mode"
                assert set(bn["subject_group"]) == set(en["subject_group"])
                assert (bn.groupby("subject_group").size() == en.groupby("subject_group").size()).all()
                assert set(bn["correct_answer"]) <= set("ABCD")

                # Verify fewshot_exemplars.json used mmlu_dev
                ex_file = os.path.join(td, "fewshot_exemplars.json")
                assert os.path.exists(ex_file), "fewshot_exemplars.json must be persisted"
                with open(ex_file, "r", encoding="utf-8") as f:
                    ex_data = json.load(f)
                assert ex_data.get("english_exemplar_source") == "mmlu_dev"
                assert len(ex_data["en"]) > 0
                assert all(str(item["question_id"]).startswith("en_dev_") for item in ex_data["en"]), "English exemplars must have en_dev_ prefix"

            # Test dev split failure with STRICT_REAL_MODE=True and allow_fallback=False -> must raise RuntimeError
            dev_should_raise = True
            with tempfile.TemporaryDirectory() as td2:
                cfg_strict = dataclasses.replace(CONFIG, backend="transformers", hf_token="tok", bn_local_path=None,
                                                 exclude_markup=True, subject_match_mode="intersection",
                                                 allow_fewshot_test_pool_fallback=False)
                raised = False
                try:
                    prepare_research_subsets(output_dir=td2, n_items=60, seed=1, hf_token="tok", is_mock=False, config=cfg_strict)
                except RuntimeError as re_err:
                    raised = True
                    assert "Could not load English MMLU dev split under STRICT_REAL_MODE" in str(re_err)
                assert raised, "prepare_research_subsets must raise RuntimeError when dev split fails and fallback is disabled"

            # Test dev split failure with allow_fallback=True -> must fall back to test_pool_fallback
            with tempfile.TemporaryDirectory() as td3:
                cfg_fallback = dataclasses.replace(CONFIG, backend="transformers", hf_token="tok", bn_local_path=None,
                                                   dev_n=10,
                                                   exclude_markup=True, subject_match_mode="intersection",
                                                   allow_fewshot_test_pool_fallback=True)
                prepare_research_subsets(output_dir=td3, n_items=60, seed=1, hf_token="tok", is_mock=False, config=cfg_fallback)
                ex_file3 = os.path.join(td3, "fewshot_exemplars.json")
                with open(ex_file3, "r", encoding="utf-8") as f:
                    ex_data3 = json.load(f)
                assert ex_data3.get("english_exemplar_source") == "test_pool_fallback"
        finally:
            dev_should_raise = False
            if old_mod is None: sys.modules.pop("datasets", None)
            else: sys.modules["datasets"] = old_mod
            if old_rev is not None: globals()["_hub_revision"] = old_rev
            if orig_strict is not None: globals()["STRICT_REAL_MODE"] = orig_strict
    test_registry.append(("Real-mode loader on Hub-schema stubs (gated, config, MathML, subject match)", test_prepare_subsets_hub_schema_stub))

    def test_subject_group_pairing():
        df = pd.DataFrame({
            "is_correct": [1, 0, 1, 0], "confidence": [0.9, 0.6, 0.8, 0.7], "overall_tpw": [2.0, 3.0, 1.2, 1.5],
            "language": ["bn", "bn", "en", "en"], "model_key": ["m"] * 4,
            "subject": ["Advanced_Physics", "Advanced_Physics", "college_physics", "high_school_physics"],
            "subject_group": ["advanced physics"] * 4,
        })
        out = prepare_analysis_dataframe(df)
        assert out["subject_norm"].nunique() == 1, "paired bootstrap needs both languages on the same subject key"
    test_registry.append(("Analysis uses subject_group so bn/en subjects pair up", test_subject_group_pairing))

    def test_paired_bootstrap_runs():
        rng = np.random.RandomState(0)
        def _mk(lang, n=200):
            return pd.DataFrame({"confidence": rng.uniform(0.3, 1.0, n), "is_correct": rng.randint(0, 2, n),
                                 "subject_norm": rng.choice(["a", "b", "c", "d"], n), "language": lang})
        res = compute_paired_bootstrap_delta_ece(_mk("bn"), _mk("en"), n_boot=50, seed=1)
        assert "mean_delta_ece" in res and res["ci_95"][0] <= res["ci_95"][1]
    test_registry.append(("Paired subject-cluster bootstrap runs", test_paired_bootstrap_runs))

    def test_baseline_mode_keeps_plain_prompt():
        import tempfile
        seen = []

        class CapturingBackend(MockInferenceBackend):
            def generate_text(self, prompt, max_tokens=64):
                seen.append(prompt)
                return super().generate_text(prompt, max_tokens)

        with tempfile.TemporaryDirectory() as td:
            ev = BenchmarkEvaluator(CapturingBackend(seed=1), model_key="base_test", mode="baseline", output_dir=td,
                                    language="bn", benchmark_name="BnMMLU", prompt_variant="answer_cue")
            item = [dict(it) for it in create_synthetic_benchmark_dataset() if it["language"] == "bn"][0]
            ev.evaluate_item(item)
            assert seen and seen[0].endswith(":\n"), "baseline mode must keep the plain prompt"
            try:
                BenchmarkEvaluator(MockInferenceBackend(seed=1), model_key="bad_variant", mode="extended", output_dir=td,
                                   language="bn", prompt_variant="nonsense")
                assert False, "unknown prompt_variant must raise ValueError"
            except ValueError:
                pass
    test_registry.append(("Baseline mode keeps plain prompt; evaluator validates variant", test_baseline_mode_keeps_plain_prompt))


    def test_resolve_run_settings_precedence():
        # Registry key overrides CONFIG
        import dataclasses
        cfg_plain = dataclasses.replace(CONFIG, prompt_variant="plain", backend="mock")
        s = resolve_run_settings("titullm_1b", cfg_plain)
        assert s["prompt_variant"] == "answer_cue", f"Expected answer_cue from registry, got {s['prompt_variant']}"
        assert s["debias"] == "none"
        assert s["n_shots"] in (0, 4, 8)
        assert isinstance(s["collapse_threshold"], float)
        assert "collapse_gate_overridden" in s
    test_registry.append(("resolve_run_settings: registry overrides CONFIG", test_resolve_run_settings_precedence))

    def test_resolve_run_settings_bad_key():
        try:
            resolve_run_settings("nonexistent_model", CONFIG)
            assert False, "Should raise ValueError"
        except ValueError:
            pass
    test_registry.append(("resolve_run_settings: unknown model_key raises ValueError", test_resolve_run_settings_bad_key))

    def test_resolve_run_settings_bad_debias():
        old_info = MODEL_REGISTRY["titullm_1b"].copy()
        MODEL_REGISTRY["titullm_1b"]["debias"] = "bad_value"
        try:
            resolve_run_settings("titullm_1b", CONFIG)
            assert False, "Should raise ValueError for bad debias"
        except ValueError:
            pass
        finally:
            MODEL_REGISTRY["titullm_1b"] = old_info
    test_registry.append(("resolve_run_settings: invalid debias raises ValueError", test_resolve_run_settings_bad_debias))

    # --- Task 5: Cyclic4 Debiasing & Exemption Tests (v3b) ---
    def test_cyclic4_resolve_run_settings():
        old_info = MODEL_REGISTRY["titullm_1b"].copy()
        MODEL_REGISTRY["titullm_1b"]["debias"] = "cyclic4"
        try:
            s = resolve_run_settings("titullm_1b", CONFIG)
            assert s["debias"] == "cyclic4"
        finally:
            MODEL_REGISTRY["titullm_1b"] = old_info
    test_registry.append(("resolve_run_settings supports debias=cyclic4", test_cyclic4_resolve_run_settings))

    class MockKnowledgeBackend(BaseInferenceBackend):
        """Mock backend that inspects prompt options and puts 0.9 mass on whichever letter holds the gold text."""
        def __init__(self, gold_text: str):
            super().__init__(model_id="mock_knowledge")
            self.gold_text = gold_text
            self.call_count = 0

        def predict_option_probabilities(self, prompt: str) -> InferenceResult:
            self.call_count += 1
            # Find which letter holds gold_text
            probs = [0.0333, 0.0333, 0.0333, 0.0333]
            for idx, letter in enumerate(OPTION_LETTERS):
                # Pattern: "A) <text>" or "A. <text>"
                if f"{letter}) {self.gold_text}" in prompt or f"{letter}. {self.gold_text}" in prompt or f"{letter}: {self.gold_text}" in prompt:
                    probs = [0.0333, 0.0333, 0.0333, 0.0333]
                    probs[idx] = 0.9001
                    break
            s = sum(probs)
            probs = [p / s for p in probs]
            best_idx = int(np.argmax(probs))
            return InferenceResult(
                predicted_letter=OPTION_LETTERS[best_idx],
                confidence=float(probs[best_idx]),
                probabilities=probs,
                option_mass=0.95,
                prompt_tokens=40,
                probability_source="option_logits_softmax",
                raw_response=None,
                chat_template_used=False,
            )

        def generate_text(self, prompt: str) -> InferenceResult:
            raise NotImplementedError()

    def test_cyclic4_knowledge_backend():
        gold_text = "The capital of France is Paris"
        item = {
            "question_id": "test_k_1",
            "question": "What is the capital of France?",
            "options": ["London", "Berlin", gold_text, "Madrid"],  # Gold is C
            "correct_answer": "C",
            "language": "en",
        }
        backend = MockKnowledgeBackend(gold_text=gold_text)
        settings = {
            "prompt_variant": "plain",
            "n_shots": 4,
            "use_chat_template": False,
            "debias": "cyclic4",
            "collapse_threshold": 0.8,
            "collapse_gate_overridden": False,
        }
        res = predict_item(backend, item, language="en", settings=settings)
        assert backend.call_count == 4, f"Expected 4 passes, got {backend.call_count}"
        assert res.perm_applied is True
        assert res.predicted_letter == "C", f"Expected predicted_letter C, got {res.predicted_letter}"
        assert res.probability_source == "option_logits_cyclic4"
    test_registry.append(("cyclic4 knowledge backend recovers original gold letter", test_cyclic4_knowledge_backend))

    def test_cyclic4_cross_referencing_exempt():
        gold_text = "All of the above"
        item_en = {
            "question_id": "test_exempt_en",
            "question": "Which of these are mammals?",
            "options": ["Whale", "Dolphin", "Bat", "All of the above"],
            "correct_answer": "D",
            "language": "en",
        }
        backend = MockKnowledgeBackend(gold_text="Whale")
        settings = {
            "prompt_variant": "plain",
            "n_shots": 4,
            "use_chat_template": False,
            "debias": "cyclic4",
            "collapse_threshold": 0.8,
            "collapse_gate_overridden": False,
        }
        res_en = predict_item(backend, item_en, language="en", settings=settings)
        assert backend.call_count == 1, f"Expected 1 pass for exempt item, got {backend.call_count}"
        assert res_en.perm_applied is False
        assert res_en.pred_raw == res_en.predicted_letter

        # Bangla cross-referencing item
        item_bn = {
            "question_id": "test_exempt_bn",
            "question": "কোনটি সঠিক?",
            "options": ["ক", "খ", "উপরের সবগুলো", "কোনটিই নয়"],
            "correct_answer": "C",
            "language": "bn",
        }
        backend_bn = MockKnowledgeBackend(gold_text="ক")
        res_bn = predict_item(backend_bn, item_bn, language="bn", settings=settings)
        assert backend_bn.call_count == 1
        assert res_bn.perm_applied is False
    test_registry.append(("cross-referencing items are exempt from cyclic4 permutation", test_cyclic4_cross_referencing_exempt))

    class MockAlwaysABackend(BaseInferenceBackend):
        """Mock backend that always predicts letter A with 0.99 probability regardless of prompt."""
        def __init__(self):
            super().__init__(model_id="mock_always_a")

        def predict_option_probabilities(self, prompt: str) -> InferenceResult:
            probs = [0.99, 0.0033, 0.0033, 0.0034]
            return InferenceResult(
                predicted_letter="A",
                confidence=0.99,
                probabilities=probs,
                option_mass=0.98,
                prompt_tokens=40,
                probability_source="option_logits_softmax",
                raw_response=None,
                chat_template_used=False,
            )

        def generate_text(self, prompt: str) -> InferenceResult:
            raise NotImplementedError()

    def test_cyclic4_always_a_debiasing():
        backend = MockAlwaysABackend()
        settings = {
            "prompt_variant": "plain",
            "n_shots": 4,
            "use_chat_template": False,
            "debias": "cyclic4",
            "collapse_threshold": 0.8,
            "collapse_gate_overridden": False,
        }
        item = {
            "question_id": "test_a_1",
            "question": "Test question?",
            "options": ["Option 1", "Option 2", "Option 3", "Option 4"],
            "correct_answer": "B",
            "language": "en",
        }
        res = predict_item(backend, item, language="en", settings=settings)
        assert res.pred_raw == "A"
        assert res.confidence_raw == 0.99
        assert res.perm_applied is True
        # Across the 4 passes, letter A was placed at each original option once
        # So each original option gets 0.99 / 4 + 0.0033 * 3 / 4 ~ 0.25 (uniform)
        for p in res.probabilities:
            assert abs(p - 0.25) < 0.02, f"Expected uniform probability near 0.25, got {p}"
    test_registry.append(("cyclic4 turns always-A model predictions near-uniform", test_cyclic4_always_a_debiasing))

    def test_predict_item_basic():
        class _FakeInfRes:
            predicted_letter = "A"
            confidence = 0.7
            probabilities = [0.7, 0.1, 0.1, 0.1]
            option_mass = 1.0
            prompt_tokens = 5
            probability_source = "softmax"
            raw_response = "prompt"

        class FakeBackend:
            model_id = "fake"
            use_chat_template = False
            class _Tok:
                name_or_path = "fake"
                def encode(self, t, add_special_tokens=False): return [1, 2, 3]
                def tokenize(self, t): return ["a", "b"]
            tokenizer = _Tok()
            def predict_option_probabilities(self, prompt):
                return _FakeInfRes()

        item = {"question": "Test?", "options": ["Op A", "Op B", "Op C", "Op D"],
                "correct_answer": "A", "language": "en", "question_id": "t1"}
        settings = {"prompt_variant": "plain", "n_shots": 0, "use_chat_template": False,
                    "debias": "none", "collapse_threshold": 0.8, "collapse_gate_overridden": False}
        pr = predict_item(FakeBackend(), item, "en", settings)
        assert pr.predicted_letter == "A"
        assert pr.pred_raw == "A"
        assert pr.perm_applied is False
        assert pr.confidence_raw == 0.7
    test_registry.append(("predict_item returns pred_raw==predicted for debias=none", test_predict_item_basic))

    def test_resume_guard_inference_signature():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            be = MockInferenceBackend(seed=1)
            plain_settings = {"prompt_variant": "plain", "n_shots": 0, "use_chat_template": False,
                              "debias": "none", "collapse_threshold": 0.8, "collapse_gate_overridden": False}
            ev = BenchmarkEvaluator(be, model_key="sig_test", mode="extended", output_dir=td,
                                    language="bn", benchmark_name="BnMMLU",
                                    prompt_variant="plain", settings=plain_settings)
            items = [dict(it) for it in create_synthetic_benchmark_dataset() if it["language"] == "bn"][:3]
            ev.evaluate_all(items, resume=False, show_progress=False)

            # 1. Same prompt_variant but different chat setting → different signature
            diff_sig_settings = {"prompt_variant": "plain", "n_shots": 0, "use_chat_template": True,
                                  "debias": "none", "collapse_threshold": 0.8, "collapse_gate_overridden": False}
            try:
                BenchmarkEvaluator(be, model_key="sig_test", mode="extended", output_dir=td,
                                   language="bn", benchmark_name="BnMMLU",
                                   prompt_variant="plain", settings=diff_sig_settings)
                assert False, "Should raise RuntimeError for signature mismatch"
            except RuntimeError as e:
                assert "signature" in str(e).lower() or "mismatch" in str(e).lower(), str(e)

            # 2. Same prompt_variant but different debias setting → different signature
            diff_debias_settings = {"prompt_variant": "plain", "n_shots": 0, "use_chat_template": False,
                                    "debias": "cyclic4", "collapse_threshold": 0.8, "collapse_gate_overridden": False}
            try:
                BenchmarkEvaluator(be, model_key="sig_test", mode="extended", output_dir=td,
                                   language="bn", benchmark_name="BnMMLU",
                                   prompt_variant="plain", settings=diff_debias_settings)
                assert False, "Should raise RuntimeError for signature mismatch on debias"
            except RuntimeError as e:
                assert "signature" in str(e).lower() or "mismatch" in str(e).lower(), str(e)
    test_registry.append(("Resume guard raises on inference_signature mismatch", test_resume_guard_inference_signature))

    # --- Task 5: Few-Shot Selection & Prompt Formatting Tests (v3c) ---
    def test_fewshot_selection_balance_and_prefix():
        candidates = []
        subjects = ["physics", "chemistry", "history", "math"]
        for i in range(40):
            letter = OPTION_LETTERS[i % 4]
            candidates.append({
                "question_id": f"cand_{i}",
                "question": f"Question {i}?",
                "options": ["A", "B", "C", "D"],
                "correct_answer": letter,
                "subject": subjects[i % len(subjects)],
                "has_markup": False,
            })
        eval_ids = {f"cand_{i}" for i in range(10)}  # first 10 in eval_ids
        ex = select_fewshot_exemplars(candidates, eval_ids, language="en", seed=42)
        assert len(ex) == 8, f"Expected 8 exemplars, got {len(ex)}"
        # Check overall balance (each A-D exactly twice)
        letters_all = [it["correct_answer"] for it in ex]
        for l in OPTION_LETTERS:
            assert letters_all.count(l) == 2, f"Letter {l} count in 8: {letters_all.count(l)}"
        # Check prefix of 4 contains each letter exactly once
        prefix_4 = [it["correct_answer"] for it in ex[:4]]
        assert set(prefix_4) == set(OPTION_LETTERS), f"Prefix of 4 not balanced: {prefix_4}"

        # Overlap guard
        ex_ids = {it["question_id"] for it in ex}
        assert not ex_ids.intersection(eval_ids), "Exemplars overlap with eval_ids!"

        # Determinism
        ex2 = select_fewshot_exemplars(candidates, eval_ids, language="en", seed=42)
        assert [it["question_id"] for it in ex] == [it["question_id"] for it in ex2], "Selection not deterministic!"

    test_registry.append(("select_fewshot_exemplars balance, prefix, determinism", test_fewshot_selection_balance_and_prefix))

    def test_fewshot_overlap_raises():
        candidates = []
        for i in range(8):
            candidates.append({
                "question_id": f"q_{i}",
                "question": "Q?",
                "options": ["1", "2", "3", "4"],
                "correct_answer": OPTION_LETTERS[i % 4],
                "subject": "math",
                "has_markup": False,
            })
        # If all candidates are in eval_ids, must raise RuntimeError
        eval_ids = {f"q_{i}" for i in range(8)}
        try:
            select_fewshot_exemplars(candidates, eval_ids, language="en", seed=42)
            assert False, "Should raise RuntimeError when balanced set cannot be built due to eval_ids"
        except RuntimeError:
            pass
    test_registry.append(("select_fewshot_exemplars raises on overlap/insufficient", test_fewshot_overlap_raises))

    def test_fewshot_prompt_formatting():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            ex_path = os.path.join(td, "fewshot_exemplars.json")
            exemplars_data = {
                "en": [
                    {"question_id": f"ex_{i}", "question": f"Ex Q {i}", "options": ["O1","O2","O3","O4"], "correct_answer": OPTION_LETTERS[i%4], "subject": "STEM"}
                    for i in range(8)
                ],
                "bn": [
                    {"question_id": f"bn_ex_{i}", "question": f"Bn Q {i}", "options": ["ব১","ব২","ব৩","ব৪"], "correct_answer": OPTION_LETTERS[i%4], "subject": "STEM"}
                    for i in range(8)
                ],
            }
            with open(ex_path, "w", encoding="utf-8") as f:
                json.dump(exemplars_data, f)

            # Test k=4
            p4 = format_prompt("Main Q?", ["A1","A2","A3","A4"], language="en", variant="few_shot", n_shots=4, exemplar_file=ex_path)
            assert p4.endswith(ANSWER_CUES["en"]), f"Prompt should end with bare cue, got ending: {p4[-20:]}"
            assert not p4.endswith(chr(10)), "Prompt should have no trailing newline"
            # Count cue occurrences: 4 worked examples with letters + 1 bare cue = 5 cues
            cue_count = p4.count(ANSWER_CUES["en"])
            assert cue_count == 5, f"Expected 5 occurrences of cue, found {cue_count}"

            # Test k=8
            p8 = format_prompt("Main Q?", ["A1","A2","A3","A4"], language="en", variant="few_shot", n_shots=8, exemplar_file=ex_path)
            assert p8.count(ANSWER_CUES["en"]) == 9, "Expected 9 occurrences of cue for k=8"

            # Invalid n_shots
            try:
                format_prompt("Main Q?", ["A1","A2","A3","A4"], language="en", variant="few_shot", n_shots=5, exemplar_file=ex_path)
                assert False, "Should raise ValueError for n_shots=5"
            except ValueError:
                pass
    test_registry.append(("format_prompt few_shot structure and validation", test_fewshot_prompt_formatting))

    def test_fewshot_missing_file_raises():
        try:
            format_prompt("Main Q?", ["A1","A2","A3","A4"], language="en", variant="few_shot", n_shots=4, exemplar_file="/nonexistent/fewshot.json")
            assert False, "Should raise RuntimeError on missing exemplar file"
        except RuntimeError as e:
            assert "Few-shot exemplar file missing" in str(e)
    test_registry.append(("format_prompt raises RuntimeError on missing exemplar file", test_fewshot_missing_file_raises))

    def test_fewshot_inference_signature_differs():
        s4 = {"prompt_variant": "few_shot", "n_shots": 4, "use_chat_template": False, "debias": "none"}
        s8 = {"prompt_variant": "few_shot", "n_shots": 8, "use_chat_template": False, "debias": "none"}
        s_plain = {"prompt_variant": "plain", "n_shots": 4, "use_chat_template": False, "debias": "none"}
        sig4 = _make_inference_signature(s4)
        sig8 = _make_inference_signature(s8)
        sig_plain = _make_inference_signature(s_plain)
        assert sig4 != sig8, f"Signatures should differ: {sig4} vs {sig8}"
        assert "shots=4" in sig4
        assert "shots=8" in sig8
        assert "shots=0" in sig_plain, f"Plain variant should have shots=0, got {sig_plain}"
    test_registry.append(("inference_signature distinguishes few_shot k=4 vs k=8", test_fewshot_inference_signature_differs))
    def test_fewshot_loader_default_dir_and_cache_invalidation():
        import tempfile
        orig_dir = CONFIG.output_dir
        with tempfile.TemporaryDirectory() as td:
            p = os.path.join(td, "fewshot_exemplars.json")
            try:
                CONFIG.output_dir = td
                with open(p, "w", encoding="utf-8") as f:
                    json.dump({"bn": [], "en": [], "version": "a"}, f)
                assert load_fewshot_exemplars()["version"] == "a", "default path must follow CONFIG.output_dir"
                with open(p, "w", encoding="utf-8") as f:
                    json.dump({"bn": [], "en": [], "version": "b"}, f)
                os.utime(p, (os.path.getmtime(p) + 5, os.path.getmtime(p) + 5))
                assert load_fewshot_exemplars()["version"] == "b", "cache must refresh when the file changes"
            finally:
                CONFIG.output_dir = orig_dir
                _FEWSHOT_EXEMPLARS_CACHE.pop(p, None)
    test_registry.append(("Few-shot loader follows CONFIG.output_dir and refreshes its cache", test_fewshot_loader_default_dir_and_cache_invalidation))

    def test_predict_item_forwards_exemplar_file():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            p = os.path.join(td, "custom_ex.json")
            ex = [{"question": f"Q{i}", "options": ["a", "b", "c", "d"], "correct_answer": OPTION_LETTERS[i % 4]} for i in range(8)]
            with open(p, "w", encoding="utf-8") as f:
                json.dump({"bn": ex, "en": ex, "version": "t"}, f)
            seen = []
            class Spy:
                model_id = "spy"
                def predict_option_probabilities(self, prompt):
                    seen.append(prompt)
                    return InferenceResult(predicted_letter="A", confidence=0.4, probabilities=[0.4, 0.2, 0.2, 0.2],
                                           raw_response="A", prompt_tokens=1, completion_tokens=1,
                                           probability_source="fake", option_mass=0.9)
            item = {"question": "Main question?", "options": ["w", "x", "y", "z"]}
            predict_item(Spy(), item, "en", {"prompt_variant": "few_shot", "n_shots": 4, "debias": "none", "exemplar_file": p})
            assert "Q0" in seen[0], "exemplars from settings['exemplar_file'] must be used"
    test_registry.append(("predict_item forwards settings['exemplar_file'] to few_shot prompts", test_predict_item_forwards_exemplar_file))

    def test_resolve_run_settings_n_shots_rules():
        # Test 1: registry n_shots=0 with prompt_variant="plain" resolves without error
        orig_plain = MODEL_REGISTRY["llama32_3b"].copy()
        try:
            MODEL_REGISTRY["llama32_3b"]["prompt_variant"] = "plain"
            MODEL_REGISTRY["llama32_3b"]["n_shots"] = 0
            s_plain = resolve_run_settings("llama32_3b", CONFIG)
            assert s_plain["prompt_variant"] == "plain"
            assert s_plain["n_shots"] == 0

            # Test 2: registry n_shots=5 with prompt_variant="few_shot" raises ValueError
            MODEL_REGISTRY["llama32_3b"]["prompt_variant"] = "few_shot"
            MODEL_REGISTRY["llama32_3b"]["n_shots"] = 5
            raised = False
            try:
                resolve_run_settings("llama32_3b", CONFIG)
            except ValueError as ve:
                raised = True
                assert "Allowed: {4, 8}" in str(ve)
            assert raised, "Expected ValueError for n_shots=5 with few_shot"

            # Test 3: registry n_shots=8 with prompt_variant="few_shot" resolves without error
            MODEL_REGISTRY["llama32_3b"]["prompt_variant"] = "few_shot"
            MODEL_REGISTRY["llama32_3b"]["n_shots"] = 8
            s_fs8 = resolve_run_settings("llama32_3b", CONFIG)
            assert s_fs8["prompt_variant"] == "few_shot"
            assert s_fs8["n_shots"] == 8
        finally:
            MODEL_REGISTRY["llama32_3b"] = orig_plain
    test_registry.append(("resolve_run_settings n_shots validation rules", test_resolve_run_settings_n_shots_rules))

    def test_analysis_valid_only_filtering():
        # Build synthetic results DataFrame with 1 VALID model-lang and 1 INVALID model-lang
        synth_data = [
            {"model_key": "m_valid", "language": "bn", "question_id": f"qv_{i}", "is_correct": i % 2 == 0, "confidence": 0.8, "tpw": 2.0, "subject_group": "biology", "subject": "biology", "total_tokens": 50, "prompt_tokens": 50, "calibration_residual": 0.2}
            for i in range(12)
        ] + [
            {"model_key": "m_invalid", "language": "bn", "question_id": f"qi_{i}", "is_correct": False, "confidence": 0.99, "tpw": 2.0, "subject_group": "physics", "subject": "physics", "total_tokens": 40, "prompt_tokens": 40, "calibration_residual": 0.99}
            for i in range(12)
        ]
        df_combined = pd.DataFrame(synth_data)
        df_combined["validity_flag"] = ["VALID"] * 12 + ["INVALID_COLLAPSE"] * 12
        df_combined["run_valid"] = df_combined["validity_flag"] == "VALID"

        # Section 18 filter
        analysis_df = df_combined[df_combined["validity_flag"] == "VALID"].copy()
        assert len(analysis_df) == 12
        assert set(analysis_df["model_key"]) == {"m_valid"}
        assert len(df_combined) == 24
        assert set(df_combined["model_key"]) == {"m_valid", "m_invalid"}

        # run_full_statistical_analysis on filtered frame
        rep = run_full_statistical_analysis(analysis_df)
        assert "m_valid_bn" in rep.get("within_model_results", {})
        assert "m_invalid_bn" not in rep.get("within_model_results", {})
        for r in rep.get("cross_model_descriptive_table", []):
            assert r["Model"] != "m_invalid"
    test_registry.append(("analysis_valid_only filters invalid runs from analysis subset and statistical report", test_analysis_valid_only_filtering))

    # --- Task H: Calibration & Fragmentation Pipeline Enhancements (v3.2) ---
    def test_evaluator_prompt_tokens_and_resume_guard():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            be = MockInferenceBackend(seed=1)
            ev = BenchmarkEvaluator(be, model_key="unit_test_model", mode="extended", output_dir=td,
                                    language="bn", benchmark_name="BnMMLU")
            items = [dict(it) for it in create_synthetic_benchmark_dataset() if it["language"] == "bn"][:3]
            ev.evaluate_all(items, resume=False, show_progress=False)
            df_res = pd.read_csv(ev.results_csv)
            assert "prompt_tokens" in df_res.columns, "prompt_tokens column missing from evaluator output CSV"
            assert df_res["prompt_tokens"].notna().all(), "prompt_tokens should not be all NaN"

            # Create an older results CSV without prompt_tokens and assert resume guard raises RuntimeError
            td_old = os.path.join(td, "old_model")
            os.makedirs(td_old, exist_ok=True)
            old_csv = os.path.join(td_old, "bn_results.csv")
            df_old = df_res.drop(columns=["prompt_tokens"])
            df_old.to_csv(old_csv, index=False)
            try:
                ev_old = BenchmarkEvaluator(be, model_key="old_model", mode="extended", output_dir=td,
                                            language="bn", benchmark_name="BnMMLU")
                assert False, "Should raise RuntimeError when prompt_tokens column is missing from existing CSV"
            except RuntimeError as e:
                assert "prompt_tokens" in str(e).lower()
    test_registry.append(("evaluator rows include prompt_tokens and resume guard raises on missing column", test_evaluator_prompt_tokens_and_resume_guard))

    def test_fit_confidence_correctness_models_planted_effect():
        rng = np.random.RandomState(42)
        n = 300
        frag_z = rng.randn(n)
        length_z = rng.randn(n)
        # confidence = 0.5 + 0.15*frag_z + noise
        confidence = np.clip(0.5 + 0.15 * frag_z + 0.05 * length_z + 0.05 * rng.randn(n), 0.0, 1.0)
        # is_correct independent of frag_z
        is_correct = (rng.rand(n) > 0.6).astype(float)
        subjects = ["math", "bio", "chem", "phys", "hist", "lit", "geo", "econ", "cs", "phil"]
        df_planted = pd.DataFrame({
            "confidence": confidence,
            "is_correct": is_correct,
            "tpw": frag_z * 2.0 + 5.0,
            "prompt_tokens": length_z * 10.0 + 50.0,
            "subject_norm": [subjects[i % len(subjects)] for i in range(n)],
        })
        res = fit_confidence_correctness_models(df_planted, frag_metric="tpw")
        assert res["status"] == "success"
        m1 = res["models"]["M1_confidence"]
        m2 = res["models"]["M2_is_correct"]
        m3 = res["models"]["M3_overconfidence"]
        assert m1["coef"] > 0.08 and m1["pvalue"] < 0.01, f"M1 coef expected positive & significant, got {m1}"
        assert abs(m2["coef"]) < 0.06 and m2["pvalue"] > 0.05, f"M2 coef expected near zero / non-significant, got {m2}"
        assert m3["coef"] > 0.05 and m3["pvalue"] < 0.01, f"M3 coef expected positive matching M1, got {m3}"
    test_registry.append(("fit_confidence_correctness_models recovers planted effect", test_fit_confidence_correctness_models_planted_effect))

    def test_fit_language_interaction_planted_difference_and_skip():
        rng = np.random.RandomState(42)
        n_each = 200
        subjects = ["math", "bio", "chem", "phys", "hist", "lit", "geo", "econ", "cs", "phil"]
        # bn: slope 0.15
        frag_bn = rng.randn(n_each)
        len_bn = rng.randn(n_each)
        conf_bn = np.clip(0.5 + 0.15 * frag_bn + 0.05 * len_bn + 0.05 * rng.randn(n_each), 0.0, 1.0)
        corr_bn = (rng.rand(n_each) > 0.5).astype(float)
        df_bn = pd.DataFrame({
            "model_key": ["m1"] * n_each,
            "language": ["bn"] * n_each,
            "confidence": conf_bn,
            "is_correct": corr_bn,
            "tpw": frag_bn * 2.0 + 5.0,
            "prompt_tokens": len_bn * 10.0 + 50.0,
            "subject_norm": [subjects[i % len(subjects)] for i in range(n_each)],
            "validity_flag": ["VALID"] * n_each,
        })
        # en: slope 0.0
        frag_en = rng.randn(n_each)
        len_en = rng.randn(n_each)
        conf_en = np.clip(0.5 + 0.0 * frag_en + 0.05 * len_en + 0.05 * rng.randn(n_each), 0.0, 1.0)
        corr_en = (rng.rand(n_each) > 0.5).astype(float)
        df_en = pd.DataFrame({
            "model_key": ["m1"] * n_each,
            "language": ["en"] * n_each,
            "confidence": conf_en,
            "is_correct": corr_en,
            "tpw": frag_en * 0.5 + 1.2,
            "prompt_tokens": len_en * 8.0 + 40.0,
            "subject_norm": [subjects[i % len(subjects)] for i in range(n_each)],
            "validity_flag": ["VALID"] * n_each,
        })
        df_both = pd.concat([df_bn, df_en], ignore_index=True)
        res = fit_language_interaction(df_both, frag_metric="tpw")
        assert res["status"] == "success"
        conf_inter = res["outcomes"]["confidence"]
        ci = conf_inter["ci_95"]
        assert not (ci[0] <= 0 <= ci[1]), f"Confidence interval should exclude 0: {ci}"

        # Test skipped status when one language is missing or invalid
        res_skip = fit_language_interaction(df_bn, frag_metric="tpw")
        assert res_skip.get("status") == "skipped_invalid_or_missing_language"

        # Test skipped status with correct informative message when one language is INVALID
        df_one_invalid = df_both.copy()
        df_one_invalid.loc[df_one_invalid["language"] == "en", "validity_flag"] = "INVALID"
        res_invalid = fit_language_interaction(df_one_invalid, frag_metric="tpw")
        assert res_invalid.get("status") == "skipped_invalid_or_missing_language"
        msg = res_invalid.get("message", "")
        assert "Languages before filter" in msg and "bn" in msg and "en" in msg
        assert "dropped invalid" in msg and "en" in msg
    test_registry.append(("fit_language_interaction recovers planted slope difference and handles missing or invalid language", test_fit_language_interaction_planted_difference_and_skip))

    def test_holm_adjustment_monotone():
        raw_p = [0.001, 0.04, 0.02, 0.5, 0.005]
        reject, adj_p, _, _ = multipletests(raw_p, method="holm")
        order = np.argsort(raw_p)
        sorted_raw = np.array(raw_p)[order]
        sorted_adj = np.array(adj_p)[order]
        assert np.all(sorted_adj >= sorted_raw), "Adjusted p must be >= raw p"
        assert np.all(np.diff(sorted_adj) >= -1e-12), "Adjusted p must be monotone in sorted raw p order"
    test_registry.append(("Holm adjustment: adjusted p >= raw p and monotone", test_holm_adjustment_monotone))

    def test_length_control_fallback():
        # Missing prompt_tokens
        df_missing = pd.DataFrame({"total_tokens": [10, 20, 30]})
        s, col = get_length_control_series(df_missing)
        assert col == "total_tokens" and list(s) == [10, 20, 30]

        # Constant prompt_tokens
        df_const = pd.DataFrame({"prompt_tokens": [50, 50, 50], "total_tokens": [10, 20, 30]})
        s_c, col_c = get_length_control_series(df_const)
        assert col_c == "total_tokens"

        # Valid prompt_tokens
        df_valid = pd.DataFrame({"prompt_tokens": [40, 50, 60], "total_tokens": [10, 20, 30]})
        s_v, col_v = get_length_control_series(df_valid)
        assert col_v == "prompt_tokens" and list(s_v) == [40, 50, 60]
    test_registry.append(("Length-control helper falls back to total_tokens with warning", test_length_control_fallback))

    def test_plot_reliability_diagram_headless():
        import tempfile
        bins = compute_calibration_bins(np.array([0.2, 0.8, 0.9]), np.array([0, 1, 1]), n_bins=5)
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tf:
            out_path = tf.name
        try:
            # Simulate or use available matplotlib Agg
            plot_reliability_diagram(bins, title="Headless Test Diagram", save_path=out_path)
            if HAS_VISUALIZATION:
                assert os.path.exists(out_path)
                assert os.path.getsize(out_path) > 0
        finally:
            if os.path.exists(out_path):
                os.unlink(out_path)
    test_registry.append(("plot_reliability_diagram runs headless and writes PNG", test_plot_reliability_diagram_headless))

    def test_option_mass_filtered_ece():
        # Planted items: 10 high-mass well-calibrated, 10 low-mass (mass=0.1) miscalibrated
        conf = [0.9] * 10 + [0.9] * 10
        corr = [1] * 10 + [0] * 10
        mass = [0.8] * 10 + [0.1] * 10
        df_mass = pd.DataFrame({
            "is_correct": corr,
            "confidence": conf,
            "option_mass": mass,
            "overall_tpw": [2.0] * 20,
            "language": ["bn"] * 20,
            "model_key": ["m_test"] * 20,
            "subject": ["bio"] * 20,
            "subject_group": ["bio"] * 20,
        })
        rep = run_full_statistical_analysis(df_mass)
        wres = rep["within_model_results"]["m_test_bn"]
        mf = wres["mass_filtered_calibration"]
        assert mf["n_mass_filtered"] == 10
        assert mf["share_mass_filtered_out"] == 0.5
        assert mf["ece_mass_filtered"] < wres["calibration"]["ece_uniform_10"]
    test_registry.append(("Option-mass filtered ECE excludes low-mass items", test_option_mass_filtered_ece))

    def test_run_full_statistical_analysis_legacy_tiny_frame():
        df_tiny = pd.DataFrame({
            "is_correct": [1, 0] * 10,
            "confidence": [0.8, 0.6] * 10,
            "overall_tpw": [1.5, 3.5] * 10,
            "language": ["bn"] * 20,
            "model_key": ["mock-model"] * 20,
            "subject": ["general"] * 20,
        })
        rep = run_full_statistical_analysis(df_tiny)
        assert rep["dataset_summary"]["total_analyzed_items"] == 20
        assert "m3_overconfidence" in str(rep).lower() or "h2_models" in rep["within_model_results"]["mock-model_bn"]
    test_registry.append(("run_full_statistical_analysis still works on old tiny frame without new columns", test_run_full_statistical_analysis_legacy_tiny_frame))

    def test_config_validation_rules():
        # Valid config: prompt_variant='answer_cue', n_shots=0
        cfg_valid = dataclasses.replace(CONFIG, prompt_variant="answer_cue", n_shots=0)
        validate_config(cfg_valid)  # Should not raise

        # Invalid config: prompt_variant='few_shot', n_shots=5
        cfg_bad = dataclasses.replace(CONFIG, prompt_variant="few_shot", n_shots=5)
        raised = False
        try:
            validate_config(cfg_bad)
        except ValueError:
            raised = True
        assert raised, "validate_config should raise ValueError for prompt_variant='few_shot' and n_shots=5"

        # Valid few_shot configs: 4 and 8
        validate_config(dataclasses.replace(CONFIG, prompt_variant="few_shot", n_shots=4))
        validate_config(dataclasses.replace(CONFIG, prompt_variant="few_shot", n_shots=8))
    test_registry.append(("validate_config enforces conditional n_shots rules", test_config_validation_rules))

    def test_config_n_samples_is_1500():
        assert CONFIG.n_samples == 1500, f"Expected CONFIG.n_samples == 1500, but got {CONFIG.n_samples}"
        assert ExperimentConfig().n_samples == 1500, f"Expected ExperimentConfig default n_samples == 1500, but got {ExperimentConfig().n_samples}"
    test_registry.append(("CONFIG.n_samples is exactly 1500 in CONFIG and default", test_config_n_samples_is_1500))

    def test_balanced_subject_sampler():
        import tempfile, types, dataclasses

        class _DS:
            def __init__(self, df): self._df = df
            def to_pandas(self): return self._df.copy()

        # 10 mapped subjects with very different sizes
        diff_sizes = [5000, 3000, 800, 500, 300, 200, 150, 100, 80, 60]
        subj_names = [
            "high_school_physics", "high_school_biology", "high_school_chemistry",
            "college_physics", "college_biology", "college_chemistry",
            "high_school_mathematics", "college_mathematics",
            "high_school_psychology", "professional_psychology"
        ]

        def _make_stubs():
            bn_rows = []
            for s, sz in zip(subj_names, diff_sizes):
                for k in range(sz):
                    bn_rows.append({
                        "Unique_Serial": f"bn_{s}_{k}",
                        "subject_name": s,
                        "question": f"Question {s} {k}",
                        "correct_answer": "abcd"[k % 4],
                        "options": np.array(["ক", "খ", "গ", "ঘ"], dtype=object),
                        "question_char_count": 20,
                    })
            en_rows = []
            for s, sz in zip(subj_names, diff_sizes):
                for k in range(sz):
                    en_rows.append({
                        "question": f"Question {s} {k}",
                        "subject": s,
                        "choices": ["w", "x", "y", "z"],
                        "answer": k % 4,
                    })
            return pd.DataFrame(bn_rows), en_rows

        bn_df_stub, en_rows_stub = _make_stubs()

        def _fake_load_dataset(repo, cfg=None, split=None, token=None, **kw):
            if repo == "samanjoy2/BnMMLU":
                return {"train": _DS(bn_df_stub)}
            if repo == "cais/mmlu":
                if split == "dev":
                    return [{"question": f"Dev {s} {k}", "subject": s, "choices": ["a","b","c","d"], "answer": k % 4} for s in subj_names for k in range(8)]
                return en_rows_stub
            raise AssertionError(repo)

        fake = types.ModuleType("datasets"); fake.load_dataset = _fake_load_dataset
        old_mod = sys.modules.get("datasets"); old_rev = globals().get("_hub_revision")
        sys.modules["datasets"] = fake; globals()["_hub_revision"] = lambda *a, **k: "test-sha"
        try:
            with tempfile.TemporaryDirectory() as td1, tempfile.TemporaryDirectory() as td2:
                cfg = dataclasses.replace(
                    CONFIG, backend="transformers", hf_token="tok", bn_local_path=None,
                    exclude_markup=False, subject_match_mode="intersection",
                    allow_fewshot_test_pool_fallback=True
                )
                # Request n_items=600
                bn_p1, en_p1 = prepare_research_subsets(output_dir=td1, n_items=600, seed=42, hf_token="tok", is_mock=False, config=cfg)
                bn1 = pd.read_csv(bn_p1); en1 = pd.read_csv(en_p1)

                # Exactly 600 per language
                assert len(bn1) == 600, f"Expected 600 items, got {len(bn1)}"
                assert len(en1) == 600, f"Expected 600 items, got {len(en1)}"

                # Identical per-subject counts in bn and en
                bn_counts = bn1["subject_group"].value_counts().to_dict()
                en_counts = en1["subject_group"].value_counts().to_dict()
                assert bn_counts == en_counts, f"Subject counts mismatch between bn and en: {bn_counts} vs {en_counts}"

                # Capped allocation check: per_subject_cap = ceil(600 / 10 * 1.5) = 90
                cap = math.ceil(600 / 10 * 1.5)
                for s, cnt in bn_counts.items():
                    assert cnt <= cap, f"Subject '{s}' count {cnt} exceeds per_subject_cap {cap}"

                # Determinism check with same seed
                bn_p2, en_p2 = prepare_research_subsets(output_dir=td2, n_items=600, seed=42, hf_token="tok", is_mock=False, config=cfg)
                bn2 = pd.read_csv(bn_p2); en2 = pd.read_csv(en_p2)
                assert bn1["question_id"].tolist() == bn2["question_id"].tolist()
                assert en1["question_id"].tolist() == en2["question_id"].tolist()

                # Few-shot candidate safety check (no overlap with evaluation question IDs)
                fewshot_p = os.path.join(td1, "fewshot_exemplars.json")
                assert os.path.exists(fewshot_p)
                with open(fewshot_p, "r", encoding="utf-8") as fs_f:
                    fs_data = json.load(fs_f)
                bn_eval_set = set(bn1["question_id"].astype(str).tolist())
                for ex in fs_data.get("bn", []):
                    assert str(ex.get("question_id", "")) not in bn_eval_set, "Few-shot exemplar overlaps evaluation set!"
        finally:
            if old_mod is not None: sys.modules["datasets"] = old_mod
            else: sys.modules.pop("datasets", None)
            if old_rev is not None: globals()["_hub_revision"] = old_rev
    test_registry.append(("balanced subject sampler caps subjects, matches bn and en, and is deterministic", test_balanced_subject_sampler))

    def test_insufficient_pairs_runtime_error_with_table():
        import tempfile, types, dataclasses

        class _DS:
            def __init__(self, df): self._df = df
            def to_pandas(self): return self._df.copy()

        bn_df_stub = pd.DataFrame([
            {"Unique_Serial": "bn_0", "subject_name": "high_school_physics", "question": "Q1", "correct_answer": "a", "options": np.array(["a","b","c","d"], dtype=object), "question_char_count": 5},
            {"Unique_Serial": "bn_1", "subject_name": "high_school_physics", "question": "Q2", "correct_answer": "b", "options": np.array(["a","b","c","d"], dtype=object), "question_char_count": 5},
        ])
        en_rows_stub = [
            {"question": "Q1", "subject": "high_school_physics", "choices": ["a","b","c","d"], "answer": 0},
            {"question": "Q2", "subject": "high_school_physics", "choices": ["a","b","c","d"], "answer": 1},
        ]

        def _fake_load_dataset(repo, cfg=None, split=None, token=None, **kw):
            if repo == "samanjoy2/BnMMLU": return {"train": _DS(bn_df_stub)}
            if repo == "cais/mmlu": return en_rows_stub
            raise AssertionError(repo)

        fake = types.ModuleType("datasets"); fake.load_dataset = _fake_load_dataset
        old_mod = sys.modules.get("datasets"); old_rev = globals().get("_hub_revision")
        sys.modules["datasets"] = fake; globals()["_hub_revision"] = lambda *a, **k: "test-sha"
        try:
            with tempfile.TemporaryDirectory() as td:
                cfg = dataclasses.replace(
                    CONFIG, backend="transformers", hf_token="tok", bn_local_path=None,
                    exclude_markup=False, subject_match_mode="intersection"
                )
                raised = False
                try:
                    prepare_research_subsets(output_dir=td, n_items=100, seed=42, hf_token="tok", is_mock=False, config=cfg)
                except RuntimeError as re:
                    raised = True
                    err_msg = str(re)
                    assert "Fewer than 100" in err_msg
                    assert "Subject-matched pairs per mapped subject (n_bn/n_en):" in err_msg
                    assert "high school physics" in err_msg
                assert raised, "Expected RuntimeError when total available pairs < n_items"
        finally:
            if old_mod is not None: sys.modules["datasets"] = old_mod
            else: sys.modules.pop("datasets", None)
            if old_rev is not None: globals()["_hub_revision"] = old_rev
    test_registry.append(("prepare_research_subsets raises informative RuntimeError when available pairs < n_items", test_insufficient_pairs_runtime_error_with_table))

    def test_prepare_subsets_english_items_are_disjoint():
        # Two BnMMLU subjects share the same MMLU pool (50 English items in total): every English item may be used once.
        import tempfile, types, dataclasses

        class _DS:
            def __init__(self, df): self._df = df
            def to_pandas(self): return self._df.copy()

        bn_rows = []
        for s in ("cell biology genetics", "ecology environmental biology"):
            for j in range(40):
                bn_rows.append({"Unique_Serial": f"{s[:4]}_{j}", "subject_name": s, "question": f"{s} Q{j}",
                                "correct_answer": "abcd"[j % 4], "options": np.array(["a", "b", "c", "d"], dtype=object),
                                "question_char_count": 8})
        bn_df_stub = pd.DataFrame(bn_rows)
        en_rows = [{"question": f"{s} EQ{j}", "subject": s, "choices": ["a", "b", "c", "d"], "answer": j % 4}
                   for s, n in (("high_school_biology", 30), ("college_biology", 20)) for j in range(n)]
        dev_rows = [{"question": f"dev Q{j}", "subject": f"dev_subj_{j}", "choices": ["a", "b", "c", "d"], "answer": j % 4}
                    for j in range(16)]

        def _fake_load_dataset(repo, cfg=None, split=None, token=None, **kw):
            if repo == "samanjoy2/BnMMLU": return {"train": _DS(bn_df_stub)}
            if repo == "cais/mmlu": return dev_rows if split == "dev" else en_rows
            raise AssertionError(repo)

        fake = types.ModuleType("datasets"); fake.load_dataset = _fake_load_dataset
        old_mod = sys.modules.get("datasets"); old_rev = globals().get("_hub_revision")
        sys.modules["datasets"] = fake; globals()["_hub_revision"] = lambda *a, **k: "test-sha"
        try:
            cfg = dataclasses.replace(CONFIG, backend="transformers", hf_token="tok", bn_local_path=None,
                                      dev_n=0, exclude_markup=False, subject_match_mode="intersection")
            with tempfile.TemporaryDirectory() as td:
                b, e = prepare_research_subsets(output_dir=td, n_items=50, seed=42, hf_token="tok", is_mock=False, config=cfg)
                bd, ed = pd.read_csv(b), pd.read_csv(e)
                assert len(bd) == len(ed) == 50
                assert ed["question_id"].is_unique and bd["question_id"].is_unique, "an item was selected more than once"
                assert bd["subject_group"].value_counts().sort_index().equals(ed["subject_group"].value_counts().sort_index())
                with open(os.path.join(td, "subject_mapping.json"), encoding="utf-8") as f:
                    mj = json.load(f)
                assert "coverage_of_bn_pool" in mj and mj["n_unique_en_ids"] == 50
            with tempfile.TemporaryDirectory() as td:
                raised = False
                try:  # only 50 distinct English items exist, so 60 pairs are impossible even though per-group pools overlap
                    prepare_research_subsets(output_dir=td, n_items=60, seed=42, hf_token="tok", is_mock=False, config=cfg)
                except RuntimeError as err:
                    raised = True
                    assert "Fewer than 60" in str(err)
                assert raised, "Expected RuntimeError: disjoint English capacity is 50"
        finally:
            if old_mod is not None: sys.modules["datasets"] = old_mod
            else: sys.modules.pop("datasets", None)
            if old_rev is not None: globals()["_hub_revision"] = old_rev
    test_registry.append(("Prepared English subset uses every MMLU item at most once", test_prepare_subsets_english_items_are_disjoint))

    def test_resolve_run_settings_precedence_persisted():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            cfg = dataclasses.replace(CONFIG, output_dir=td)
            assert resolve_run_settings("tigerllm_1b", cfg)["settings_source"] in ("MODEL_REGISTRY", "CONFIG")
            save_model_settings({"tigerllm_1b": {"prompt_variant": "few_shot", "n_shots": 8,
                                                 "use_chat_template": True, "debias": "none"}}, td)
            s = resolve_run_settings("tigerllm_1b", cfg)
            assert s["prompt_variant"] == "few_shot" and s["n_shots"] == 8 and s["effective_n_shots"] == 8
            assert s["settings_source"] == "model_settings.json"
    test_registry.append(("model_settings.json overrides registry", test_resolve_run_settings_precedence_persisted))

    def test_apply_recommendation_behaviour():
        import tempfile
        rows = lambda ts, share: [{"language": l, "prompt_variant": "plain", "chat_template": False, "debias": "none",
                                   "mean_mass": 0.8, "min_mass": 0.7, "top_letter_share": share,
                                   "raw_top_letter_share": share, "accuracy_pct": 50.0, "n_nan": 0,
                                   "mean_prompt_tokens": 50.0, "n_perm_exempt": None} for l in ("bn", "en")]
        orig = dict(MODEL_REGISTRY["tigerllm_1b"])
        try:
            with tempfile.TemporaryDirectory() as td:
                cfg = dataclasses.replace(CONFIG, output_dir=td)
                csv_ok = os.path.join(td, "ok.csv"); pd.DataFrame(rows(0, 0.3)).to_csv(csv_ok, index=False)
                csv_bad = os.path.join(td, "bad.csv"); pd.DataFrame(rows(0, 0.99)).to_csv(csv_bad, index=False)
                apply_recommendation("tigerllm_1b", diag_csv=csv_ok, dry_run=True, config=cfg)
                assert not os.path.exists(get_model_settings_path(td)), "dry_run must not write"
                apply_recommendation("tigerllm_1b", diag_csv=csv_ok, dry_run=False, config=cfg)
                saved = load_model_settings(td)["tigerllm_1b"]
                assert saved["provenance"]["gates_passed"] is True and "timestamp" in saved["provenance"]
                try:
                    apply_recommendation("tigerllm_1b", diag_csv=csv_bad, dry_run=True, config=cfg)
                    assert False, "must raise when no candidate passes"
                except RuntimeError:
                    pass
                forced = apply_recommendation("tigerllm_1b", diag_csv=csv_bad, dry_run=True,
                                              force_best_candidate=True, config=cfg)
                assert forced["provenance"]["gates_passed"] is False
        finally:
            MODEL_REGISTRY["tigerllm_1b"] = orig
    test_registry.append(("apply_recommendation dry-run/write/no-pass/force", test_apply_recommendation_behaviour))

    def test_reset_model_results_confirm():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            cfg = dataclasses.replace(CONFIG, output_dir=td)
            os.makedirs(os.path.join(td, "tigerllm_1b")); os.makedirs(os.path.join(td, "llama32_3b"))
            reset_model_results("tigerllm_1b", confirm=False, config=cfg)
            assert os.path.isdir(os.path.join(td, "tigerllm_1b"))
            reset_model_results("tigerllm_1b", confirm=True, config=cfg)
            assert not os.path.isdir(os.path.join(td, "tigerllm_1b"))
            assert os.path.isdir(os.path.join(td, "llama32_3b")), "must delete only that model"
            try:
                reset_model_results("nope", confirm=True, config=cfg); assert False
            except ValueError:
                pass
    test_registry.append(("reset_model_results validates key and needs confirm", test_reset_model_results_confirm))

    def test_missing_dev_files_error():
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            cfg = dataclasses.replace(CONFIG, output_dir=td)
            try:
                diagnose_prompt_variants("tigerllm_1b", n=5, config=cfg); assert False
            except RuntimeError as e:
                assert "rerun data preparation" in str(e)
    test_registry.append(("diagnose raises clear error when dev files are missing", test_missing_dev_files_error))

    def test_build_backend_forwards_chat_flag():
        class _Rec:
            pass
        calls = []
        orig = globals()["create_inference_backend"]
        def _fake(backend_type, model_id, **kw):
            calls.append(kw.get("use_chat_template"))
            return _Rec()
        key = next(iter(MODEL_REGISTRY))
        reg_val = bool(MODEL_REGISTRY[key].get("use_chat_template", False))
        globals()["create_inference_backend"] = _fake
        try:
            _build_backend(key, CONFIG)
            _build_backend(key, CONFIG, use_chat_template=True)
            _build_backend(key, CONFIG, use_chat_template=False)
        finally:
            globals()["create_inference_backend"] = orig
        assert calls == [reg_val, True, False], f"unexpected use_chat_template forwarding: {calls}"
    test_registry.append(("_build_backend forwards use_chat_template (None -> registry value)", test_build_backend_forwards_chat_flag))

    def test_static_call_signatures():
        import ast as _ast, glob as _glob, json as _json, os as _os, sys as _sys
        dirs = []
        mf = getattr(_sys.modules.get("__main__"), "__file__", None)
        if mf:
            dirs.append(_os.path.dirname(_os.path.abspath(mf)))
        dirs += [_os.getcwd(), "/kaggle/working"]
        cands = []
        override = globals().get("NOTEBOOK_PATH")
        if override:
            cands.append(override)
        for d in dirs:
            cands += sorted(_glob.glob(_os.path.join(d, "*.ipynb")))
        nb_json = None
        for c in cands:
            try:
                with open(c, encoding="utf-8") as f:
                    cand = _json.load(f)
            except Exception:
                continue
            if any(x.get("cell_type") == "code" and "def _build_backend" in "".join(x.get("source", [])) for x in cand.get("cells", [])):
                nb_json = cand
                break
        if nb_json is None:
            logger.info("   (static call-signature audit skipped: notebook file not found next to the kernel)")
            return
        defs = {}
        trees = []
        for ci, cell in enumerate(nb_json.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue
            try:
                tree = _ast.parse("".join(cell.get("source", [])))
            except SyntaxError:
                continue
            trees.append((ci, tree))
            for node in _ast.walk(tree):
                if isinstance(node, _ast.FunctionDef):
                    a = node.args
                    pos = [x.arg for x in a.posonlyargs + a.args]
                    defs.setdefault(node.name, []).append({
                        "kw": set(pos) | {x.arg for x in a.kwonlyargs},
                        "varkw": a.kwarg is not None,
                        "maxpos": None if a.vararg else len(pos),
                        "method": bool(pos) and pos[0] in ("self", "cls"),
                    })
        bad = []
        for ci, tree in trees:
            for node in _ast.walk(tree):
                if not (isinstance(node, _ast.Call) and isinstance(node.func, _ast.Name)):
                    continue
                sigs = defs.get(node.func.id, [])
                if len(sigs) != 1 or sigs[0]["method"] or node.func.id.startswith("__"):
                    continue
                sig = sigs[0]
                if sig["maxpos"] is not None and len(node.args) > sig["maxpos"] and not any(isinstance(x, _ast.Starred) for x in node.args):
                    bad.append(f"cell {ci} line {node.lineno}: {node.func.id}() too many positional args")
                for kw in node.keywords:
                    if kw.arg is not None and not sig["varkw"] and kw.arg not in sig["kw"]:
                        bad.append(f"cell {ci} line {node.lineno}: {node.func.id}() unexpected keyword '{kw.arg}'")
        assert not bad, "call/signature mismatches: " + "; ".join(bad[:8])
    test_registry.append(("Static call-signature audit of notebook-defined functions", test_static_call_signatures))

    def test_apply_uniform_setting():
        import tempfile
        keys = ["llama32_3b", "tigerllm_1b", "titullm_1b"]
        orig = {k: dict(MODEL_REGISTRY[k]) for k in keys}
        try:
            with tempfile.TemporaryDirectory() as td:
                cfg = dataclasses.replace(CONFIG, output_dir=td)
                apply_uniform_setting(keys, config=cfg, dry_run=True)
                assert not os.path.exists(get_model_settings_path(td)), "dry_run must not write"
                try:
                    apply_uniform_setting(["nope"], config=cfg, dry_run=False); assert False
                except ValueError:
                    pass
                apply_uniform_setting(keys, config=cfg, dry_run=False)
                for k in keys:
                    s = resolve_run_settings(k, cfg)
                    assert s["prompt_variant"] == "few_shot" and s["effective_n_shots"] == 4
                    assert s["use_chat_template"] is False and s["debias"] == "none"
                    assert s["settings_source"] == "model_settings.json"
        finally:
            for k in keys:
                MODEL_REGISTRY[k] = orig[k]
    test_registry.append(("apply_uniform_setting dry-run/write/validation", test_apply_uniform_setting))

    passed_count = 0
    total_count = len(test_registry)

    for name, test_fn in test_registry:
        try:
            test_fn()
            passed_count += 1
            logger.info(f"  ✅ Test {passed_count}/{total_count}: {name} passed.")
        except Exception as e:
            print(f"TEST_FAILED_NAME: {name} | ERROR: {e}")
            logger.error(f"  ❌ Test failed [{name}]: {e}")

    logger.info("=" * 65)
    logger.info(f"🎯 UNIT TESTS SUMMARY: {passed_count} / {total_count} PASSED")
    logger.info("=" * 65)
    return passed_count == total_count

tests_ok = run_notebook_unit_tests()
assert tests_ok, "One or more notebook unit tests failed!"


## Section 14b: Pre-flight Tokenizer Audit (run BEFORE the expensive model runs)
Loads **only tokenizers** (no weights) and measures TPW / STRR / TPC on the real Bangla and English subsets. This answers two questions the whole study depends on:
1. Do the models really differ in Bangla fragmentation? (`TituLLM` is described as a Bangla-extended Llama, but the v1.1 model card only says "continually pre-trained"; if its tokenizer equals Llama-3.2's, H1 has no contrast between those two.)
2. Is each tokenizer accessible with your token? (Llama-3.2 is gated.)

A warning is logged if two models share an identical tokenizer. Results are saved to `tokenizer_audit.csv`; the table is also a ready-made descriptive table for the paper.


In [ ]:
def run_tokenizer_audit(model_keys: Optional[List[str]] = None, n_items: int = 300) -> pd.DataFrame:
    """Tokenizer-only pre-flight (no model weights are downloaded).

    Reports vocabulary size and TPW / STRR / TPC on the REAL Bangla and English subsets for every registered model, and
    warns when two models share an identical tokenizer (in which case they cannot contrast low vs high fragmentation).
    """
    from transformers import AutoTokenizer

    keys = model_keys or list(MODEL_REGISTRY.keys())
    bn_all = load_dataset_from_file(bn_subset_file, "bn", "BnMMLU", is_mock=False)
    en_all = load_dataset_from_file(en_subset_file, "en", "MMLU", is_mock=False)
    rng = np.random.RandomState(CONFIG.seed)
    bn_items = [bn_all[i] for i in rng.permutation(len(bn_all))[:n_items]]
    en_items = [en_all[i] for i in rng.permutation(len(en_all))[:n_items]]

    probe = "বাংলাদেশের রাজধানী ঢাকা এবং আমাদের মাতৃভাষা বাংলা।"
    rows: List[Dict[str, Any]] = []
    fingerprints: Dict[str, Tuple[int, Tuple[int, ...]]] = {}

    for key in keys:
        info = MODEL_REGISTRY[key]
        try:
            tok = AutoTokenizer.from_pretrained(info["hf_id"], token=CONFIG.hf_token)
        except Exception as e:
            logger.error(f"Tokenizer for '{key}' ({info['hf_id']}) could not be loaded: {type(e).__name__}: {e}. "
                         "For gated models (Llama) accept the license on the model page and check HF_TOKEN.")
            rows.append({"model_key": key, "hf_id": info["hf_id"], "status": f"FAILED: {type(e).__name__}"})
            continue
        probe_ids = tuple(tok.encode(probe, add_special_tokens=False))
        fingerprints[key] = (len(tok), probe_ids)
        _, bn_sum = compute_dataset_fragmentation(info["hf_id"], bn_items, CONFIG.hf_token)
        _, en_sum = compute_dataset_fragmentation(info["hf_id"], en_items, CONFIG.hf_token)
        rows.append({
            "model_key": key, "hf_id": info["hf_id"], "status": "ok",
            "tokenizer_class": type(tok).__name__, "vocab_size": len(tok),
            "probe_sentence_tokens": len(probe_ids),
            "bn_mean_tpw": round(bn_sum["mean_tpw"], 3), "bn_mean_strr": round(bn_sum["mean_strr_isolated"], 3),
            "bn_mean_tpc": round(bn_sum["mean_tpc"], 3), "en_mean_tpw": round(en_sum["mean_tpw"], 3),
            "bn_over_en_tpw": round(bn_sum["mean_tpw"] / en_sum["mean_tpw"], 2) if en_sum["mean_tpw"] else float("nan"),
        })

    audit_df = pd.DataFrame(rows)

    # Identical-tokenizer check (same vocabulary size AND same ids on a Bangla probe sentence)
    ok_keys = list(fingerprints)
    for i in range(len(ok_keys)):
        for j in range(i + 1, len(ok_keys)):
            a, b = ok_keys[i], ok_keys[j]
            if fingerprints[a] == fingerprints[b]:
                logger.warning(
                    f"IDENTICAL TOKENIZERS: '{a}' and '{b}' tokenize Bangla exactly the same way (vocab={fingerprints[a][0]:,}). "
                    "Their fragmentation is equal by construction, so this pair cannot test H1 (low vs high fragmentation). "
                    "Pick a model with a different tokenizer for the 'low-fragmentation' arm, or report them as a null contrast."
                )
            elif fingerprints[a][0] != fingerprints[b][0]:
                logger.info(f"Tokenizers differ: '{a}' vocab={fingerprints[a][0]:,} vs '{b}' vocab={fingerprints[b][0]:,}.")

    out_path = os.path.join(CONFIG.output_dir, "tokenizer_audit.csv")
    audit_df.to_csv(out_path, index=False, encoding="utf-8")
    print("\n" + "=" * 88 + "\nTOKENIZER AUDIT (real subsets, tokenizer-only)\n" + "=" * 88)
    print(audit_df.to_string(index=False))
    print("=" * 88)
    logger.info(f"Tokenizer audit saved to: {out_path}")
    return audit_df


# Run Tokenizer Audit on real tokenizers and real subsets
tokenizer_audit_df = run_tokenizer_audit()


## Section 11b: Prompt-variant diagnostic (run per model BEFORE the full run)
`diagnose_prompt_variants(model_key, n=60)` loads the backend once and evaluates the full grid:
- `prompt_variant`: `{plain, answer_cue, few_shot}` (skips `few_shot` if `fewshot_exemplars.json` is missing)
- `chat_template`: `{off, on}` (evaluates `on` only if `tokenizer.chat_template` exists)
- `debias`: `{none, cyclic4}`

### Methodology & Decision Rules
1. **Validity Gates Only:** Settings are selected using ONLY `mean_mass >= MIN_OPTION_MASS_THRESHOLD` (0.50) and `top_letter_share <= MAX_COLLAPSE_THRESHOLD` (0.80) across BOTH Bangla and English.
2. **Never Select by Accuracy:** The diagnostic subset overlaps the evaluation pool. Selecting variants by accuracy would cause data leakage and invalidate calibration results.
3. **Preference Hierarchy:** Among configurations passing both gates in both languages, prefer:
   - `debias`: `"none"` over `"cyclic4"`
   - `prompt_variant`: `plain` < `answer_cue` < `few_shot`
   - `chat_template`: `False` (off) over `True` (on)
   - Highest minimum `mean_mass` across languages
4. **Re-run Workflow Reminder:** If re-running a model after updating its registry configuration, delete its existing results folder first (`shutil.rmtree(os.path.join(CONFIG.output_dir, model_key))` or `rm -rf results/<model_key>`). Otherwise, the resume guard will detect the signature change or existing checkpoints will prevent clean re-evaluation.

> **Section 11b diagnostics are currently commented out. TituLLM (base model) and TigerLLM collapse to letter 'A' under default settings. Run diagnose_prompt_variants for both, copy the printed settings into MODEL_REGISTRY, delete results/<model_key>, then re-run.**


In [ ]:
# ============================================================
# Section 11b: Diagnostic Execution
# Uncomment and run per model BEFORE the full evaluation.
# Copy the printed RECOMMENDATION settings into MODEL_REGISTRY.
# If re-running a model with new settings, delete its folder first:
#   import shutil, os
#   shutil.rmtree(os.path.join(CONFIG.output_dir, "tigerllm_1b"), ignore_errors=True)
# ============================================================
# diagnose_prompt_variants("tigerllm_1b", n=60)
# diagnose_prompt_variants("titullm_1b", n=60)
# diagnose_prompt_variants("llama32_3b", n=60)

## Section 15: Run Model 1 — LLaMA-3.2-3B-Instruct
Runs the high-fragmentation anchor model (`meta-llama/Llama-3.2-3B-Instruct`) across Bangla and English subsets with automatic pre-flight smoke testing and GPU memory clearance.


In [ ]:
# Execute Model 1: LLaMA-3.2-3B-Instruct
logger.info("Executing Model 1: LLaMA-3.2-3B...")
m1_results = run_one_model("llama32_3b", languages=("bn", "en"), config=CONFIG)
status_table(CONFIG.output_dir)


## Section 16: Run Model 2 — TituLLM-Llama-3.2-1B
Runs `hishab/titulm-llama-3.2-1b-v1.1`, a Bangla continued-pretraining of Llama-3.2-1B (base model). Treat it as the "low-fragmentation" arm **only if the Section 14b audit shows its tokenizer differs from Llama-3.2's**.


In [ ]:
# Execute Model 2: TituLLM-1B
logger.info("Executing Model 2: TituLLM-1B...")
m2_results = run_one_model("titullm_1b", languages=("bn", "en"), config=CONFIG)
status_table(CONFIG.output_dir)


## Section 17: Run Model 3 — TigerLLM-1B-it (Gemma-3 tokenizer)
Out-of-family replication model `md-nishat-008/TigerLLM-1B-it`. Its Hub config is `gemma3_text` (the paper describes a Llama-3.2 base; the Hub repo is what actually runs). Loaded in float32 on T4.


In [ ]:
# Execute Model 3: TigerLLM-1B-it (Gemma-3 text architecture)
logger.info("Executing Model 3: TigerLLM-1B-it...")
m3_results = run_one_model("tigerllm_1b", languages=("bn", "en"), config=CONFIG)
status_table(CONFIG.output_dir)


## Section 18: Cross-Model Aggregation & Econometric Hypothesis Testing
Aggregates whatever model results currently exist in `results/<model_key>/` (or attached previous Kaggle session outputs from `CONFIG.previous_results_dir`), concatenates into `combined_results.csv`, runs econometric regressions, generates reliability diagrams, and writes `run_manifest.json`.


In [ ]:
# ==========================================
# AGGREGATE PER-MODEL RESULTS ACROSS SESSIONS
# ==========================================
logger.info("Gathering model evaluation outputs for econometric aggregation...")

search_dirs = [CONFIG.output_dir]
if CONFIG.previous_results_dir and os.path.exists(CONFIG.previous_results_dir):
    search_dirs.append(CONFIG.previous_results_dir)

all_dfs = []
discovered_models = set()
valid_model_langs = set()

for s_dir in search_dirs:
    for m_key in MODEL_REGISTRY.keys():
        m_path = os.path.join(s_dir, m_key)
        if os.path.exists(m_path):
            for lang in ["bn", "en"]:
                csv_path = os.path.join(m_path, f"{lang}_results.csv")
                summary_path = os.path.join(m_path, f"{lang}_summary.json")
                val_flag = "VALID"
                if os.path.exists(summary_path):
                    try:
                        with open(summary_path, "r", encoding="utf-8") as sf:
                            s_data = json.load(sf)
                            val_flag = s_data.get("validity_flag", "VALID")
                    except Exception:
                        pass

                if val_flag == "VALID":
                    valid_model_langs.add((m_key, lang))

                if os.path.exists(csv_path):
                    try:
                        df_part = pd.read_csv(csv_path, encoding="utf-8")
                        if not df_part.empty:
                            df_part["validity_flag"] = val_flag
                            all_dfs.append(df_part)
                            discovered_models.add(m_key)
                            logger.info(f"Loaded {len(df_part):,} rows for '{m_key}' [{lang}] (validity: {val_flag}) from: {csv_path}")
                    except Exception as e:
                        logger.warning(f"Error reading {csv_path}: {e}")

if not all_dfs:
    logger.warning("No model evaluation results found to aggregate yet.")
else:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    if CONFIG.backend != "mock" and "probability_source" in combined_df.columns:
        is_mock_row = combined_df["probability_source"].astype(str).str.startswith("mock")
        if is_mock_row.any():
            logger.warning(f"Excluding {int(is_mock_row.sum()):,} mock rows from real aggregation.")
            combined_df = combined_df[~is_mock_row]
    # Ensure boolean run_valid column
    combined_df["run_valid"] = combined_df["validity_flag"] == "VALID"
    # Deduplicate by (model_key, language, question_id)
    combined_df = combined_df.drop_duplicates(subset=["model_key", "language", "question_id"]).reset_index(drop=True)
    combined_csv_path = os.path.join(CONFIG.output_dir, "combined_results.csv")
    combined_df.to_csv(combined_csv_path, index=False, encoding="utf-8")
    logger.info(f"Consolidated {len(combined_df):,} total evaluations into: {combined_csv_path}")

    # Build analysis_df and collect excluded runs metadata
    excluded_runs = []
    for (m_key, lang), grp in combined_df.groupby(["model_key", "language"]):
        is_valid_run = (grp["validity_flag"] == "VALID").all()
        if not is_valid_run:
            # Read metadata from summary json
            reasons = []
            acc_val = float(grp["is_correct"].mean() * 100.0) if "is_correct" in grp.columns and len(grp) > 0 else 0.0
            top_share_val = 0.0
            for s_dir in search_dirs:
                sum_p = os.path.join(s_dir, m_key, f"{lang}_summary.json")
                if os.path.exists(sum_p):
                    try:
                        with open(sum_p, "r", encoding="utf-8") as sf:
                            sd = json.load(sf)
                        reasons = sd.get("validity_reasons", [])
                        acc_val = sd.get("accuracy_percent", acc_val)
                        dist = sd.get("diagnostics", {}).get("prediction_distribution", {})
                        if not dist:
                            dist = sd.get("raw_prediction_distribution", {})
                        tot = sum(dist.values())
                        if tot > 0:
                            top_share_val = float(max(dist.values()) / tot)
                        break
                    except Exception:
                        pass
            logger.warning(f"⚠️ EXCLUDED from statistical analysis: {m_key} [{lang}] | Validity reasons: {reasons}")
            excluded_runs.append({
                "model": m_key,
                "language": lang,
                "accuracy": acc_val,
                "top_letter_share": top_share_val,
                "reasons": reasons,
            })

    if getattr(CONFIG, "analysis_valid_only", True):
        analysis_df = combined_df[combined_df["validity_flag"] == "VALID"].copy()
    else:
        analysis_df = combined_df.copy()

    if analysis_df.empty:
        logger.error("❌ analysis_df is EMPTY: all model x language runs were excluded by validity filter. Skipping statistical analysis and plots.")
    else:
        # Section 18 Reliability Diagrams
        saved_rel_plots = []
        for (m_key, lang), grp in analysis_df.groupby(["model_key", "language"]):
            c_vals = grp["confidence"].values
            y_vals = grp["is_correct"].values
            acc = float(np.mean(y_vals) * 100.0) if len(y_vals) > 0 else 0.0
            ece_val = expected_calibration_error(c_vals, y_vals, n_bins=10)
            rel_bins = compute_calibration_bins(c_vals, y_vals, n_bins=10)
            p_path = os.path.join(CONFIG.output_dir, f"reliability_{m_key}_{lang}.png")
            p_title = f"Reliability Diagram: {m_key} [{lang}] | Acc: {acc:.1f}% | ECE: {ece_val:.4f}"
            plot_reliability_diagram(rel_bins, title=p_title, save_path=p_path)
            saved_rel_plots.append(p_path)

            # Raw reliability diagram if cyclic4 debiasing was applied
            if "confidence_raw" in grp.columns and grp["confidence_raw"].notna().any():
                c_raw = grp["confidence_raw"].dropna().values
                pred_raw_col = grp["pred_raw"] if "pred_raw" in grp.columns else grp.get("predicted_answer")
                gold_col = grp["gold_answer"] if "gold_answer" in grp.columns else grp.get("correct_answer")
                if pred_raw_col is not None and gold_col is not None:
                    y_raw = (pred_raw_col.astype(str) == gold_col.astype(str)).astype(float).values[:len(c_raw)]
                else:
                    y_raw = y_vals[:len(c_raw)]
                acc_raw = float(np.mean(y_raw) * 100.0) if len(y_raw) > 0 else 0.0
                ece_raw = expected_calibration_error(c_raw, y_raw, n_bins=10)
                bins_raw = compute_calibration_bins(c_raw, y_raw, n_bins=10)
                p_raw_path = os.path.join(CONFIG.output_dir, f"reliability_{m_key}_{lang}_raw.png")
                p_raw_title = f"Reliability Diagram: {m_key} [{lang}] (raw, before debiasing) | Acc: {acc_raw:.1f}% | ECE: {ece_raw:.4f}"
                plot_reliability_diagram(bins_raw, title=p_raw_title, save_path=p_raw_path)
                saved_rel_plots.append(p_raw_path)

        # Run Econometric & Statistical Hypothesis Analysis
        stats_report = run_full_statistical_analysis(analysis_df)
        stats_report["saved_plots"] = saved_rel_plots

        stats_json_path = os.path.join(CONFIG.output_dir, "statistical_analysis_report.json")
        with open(stats_json_path, "w", encoding="utf-8") as f:
            json.dump(stats_report, f, indent=2, ensure_ascii=False)

        stats_md_path = os.path.join(CONFIG.output_dir, "statistical_analysis_report.md")
        report_md = format_markdown_report(stats_report, excluded_runs=excluded_runs)
        with open(stats_md_path, "w", encoding="utf-8") as f:
            f.write(report_md)

        print("\n" + report_md)

        # Render Visualizations (Per model x language)
        plot_fragmentation_vs_calibration(
            analysis_df,
            title="Tokenization TPW vs. Calibration Residual",
            save_path=os.path.join(CONFIG.output_dir, "fragmentation_vs_calibration.png"),
            metric="overall_tpw",
        )
        plot_fragmentation_vs_calibration(
            analysis_df,
            title="Tokenization TPC vs. Calibration Residual",
            save_path=os.path.join(CONFIG.output_dir, "fragmentation_tpc_vs_calibration.png"),
            metric="overall_tpc",
        )

    # Generate Run Manifest
    manifest = {
        "timestamp": datetime.now().isoformat(),
        "strict_real_mode": STRICT_REAL_MODE,
        "backend": CONFIG.backend,
        "reproducibility_seed": CONFIG.seed,
        "evaluation_sample_counts": {
            "per_language_target": CONFIG.n_samples,
            "bn_samples": int((combined_df["language"] == "bn").sum()) if "language" in combined_df.columns else 0,
            "en_samples": int((combined_df["language"] == "en").sum()) if "language" in combined_df.columns else 0,
        },
        "dataset_ids": {
            "bn_dataset_id": CONFIG.bn_dataset_id,
            "bn_hf_config": CONFIG.bn_hf_config,
            "en_dataset_id": CONFIG.en_dataset_id,
            "en_hf_config": CONFIG.en_hf_config,
            "en_split": CONFIG.en_split,
        },
        "prompt_variant": CONFIG.prompt_variant,
        "models": {
            k: {
                "hf_id": v["hf_id"],
                "chat_template_used": resolve_run_settings(k, CONFIG)["use_chat_template"],
                **{kk: resolve_run_settings(k, CONFIG)[kk] for kk in
                   ("prompt_variant", "effective_n_shots", "use_chat_template", "debias", "settings_source")},
            }
            for k, v in MODEL_REGISTRY.items()
        },
        "environment": {
            "is_kaggle": IS_KAGGLE,
            "gpu_available": GPU_AVAILABLE,
            "device_name": DEVICE_NAME,
            "python_version": sys.version,
        },
        "package_versions": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": stats.__version__ if hasattr(stats, "__version__") else "available",
            "statsmodels": getattr(sm, "__version__", "unknown"),
            "torch": torch.__version__ if HAS_TORCH else "missing",
        },
        "models_discovered": list(discovered_models),
        "total_evaluations_combined": len(combined_df),
        "completion_status": "complete" if len(discovered_models) == len(MODEL_REGISTRY) else "in_progress",
        "combined_csv": combined_csv_path,
    }
    manifest_path = os.path.join(CONFIG.output_dir, "run_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    logger.info(f"Run manifest persisted to: {manifest_path}")

    # Persist consolidated root status.json
    root_status_path = os.path.join(CONFIG.output_dir, "status.json")
    with open(root_status_path, "w", encoding="utf-8") as f:
        json.dump({
            "models": {m: {"status": "complete" if (combined_df[combined_df["model_key"] == m]["language"].value_counts().to_dict().get("bn", 0) == CONFIG.n_samples and combined_df[combined_df["model_key"] == m]["language"].value_counts().to_dict().get("en", 0) == CONFIG.n_samples) else "incomplete"} for m in discovered_models},
            "prompt_variant": CONFIG.prompt_variant,
            "timestamp": datetime.now().isoformat(),
            "total_records": len(combined_df),
        }, f, indent=2)
    logger.info(f"Consolidated root status saved to: {root_status_path}")


# Print current status dashboard
status_table(CONFIG.output_dir, CONFIG.previous_results_dir)


## Section 19: Methodological Limitations, Non-Causal Framing & Kaggle Guide

### Methodological & Epistemic Scope
1. **Observational Association vs. Causality:**
   All statistical models estimate observational associations. Confounding from pretraining corpus composition, alignment, and parameter scale cannot be eliminated in benchmark evaluations. All conclusions must be framed using "predicts" or "is associated with," never "causes."
2. **Bengali Script Characteristics:**
   Bengali is an abugida script featuring complex conjuncts (*যুক্তবর্ণ*) and bound inflectional morphemes. Whitespace segmentation is the established literature heuristic (TituLLMs, AdaptMix).
3. **Continuous Residuals vs. Partitioned ECE:**
   Individual item residuals ($e_i = |\hat{p}_i - \mathbb{I}(\hat{y}_i = y_i)|$) quantify continuous deviation per question, while ECE aggregates over probability bins. Both are reported to provide continuous and grouped perspectives.

---

### Step-by-Step Kaggle Execution Guide (real data, real models)
1. **Hugging Face access (one-time):**
   * While logged in, open the **BnMMLU** dataset page (`samanjoy2/BnMMLU`) and accept its access conditions.
   * Open `meta-llama/Llama-3.2-3B-Instruct` and accept the Llama 3.2 license.
   * Create a **read** token for that same account and store it as the Kaggle secret `HF_TOKEN` (Add-ons -> Secrets).
2. **Runtime:** GPU T4 x2, Internet **On**. No dataset attachment is needed (data comes from the Hub).
3. **Configure** (Section 3): `backend="transformers"`, `n_samples=1500`. Change settings *before* running the data-preparation cell.
4. **Run top to bottom:** Section 5 logs the real BnMMLU subject list and what could not be matched to MMLU. If subjects you care about are unmapped, add them to `SUBJECT_MAP_EXTRA` (Section 4) and re-run Section 5.
5. **Run Section 14b (tokenizer audit)** and read its warnings before spending GPU time.
6. **Run models one at a time** (Sections 15, 16, 17), saving a Kaggle version after each, then run Section 18.
7. **Multi-session continuity:** attach the previous output as an input dataset and set `CONFIG.previous_results_dir = "/kaggle/input/your-previous-run/results"`.
8. **Reporting:** cite the Hub commit SHAs stored in `subset_meta.json` and state that the BnMMLU release was marked preliminary.
